# C-test ONE · Generator

*Ctest-Generator-v2.69.0*

Composes candidate items with the composer model, one cell of the (retrieval,
transformation) grid at a time, and keeps those that pass: the verifier runs each rule cold and
at least two of three runs must reproduce the sequence. It then asks the composer for equivalent
statements of each rule and for rivals at several prefix lengths.

| reads | writes |
|---|---|
| `ctest_calibration.json`, `ctest_themes.json`, `ctest_battery.json` | `ctest_battery.json`, `ctest_programs.json`, `ctest_themes.json`, `ctest_prompts.json` |

**Needs** a Gemini API key (`GOOGLE_API_KEY`). No GPU. The main settings are `PLAN`, the cells
to fill, `N_PER_CELL` and `OVERSAMPLE`.

## Running it

**Locally.** From the repository root, install the requirements and open the notebook in Jupyter:

```
pip install -r requirements.txt
jupyter lab notebooks/
```

API keys are read from environment variables (see `.env.example`). On the first local run the
notebook copies `data/` into a `work/` folder and runs there, so the released data are never
modified; set `CTEST_WORK_DIR` to use another folder.

**Colab.** Upload the data files to `MyDrive/ctest` and mount Drive; keys go in the Secrets panel.

**Kaggle.** Attach a dataset holding the data files and set `KAGGLE_DATASET` to its name; keys go in
Add-ons, Secrets. With Kaggle credentials the notebook can also push its results back as a new
version of the dataset.


In [ ]:
# Local runs work in a copy of the data, so that data/ stays exactly as released.
# On Colab and Kaggle this cell does nothing.
import os, glob, shutil
if not (os.path.exists("/kaggle") or os.path.exists("/content")):
    _root = os.getcwd()
    if os.path.basename(_root) == "notebooks":
        _root = os.path.dirname(_root)
    try:
        from dotenv import load_dotenv          # optional: keys from a .env file
        load_dotenv(os.path.join(_root, ".env"))
    except ImportError:
        pass
    WORK_DIR = os.path.abspath(os.environ.get("CTEST_WORK_DIR", os.path.join(_root, "work")))
    os.makedirs(WORK_DIR, exist_ok=True)
    for _f in glob.glob(os.path.join(_root, "data", "*")):
        _dst = os.path.join(WORK_DIR, os.path.basename(_f))
        if not os.path.exists(_dst):
            shutil.copy(_f, _dst)
    os.chdir(WORK_DIR)
    print(f"local run: working in {WORK_DIR}")


## 1 · Install


In [ ]:
# ---------------------------------------------------------------------------
# If the key cannot be found in the secrets panel, put it here. The environment
# is checked before any provider, so this always wins.
#
# Kaggle secrets are PER NOTEBOOK: having one in another session does not carry
# over. Add-ons -> Secrets, and switch the toggle on for this notebook.
API_KEY_HERE = ""       # e.g. "AIza..."  Leave empty to use the secrets panel.
# ---------------------------------------------------------------------------
import os
if API_KEY_HERE:
    os.environ["GOOGLE_API_KEY"] = API_KEY_HERE
    print("GOOGLE_API_KEY taken from API_KEY_HERE in this cell")

%pip install -q -U google-genai

import os
try:
    import google.colab
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as _e:
    print("not on Colab, or Drive not mounted:", _e)


## 1b · Preamble


In [ ]:
import os, sys, re, json, time, glob, math, shutil, random, hashlib, platform
import collections, statistics as st, datetime, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

BUILD = "Ctest-Generator-v2.69.0"
print("=" * 72)
print(f"{BUILD}")
print("=" * 72)

# The filesystem decides, not an import. Kaggle ships packages that make
# `import google.colab` succeed, which sent ENV to "colab" on Kaggle and killed
# the run on the unmounted-Drive guard.
if os.path.exists("/kaggle"):
    ENV = "kaggle"
elif os.path.isdir("/content"):
    ENV = "colab"
else:
    ENV = "local"
# Two separate places, because they are separate things and always were.
DRIVE_FOLDER   = "ctest"        # Colab: /content/drive/MyDrive/ctest
KAGGLE_DATASET = "ctest-one-data"    # Kaggle: the dataset to read from and push to.
                                # A Kaggle dataset is a FOLDER of files, not a
                                # single file.

DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
# OUTDIR is a second place to copy finished files to. On Colab that is Drive.
# On Kaggle there is no such place: an attached dataset is READ-ONLY at
# /kaggle/input, and the working directory is the only writable one and is
# already where files are written. Copying a file onto itself raises
# SameFileError, so OUTDIR stays "." and persistence is the dataset push in the
# cell below.
if ENV == "colab" and os.path.isdir("/content/drive/MyDrive"):
    OUTDIR = DRIVE
else:
    OUTDIR = "."
print(f"environment      {ENV}   python {platform.python_version()}")

# shutil.copy writes a FILE when its destination is not an existing directory, so
# without this every save lands in a file named after the folder and the results
# leave with the session.
if OUTDIR != ".":
    if os.path.exists(OUTDIR) and not os.path.isdir(OUTDIR):
        raise SystemExit(f"{OUTDIR} exists but is a file. Rename or delete it.")
    os.makedirs(OUTDIR, exist_ok=True)
elif ENV == "colab":
    raise SystemExit("Drive is not mounted, so anything written would be lost. Run\n"
                     "    from google.colab import drive; drive.mount('/content/drive')")
if ENV == "kaggle":
    print(f"outputs go to    the working directory, then pushed to the "
          f"{KAGGLE_DATASET} dataset")
    print( "                 (a dataset cannot be written to in place; see the "
           "persistence cell)")
if ENV != "kaggle":
    print(f"outputs go to    {OUTDIR}")

try:
    import torch
    if torch.cuda.is_available():
        print("GPU              attached, and not needed. Switch to CPU and save quota.")
except Exception:
    pass

def _secret(n):
    """Look for a secret, and SAY what each provider reported.

    An earlier version swallowed the Kaggle error and fell through to the Colab
    branch, which raises on Kaggle, so the result was a bare None and nothing
    said why. Kaggle secrets are PER NOTEBOOK: one attached to another session
    does not carry over."""
    if os.environ.get(n):
        return os.environ[n]
    notes = []
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(n)
        if v:
            return v
        notes.append("kaggle: returned nothing")
    except ImportError:
        pass                                  # not on Kaggle, which is fine
    except Exception as e:
        notes.append(f"kaggle: {type(e).__name__}: {str(e)[:90]}")
    try:
        from google.colab import userdata
        v = userdata.get(n)
        if v:
            return v
        notes.append("colab: returned nothing")
    except ImportError:
        pass
    except Exception as e:
        notes.append(f"colab: {type(e).__name__}: {str(e)[:90]}")
    for m in notes:
        print(f"                 {m}")
    return None

_key = _secret("GOOGLE_API_KEY")
if _key: os.environ["GOOGLE_API_KEY"] = _key
print(f"GOOGLE_API_KEY   {'found' if _key else 'MISSING'}")

# The dataset, from a secret when there is one, so two accounts can work in
# parallel on separate copies with this notebook byte-identical in both.
_ds = _secret("KAGGLE_DATASET")
if _ds:
    KAGGLE_DATASET = _ds.strip()
print(f"KAGGLE_DATASET   {KAGGLE_DATASET}"
      + ("   from the secret" if _ds else "   the built-in default"))
if not _key:
    print()
    print("=" * 72)
    print("STOPPING: no GOOGLE_API_KEY, and the composer is an API model")
    print("=" * 72)
    if ENV == "kaggle":
        print("  Kaggle secrets are PER NOTEBOOK. Having one in another session")
        print("  does not carry over: Add-ons -> Secrets, and switch the toggle")
        print("  ON for this notebook. Any lines above say what the lookup")
        print("  reported.")
    elif ENV == "colab":
        print("  Colab: the key panel on the left, and the notebook-access")
        print("  toggle has to be on for this notebook.")
    print()
    print("  Or set API_KEY_HERE in the install cell, which is checked first.")
    raise SystemExit("no GOOGLE_API_KEY")

_PRINT_LOCK = threading.Lock()

def log(*a, **kw):
    with _PRINT_LOCK:
        print(*a, flush=True, **kw)

def stage(t):
    with _PRINT_LOCK:
        print(f"\n{'='*72}\n== {t}\n{'='*72}", flush=True)

def find(f):
    for p in [f, f"/content/{f}", f"{DRIVE}/{f}"] + glob.glob(f"/kaggle/input/**/{f}", recursive=True):
        if os.path.exists(p): return p
    return None

def load(name, default):
    p = find(name)
    if not p:
        log(f"  {name}: absent, starting empty"); return default
    log(f"  {name}: loaded from {p}")
    return json.load(open(p))


## 2 · Config

`PLAN` is the run.
Give it `(r, t)` pairs, or `("load", load, mix)` triples and
they are converted.
`GRIDS` holds a few ready-made spreads.

`RIVAL_PREFIXES` decides which prefix lengths get a rival search.
Anything
longer than the sequence is skipped.


In [ ]:
N_PER_CELL = 1 
COMPOSER     = "gemini-3.6-flash"
MAX_WORKERS  = 6          # API calls in flight
WORLD_FROM   = 1
WORLD_PUSH   = False

OPEN_CASE_TERMS = 2

REPAIR_MAX   = 8          # how many lost rules a run will try to repair.
                          # Each costs up to REPAIR_ROUNDS calls and a produce
                          # apiece, so about a minute a round, and the cap is
                          # what keeps a run of bad luck from eating the night.

REPAIR_ROUNDS = 5

OVERSAMPLE = 12
ID_PREFIX    = "v17"      # a fresh series, so the new items are separable from
                          # the old ones by id alone. The v13 items were made
                          # under DIAL_MARGIN = 2, where an item asked for t=5
                          # that did 3 operations passed, and at sequence
                          # lengths that cap a certifying rule at 64 characters.
                          # Their t labels are loose and most of them cannot
                          # certify at the length they have. They stay in the
                          # battery, untouched and still usable for comparison,
                          # but a pricing pass should be pointed at v14.

# A cell is (r, t): r = memorised tables, t = iterated computation.
# Equivalently ("load", load, mix): load = r + t, mix = crystallised share.
GRIDS = {
  "grid":     [(r, t) for r in range(3) for t in range(8)],

  "deep":     [(r, t) for r in range(3) for t in (5, 6, 7)],

  "t7":       [(r, 7) for r in range(3)],

  "grid_nw":  [(r, t) for r in range(3) for t in range(8)
               if not (r >= 2 and t >= 5)],

  "coarse":   [(r, t) for r in range(3) for t in (0, 1, 3, 5, 6)
               if not (r >= 2 and t >= 5)],

  "mid": [(1, 2), (1, 3), (1, 4), (1, 5), (2, 2), (2, 3)],

  "gaps":     [(r, t) for r in range(3) for t in (2, 4)],

  "thin": [(1, 0), (0, 1), (2, 0), (2, 1)],

  "top": [(1, 5), (0, 5), (2, 2), (3, 2)],

  "cheap": [(0, 0), (0, 1), (1, 0), (1, 1), (2, 0), (2, 1), (2, 2)],

  "easy":     [(0,0),(0,1),(1,0),(1,1),(2,0)],

  "vryeasy":  [(0,0),(1,0)],

  # One cell, to see whether the calibrated ask lands on it. r0t0 holds one
  # valid measured item out of six asked for, which is what the calibration
  # exists to fix: the ask for this target is t = -1 always and r = -0.4, so
  # r = -1 two times in five.
  "r0t0":     [(0,0)],
  # where the frontier fails
  "frontier": [(0,4),(0,5),(0,6),(1,5),(1,6)],
  # one balanced sweep by load and mix, to show the parametrisation
  "sweep":    [("load",2,0.0),("load",2,0.5),("load",4,0.0),("load",4,0.25),
               ("load",4,0.5),("load",6,0.0),("load",6,0.17),("load",6,0.33)],
}
PLAN = GRIDS["grid"]      # every cell at r <= 2, t <= 7: what the paper needs

# ----------------------------------------------------------- the item -----
# FILL reads the battery, counts usable items per cell, and asks only for the
# shortfall against N_PER_CELL. So PLAN becomes a TARGET the battery is brought
# up to, and a run can be repeated until the grid is populated without
# generating anything already held.
FILL         = False
FILL_COUNTS_EXCLUDED = False   # True counts excluded items towards the target

SHUFFLE_SLOTS = True      # see the note below: order decides who starves
THEMES_PER_CALL = 30      # a larger ask overflows the reply budget
def _uses(t):
    """How many items a theme has carried. `used` was a boolean; a count is
    the same thing with room above it."""
    if "uses" in t:
        return int(t.get("uses") or 0)
    return 1 if t.get("used") else 0

THEME_REUSE     = 1
THEME_STORE_MIN = 60

THEME_TOPUP  = 0

TEXT_SHARE   = "about a quarter"   # how much of a fetched theme list should be
                          # well-known texts: novel openings, speeches, nursery
                          # rhymes, hymns. Written as a phrase because it goes
                          # into the prompt. "about a quarter" is the settled
                          # value; "about half" is for a run meant to bring
                          # them in, and should come back down afterwards, in
                          # the same way WORLD_PUSH does.

THEME_PREFER_NEW = True

THEMES_PER_SLOT = 3
THEMES_MAX_CALLS = 14
THEMES_MAX_ROUNDS = 4     # how many times to ask within one call when short. One was enough
                          # while the file held 230 themes and the call never
                          # fired; from an empty file one call caps at
                          # THEMES_PER_CALL and cannot fill a 38-slot run.

TERMS_FROM     = "produce"

VALIDITY_SECOND_PASS = True
VALIDITY_TO = 100
EXTEND_REPS    = 3         # independent runs of the rule
EXTEND_WINS    = 2         # how many must agree
EXTEND_TEMPS   = (0.0, 0.3, 0.6)   # a spread, so agreement is evidence rather
                                   # than the same computation echoed back

TERMS_MIN      = 28        # floor, for the shallow cells
TERMS_MAX      = 72

K_ADMIN        = 12        # terms an examinee is asked for
TERMS_SLACK    = 4         # terms a candidate may fall short by, so that a
                           # couple lost to a formatting slip does not throw
                           # the item away

EXPECTED_CHARS = {}
_raw = {0: 63, 1: 69, 2: 94, 3: 106, 4: 114, 5: 106, 6: 158, 7: 240, 8: 268}
_run = 0
for _L in sorted(_raw):
    _run = max(_run, _raw[_L]); EXPECTED_CHARS[_L] = _run
del _raw, _run, _L

CALIBRATION    = "ctest_calibration.json"
REF_FOR_CALIB  = "Qwen/Qwen3-8B"   # whose constants to read, when present
CALIB_DEFAULTS = dict(bits_per_char=2.9, beta_listing=4.1, beta_gate=6.5,
                      c0_listing=86.0, delta=15.0)
MAX_TERM_LEN   = 24
VERIFY_REPS, VERIFY_WINS = 3, 2

N_EQUIVALENTS  = 8
N_RIVALS       = 6        # per prefix
CLAIM_SCREEN = True      # reject in phase 3, before any terms are produced,
                         # when the composer's own list of operations or tables
                         # is the wrong length for the cell. Free, and it saves
                         # the three produce calls a phase 4 rejection costs.
                         # Set False to record the claims without acting on them.
DOMAIN_GATE  = True
DIAL_MARGIN  = 1
DOMAIN_GATE_NONE = True  # reject when the rule needs NOTHING at all and r >= 1
DOMAIN_RETRY = True      # a slot that runs out gets one more generation call,
                         # told how many tables it needed against how many the
                         # cell asked for

EQUIV_RETRY    = True
EQUIV_RETRY_IF = 5

# An item with no verified equivalent cannot be priced, so its rivals would be
# compared against nothing. Four such items bought 40 rivals on one run, about
# twenty minutes wasted. This pays for the retry above and then some.
SKIP_RIVALS_IF_NO_EQUIV = True

RIVAL_RETRY_IF = 3
RIVAL_PREFIXES     = [5, 13]      # the floor: always asked, whatever the estimate
RIVAL_PREFIX_MAX   = 60
RIVAL_ADAPTIVE     = True         # add one prefix where certification is expected
RIVAL_ADAPTIVE_PAD  = 2     # kept, and applied after the factor
RIVAL_ADAPTIVE_MULT = 1.8
RIVAL_ADAPTIVE_N    = 2     # how many adaptive points: one at the estimate
                            # and one above it, so an item whose estimate is
                            # right is not paying for the item whose is not

DO_GENERATE    = True

# ---- extending the terms --------------------------------------------------
# A pass on its own: run each item's verified rule further and keep whatever
# the runs agree on. No pricing, no GPU, minutes for the battery.
#
# It is needed because the listing has to be priced at N_#, which reaches 82
# terms while the sequences hold 72. Fifty more covers every item. It is worth
# having anyway: an item whose rule falls apart at term 80 is one to know about.
EXTEND_ONLY  = False       # OFF. The validity pass inside generation writes
SECOND_ONLY  = False
EXTEND_TO    = 100         # terms to aim for, counting from the start

SECOND_MODEL   = "gemini-3.1-flash-lite"
# THE TAG FOLLOWS THE MODEL. Set by hand it goes stale the moment the model
# changes, and SECOND_ONLY_MISSING then reads the previous model's verdicts
# and skips every item: a run under gemini-2.0-flash-lite found "nothing to
# do" because seventeen items already carried a haiku verdict.
SECOND_K       = 15            # terms to ask for after the prefix. The
                               # annotator asks k_u = min(20, N - L_u), which
                               # is 12 to 20 across the battery, so 15 sits in
                               # the middle of it.
SECOND_PREFIX_TAG = "qwen3-14b"   # whose L_u to show. An item not priced under
                                  # it falls back to SHOWN_N.
SECOND_THINK   = 8000          # thinking tokens to ask for. The lite models
                               # do little by default and the job needs it:
                               # at the same settings 3.6-flash reproduced
                               # every item and 3.1-flash-lite failed at term
                               # 1. None leaves the model's default.

SECOND_TAG     = (re.sub(r"[^a-z0-9]+", "-",
                         SECOND_MODEL.split("/")[-1].lower()).strip("-")
                  + f"-t{SECOND_THINK if SECOND_THINK is not None else 'def'}"
                  + f"-k{SECOND_K}")
SECOND_REPS    = 3             # [V2(p)]^{1@3}: any one reproducing is a pass
SECOND_WINS    = 1
SECOND_WHICH   = [
    # certified under Qwen3-14B: a filter should pass these
    "v17-0002", "v17-0003", "v17-0004", "v17-0005", "v17-0006", "v17-0009",
    "v17-0011", "v17-0016", "v17-0023", "v17-0030", "v17-0031", "v17-0032",
    "v17-0034", "v17-0043", "v17-0046", "v17-0049", "v17-0050", "v17-0054",
    "v17-0058", "v17-0074", "v17-0076", "v17-0083", "v17-0084", "v17-0096",
    "v17-0100", "v17-0101",
    # fell to the listing: a filter should fail these
    "v17-0020", "v17-0042", "v17-0045", "v17-0066",
    "v17-0072", "v17-0089", "v17-0090", "v17-0091",
]
SECOND_ONLY_MISSING = False    # OFF while the model is still being settled:
                               # the failed runs left verdicts under their own
                               # tags and those would otherwise skip everything
SECOND_PUSH_EVERY   = 10
EXTEND_WHICH = ["v17-0010", "v17-0027", "v17-0031", "v17-0032", "v17-0038",
                "v17-0043", "v17-0058", "v17-0060", "v17-0064", "v17-0076"]

# Which reproducibility trials to run.
#   "shown"  the first SHOWN_N terms given, then the rule. The signal we do
#            not have yet, and the quick one: about 3 calls an item.
#   "cold"   the rule alone. Already recorded for this battery from earlier
#            runs, so repeating it says nothing new about reproducibility,
#            though it is what extends the terms.
#   "both"   one pass, both trials, about 6 calls an item.
TRIALS = "cold"

# Which description to run. Together with TRIALS this gives the four blocks:
# bare_cold, bare_shown, seed_cold, seed_shown. The verdict is written under
# that name, so a run never overwrites another trial's answer.
#
#   "seed"  the composer's rule, which is what generation ran, so a comparison
#           with the stored terms is like for like
#   "bare"  the shortest rewriting verified against those terms, which is a
#           different claim and often the better specified of the two
#
# Only items that HAVE a bare can be run with "bare"; eight in this battery
# have none and are skipped.
RULE_FROM = "seed"

DO_PROGRAMS    = True      # False gives a fast discovery run, items only
DO_COMPONENTS  = True     # the manipulation check on the dials, 1 call per item

PROGRAMS_FOR   = "new"
COMPONENTS_FOR = "new"
THIN_EQUIV     = 2        # verified equivalents an item needs
THIN_RIVALS    = 1        # rivals at any prefix an item needs

# Every programs pass records an attempt on the item. An item that has been
# through this many passes and is still short is marked `flawed` and stops being
# retried: three of them cost about 20 minutes and $2 per run, every run, and
# fail the same way each time.
MAX_PROGRAM_ATTEMPTS = 3  # raised from 2 for this round: the equivalents retry
                          # is newer than most of the thin items, so they
                          # deserve one more attempt before being marked
MARK_HOPELESS  = True     # False keeps retrying them and just says so

# Short of an item budget, do the most promising first: fewest attempts so far,
# then closest to the bar. A run cut short then leaves behind the ones least
# likely to have worked anyway.
# An unattended overnight run needs a clean stop of its own. Kaggle cuts a
# session at its limit and phase 5 has no natural end, so without this the run
# is killed mid-item and the last one is lost. Below the platform limit, so the
# summary, the accounting row and the final dataset push all still happen.
# This notebook needs no GPU, so run it on a CPU session: a longer limit and no
# GPU quota spent.
TIME_BUDGET_MIN = 0     # 4h, the backstop rather than the plan.
                          # MAX_SLOTS sizes the run; this only stops a run that
                          # is going worse than expected.

SHOW_TERMS   = 30         # terms printed per item in the listing before
                          # phase 5. The whole sequence is in the battery file;
                          # this is for reading the run.

MAX_SLOTS  = 4 

PROGRAMS_MAX_ITEMS = 0    # 0 means no limit. Phase 5 is the slow part, about
                          # 5 minutes an item, so this is what keeps a
                          # diagnostic run to half an hour.

# ------------------------------------------- screens on the terms (free) ---
REPEAT_EVIDENCE = 3
GUESS_START     = 8
# The collapse screen rejects an item a dull predictor can continue. At the
# bottom of the grid that forbids exactly what the cell asks for: "a, a, b, b,
# c, c" is a good r0t0 item and a field predictor gets nearly all of it, so the
# composer added machinery instead. The bar therefore relaxes as load falls,
# squared so the relaxation stays at the very bottom: off at load 0, 0.67 at
# load 1, 0.47 at load 2, and 0.40 from load 3 up, which is where it is today.
GUESS_REJECT      = 0.40   # the bar at load 3 and above
GUESS_REJECT_EASY = 1.01   # the bar at load 0, which is to say no bar at all.
                           # 0.90 was not enough: "a, a, b, b, c, c" is 100%
                           # continuable by a dull predictor, and that is what
                           # a trivial item IS. At load 0 the screen has to be
                           # off rather than lenient, or the cell cannot be
                           # filled with what it asks for.
GUESS_EASY_LOAD   = 3      # where the relaxation runs out
GUESS_WINDOW    = 8
FREE_SHARE_WARN = 0.60
PREFIX_CLASH    = 5       # reject a candidate sharing this many opening terms
SCREEN_CHAR_COUNTS = False  # OFF. It caught two rules and missed three, which
                            # said "the rank's letter count" and "color len"
                            # rather than len(...), and a regex over prose will
                            # keep losing that race. hidden_line now carries it
                            # as a worked negative example instead, which acts
                            # on the cause rather than the wording.
                          # with an item already held

# Duplicate detection, with thresholds MEASURED on the v5 battery rather than
# guessed. Rule-word overlap ran 0.40 to 0.62 across pairs that should be
# merged and 0.00 to 0.22 across pairs that must stay separate, so 0.35
# separates them WITHIN a cell. Across cells the bar is much higher, because
# the cell is the design variable and items in different cells are meant to
# differ.
# The "already worn out" note in the generation prompt is read off the battery
# rather than hard-coded, so it is silent on a fresh battery and never bans a
# mechanism forever. These cap how much of the prompt it can take.
WORN_TOP   = 6            # phrases listed, at most
WORN_FLOOR = 2            # items a phrase must appear in before it is listed

# The SDK warns about automatic function calling on every call. Disabling it
# explicitly should change nothing (no tools are configured) and should silence
# the warning, which is the confirmation that no interpreter is involved.
# Two theme filters, both OFF by default. They rejected 45 candidates against 6
# planned in one run. See theme_fits for why neither is needed now.
THEME_NUMERIC_SKIP = 0.8
SKIP_LONG_THEMES = False   # skip a closed-list theme when t >= 4
SKIP_HARD_THEMES = False   # skip a non-elementary theme at (0, 0)

DISABLE_AFC = True

DEDUP_SAME_CELL = 0.35
DEDUP_ANY_CELL  = 0.80

# ------------------------------------------------------------- the API ----
# A 39-item run spent about 5M tokens and a 19-item run about 5.3M, so a 47-item
# run under a longer prompt wants headroom rather than a limit that stops the
# night halfway. Both are safety rails, not targets.
TOKEN_BUDGET, MAX_SPEND_USD = 150_000_000, 120.0
USD_PER_MTOK_IN, USD_PER_MTOK_OUT = 0.30, 3.75
API_RETRIES, API_BACKOFF = 3, 5
# A rival call grows with the prefix: measured at 92s at L=5, 173s at L=11 and
# 312s at L=17, because the search happens in thinking tokens and there is more
# prefix to fit. At 240 the L=17 calls were over the limit and came back as
# 408 Request Timeout with nothing usable.
API_TIMEOUT_S      = 900   # generous, since a slow call is better than none
API_TIMEOUT_RIVALS = 900
GEN_MAX_OUT, PROG_MAX_OUT, OTHER_MAX_OUT = 12000, 10000, 8000
RIVAL_MAX_OUT = 24000
GEN_MAX_OUT_PER_T = 2000

BATTERY = "ctest_battery.json"
PROGRAMS = "ctest_programs.json"
THEMES   = "ctest_themes.json"
PROMPTS  = "ctest_prompts.json"
# ===========================================================================

def as_cell(spec):
    """(r, t) from either form. ('load', load, mix) -> r = round(load*mix)."""
    if spec and spec[0] == "load":
        _, load, mix = spec
        r = int(round(load * mix))
        return r, max(0, load - r)
    return tuple(spec)

# ---- the calibration file -------------------------------------------------
def load_calibration():
    """Constants describing the reference machines, or the defaults.

    Read here, never written, so this notebook needs no GPU.

    The file is keyed by provenance and this reads them in order:

      pooled     from the calibrator, across every machine on the current
                 probe set. The conservative end of each, so it asks for more
                 terms rather than fewer. What a generator should use.
      reference  one machine's own block, when there is no pooled one.
      fitted     what the pricing pass measured while pricing a battery, kept
                 per battery. Last resort, because it describes that battery
                 as much as the machine.
      defaults   the constants below, measured on earlier runs and hardcoded.

    Before v2.30.0 this read flat keys off `machines[name]`, which is the shape
    the file had when only the pricing pass wrote it. The calibrator nests them
    now, so the old reader found an entry with no `beta_listing` in it and
    silently fell back to the defaults.
    """
    c = dict(CALIB_DEFAULTS); c["source"] = "defaults"
    p = find(CALIBRATION)
    if not p:
        return c
    try:
        d = json.load(open(p))
    except Exception as e:
        print(f"calibration      {CALIBRATION} unreadable ({str(e)[:50]}), "
              f"using defaults")
        return c

    entry, src = None, None
    if isinstance(d.get("pooled"), dict) and d["pooled"].get("beta_listing"):
        entry = d["pooled"]
        src = (f"pooled over {len(d['pooled'].get('machines', []))} machine(s)"
               f" on probes {d['pooled'].get('probes', '?')}")
    else:
        m = (d.get("machines") or {}).get(REF_FOR_CALIB) or {}
        if isinstance(m.get("reference"), dict):
            entry, src = m["reference"], f"{REF_FOR_CALIB} reference block"
        elif isinstance(m.get("fitted"), dict) and m["fitted"]:
            k = sorted(m["fitted"])[-1]
            entry, src = m["fitted"][k], (f"{REF_FOR_CALIB} fitted on {k}, "
                                          f"which describes that battery too")
        elif m:
            entry, src = m, f"{REF_FOR_CALIB}, old flat shape"
    if not entry:
        print(f"calibration      {CALIBRATION} has nothing usable for "
              f"{REF_FOR_CALIB}, using defaults")
        return c

    c["source"] = src
    for k in CALIB_DEFAULTS:
        if entry.get(k) is not None:
            c[k] = float(entry[k])
    # the calibrator names it delta_gen, to keep it apart from the delta a
    # pricing run uses, which is capped by Definition 4.1 and is per machine
    if entry.get("delta_gen") is not None:
        c["delta"] = float(entry["delta_gen"])
    # size by the p80 of bits per character, not the median: the median says
    # what the population does and the ladder has to cover single items
    if entry.get("bits_per_char_p80"):
        c["bits_per_char"] = float(entry["bits_per_char_p80"])
        c["bits_per_char_median"] = float(entry.get("bits_per_char", 0)) or None
    # beta_gate is the optimistic value the spending gate uses. The calibrator
    # does not measure one, so derive it from the spread it does record.
    if entry.get("beta_high") and not entry.get("beta_gate"):
        c["beta_gate"] = float(entry["beta_high"])
    c["measured_on"] = entry.get("measured_on")
    return c

CALIB = load_calibration()
BITS_PER_CHAR      = CALIB["bits_per_char"]
BETA_BITS_PER_TERM = CALIB["beta_listing"]
BETA_GATE          = CALIB["beta_gate"]
C0_BITS            = CALIB["c0_listing"]
DELTA_BITS         = CALIB["delta"]

def min_terms_for(t, r=0):
    """Terms asked for, from what this cell's rule is expected to cost.

    Prop 4.1 read forwards: the rule has to beat listing the prefix by delta,
    so L > (K + delta - c0)/beta, and the sequence needs that prefix plus
    K_ADMIN terms to ask a question with.

    It was 24 + 4t, which was too shallow in 18 of 20 cells and keyed on the
    wrong dial: r2t0 needs 49 terms and got 24, because naming tables costs
    characters too. It is keyed on LOAD now. Then v2.22.0 flattened it to 32,
    which was worse.
    """
    # CLAMPED BOTH ENDS. A floor cell has r = -1 and t = -1, so the load was
    # -2, which is not a key, and .get fell through to EXPECTED_CHARS[8]: the
    # shallowest item in the battery was sized as the deepest and asked for 72
    # terms. An alphabet has 26 and was then rejected for repeating, while
    # anything that avoided repeating was rejected for transforming.
    load = min(8, max(0, (r or 0) + (t or 0)))
    chars = EXPECTED_CHARS.get(load, EXPECTED_CHARS[8])
    n = math.ceil(prefix_needed(chars)) + K_ADMIN
    return max(TERMS_MIN, min(TERMS_MAX, n))

def gen_max_out_for(t):
    """Output budget for a generation call. A deep cell needs room, since the
    composer reasons inside the reply."""
    return GEN_MAX_OUT + GEN_MAX_OUT_PER_T * t

def bits_of(chars):
    """A rule's length in characters as an estimate of c_M(p), in bits."""
    return chars * BITS_PER_CHAR

def prefix_needed(shortest_chars, beta=None):
    """The prefix L at which the structural rival overtakes this rule.

    Certification is Prop 4.1: K + delta < beta*L + c0, the rule must beat
    listing the shown prefix by more than delta. Solving for L,

        L > (K + delta - c0) / beta

    The c0 term was missing before, and beta was 6.5 against about 4 measured,
    so the estimate was wrong in both directions at once. Still a proxy: K is
    guessed from characters, and beta is deliberately low so the answer errs
    towards asking for more prefix than needed.
    """
    L = (bits_of(shortest_chars) + DELTA_BITS - C0_BITS) / (beta or BETA_BITS_PER_TERM)
    return max(RIVAL_PREFIXES[0], int(math.ceil(L)))

def terms_needed(shortest_chars):
    """Terms the item needs: the prefix, plus enough left to ask a question."""
    return prefix_needed(shortest_chars) + K_ADMIN

def chars_affordable(n_terms=None, beta=None):
    """The longest rule that can still certify inside the terms available.

    Depends on the item's own length, so it is passed n_terms wherever the
    item is known. At TERMS_MAX = 60, K_ADMIN = 12, beta = 4.1 and c0 = 86 this
    is about 115 characters; at a 32-term sequence it was 67, which is the cap
    that held the last battery down. Passing BETA_GATE gives the optimistic
    version, used by the spending gate.
    """
    L = (n_terms or TERMS_MAX) - K_ADMIN
    return int(((beta or BETA_BITS_PER_TERM) * L + C0_BITS - DELTA_BITS)
               / BITS_PER_CHAR)

def rival_prefixes_for(shortest_chars=None, n_terms=None):
    """Where to ask for rivals on this item.

    The floor is always asked. When the equivalents are in, one more prefix is
    added just above where the item is expected to certify, because a prefix
    nothing was proposed against cannot be certified honestly and a call spent
    at L = 20 on an item that certifies at 6 buys nothing.
    """
    n_terms = n_terms or TERMS_MAX
    Ls = [L for L in RIVAL_PREFIXES if L + 2 <= n_terms]
    if RIVAL_ADAPTIVE and shortest_chars:
        _e = prefix_needed(shortest_chars)
        # one at the estimate, then one further out by the factor. The second
        # is what rescues an item whose per-item beta turns out smaller than
        # the pooled one, which is the common direction.
        wants = [_e + RIVAL_ADAPTIVE_PAD]
        for _k in range(1, max(1, RIVAL_ADAPTIVE_N)):
            wants.append(int(round(_e * (RIVAL_ADAPTIVE_MULT ** _k)))
                         + RIVAL_ADAPTIVE_PAD)
        for want in wants:
            want = min(want, RIVAL_PREFIX_MAX, n_terms - 2)
            if want not in Ls and want > (max(Ls) if Ls else 0):
                Ls.append(want)
    return sorted(set(Ls))

def accept_floor(t, r=0):
    """Fewest terms worth verifying. Never below the longest rival prefix plus
    two, or the rival search has nothing to diverge into."""
    floor = min_terms_for(t, r) - TERMS_SLACK
    if DO_PROGRAMS and RIVAL_PREFIXES:
        floor = max(floor, max(RIVAL_PREFIXES) + 2)
    return floor

PLAN_NAME = next((k for k, v in GRIDS.items() if v is PLAN), "custom")
CELLS = [as_cell(s) for s in PLAN]
# SLOTS is built after the battery is loaded, because FILL needs to know what is
# already there. See build_slots() in the files cell.
SLOTS = []
print(f"\ncomposer         {COMPOSER}")
print(f"plan             {len(CELLS)} cells, target {N_PER_CELL} usable items each"
      + ("  (FILL: only the shortfall is generated)" if FILL else "  (no FILL)"))
print(f"cells            {sorted(set(CELLS))}")
_ld = [r + t for r, t in CELLS]
print(f"load             {min(_ld)} to {max(_ld)}; "
      f"mix {min(r/(r+t) if r+t else 0 for r,t in CELLS):.2f} to "
      f"{max(r/(r+t) if r+t else 0 for r,t in CELLS):.2f}")
print(f"terms            {TERMS_MIN} to {TERMS_MAX}, by load, all of them verified")
print("                 " + "  ".join(
      f"load{L}:{min_terms_for(L,0) if L<7 else min_terms_for(6,min(2,L-6))}"
      for L in range(0, 7)))
print(f"                 n_admin at most {TERMS_MAX - K_ADMIN}, leaving "
      f"{K_ADMIN} terms to ask with")
print(f"terms from       " + ("the generation call itself" if TERMS_FROM == "composer"
      else f"the rule alone: {EXTEND_REPS} runs at temperatures {EXTEND_TEMPS}, "
           f"{EXTEND_WINS} must agree on ALL of them"))
print(f"calibration      {CALIB['source']}"
      + (f", measured {CALIB['measured_on']}" if CALIB.get("measured_on") else ""))
# Say where each number came from. The old wording printed "(default, no file
# yet)" whenever a `bits_per_char_median` key was absent, which the pooled block
# does not carry, so a run reading the file correctly still announced itself as
# running on defaults. That is the one line anybody checks before committing a
# night to a run.
print(f"                 {BITS_PER_CHAR} bits/char"
      + (f" (p80; median {CALIB['bits_per_char_median']})"
         if CALIB.get("bits_per_char_median") else "")
      + f", beta {BETA_BITS_PER_TERM}, c0 {C0_BITS:.0f}, delta {DELTA_BITS:.0f}")
if CALIB["source"] == "defaults":
    print( "                 THESE ARE THE BUILT-IN DEFAULTS. No calibration "
           "file was found, or it had nothing usable in it.")
    print(f"                 The run will work, but on constants measured for "
          f"an earlier battery.")
print(f"budget           a rule of up to about {chars_affordable()} characters can "
      f"certify inside {TERMS_MAX} terms, {chars_affordable(TERMS_MIN)} inside "
      f"{TERMS_MIN}")
print(f"                 L > (K + {DELTA_BITS:.0f} - {C0_BITS:.0f}) / "
      f"{BETA_BITS_PER_TERM}, Prop 4.1")
print(f"rivals           at {RIVAL_PREFIXES}"
      + (f", plus one at the expected L + {RIVAL_ADAPTIVE_PAD}, capped at "
         f"{RIVAL_PREFIX_MAX}" if RIVAL_ADAPTIVE else ""))
print(f"slot order       " + ("shuffled, so a theme shortage does not land "
      "on the deep cells" if SHUFFLE_SLOTS else "grid order"))
print(f"programs         " + ("off" if not DO_PROGRAMS else
      f"{N_EQUIVALENTS} equivalents, {N_RIVALS} rivals at each of {RIVAL_PREFIXES}"))
print(f"components       {'on' if DO_COMPONENTS else 'off'}"
      + (f", for {COMPONENTS_FOR}" if DO_COMPONENTS else ""))
if DO_PROGRAMS:
    print(f"programs for     {PROGRAMS_FOR}"
          + (f", at most {PROGRAMS_MAX_ITEMS} items" if PROGRAMS_MAX_ITEMS else ""))
    print(f"bar              {THIN_EQUIV}+ verified equivalents and "
          f"{THIN_RIVALS}+ rival")
    print(f"give up after    {MAX_PROGRAM_ATTEMPTS} attempts"
          + (", marking the item flawed" if MARK_HOPELESS else ", but keep trying"))
if not DO_GENERATE:
    print("generation       OFF - this run only adds programs and components")
print(f"dedup            {DEDUP_ANY_CELL} across cells, {DEDUP_SAME_CELL} "
      f"within a cell, on rule words only")
print(f"theme filters    " + (", ".join(
      [n for n, v in (("skip closed lists at t>=4", SKIP_LONG_THEMES),
                      ("skip non-elementary at (0,0)", SKIP_HARD_THEMES)) if v])
      or "both off, so every theme is tried"))

## 1c · Persistent state on Kaggle

Reading needs nothing: the loaders glob `/kaggle/input/**` and find the files in
whatever dataset is attached. **Writing** is the problem.
A dataset cannot be
written to directly, and the working directory is wiped between versions, so a
run's output is lost unless it is pushed back as a new version of the dataset.

On Colab this cell does nothing, because Drive is already persistent.

### Setting it up, once

1.
Add Input, Datasets, add **ctest-one-data**.
A Kaggle dataset is a folder of
   files, and adding it makes them readable under `/kaggle/input/ctest-one-data/`.
2.
Add-ons, Secrets: add `KAGGLE_USERNAME` and `KAGGLE_KEY` from your
   `kaggle.json`, and turn the toggle on for this notebook.

With no credentials the notebook still runs and the results still appear in the
version output.
Only state across versions is lost, and the cell says so rather
than failing.

### Why the push works by pattern

`kaggle datasets version` replaces the **whole** dataset with the contents of
the pushed folder, so a file that is in the dataset but not staged is deleted.

So everything matching `ctest_*.json` and `ctest_*.csv` is staged. The log says which were carried.

### When it pushes

A push uploads the whole state, so it does not happen per item.
It happens at
the end of a run, and every `STATE_PUSH_EVERY` items if that is set above zero,
which is worth doing on a long GPU run that might be cut off.


In [ ]:
import subprocess

STATE_PATTERNS = ("ctest_*.json", "ctest_*.csv")

STATE_DATASET = ""        # "owner/slug"; empty derives it from KAGGLE_DATASET
STATE_PUSH_EVERY = 5      # 0 pushes only at the end of a run. On a long
                          # unattended run that is a gamble: save_all() keeps
                          # the working directory current, but the DATASET is
                          # only updated by a push, so a session cut before the
                          # end leaves the accumulated items unreachable from
                          # the next run.

def _kaggle_ready():
    if ENV != "kaggle":
        return False
    missing = []
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        v = _secret(k)
        if v:
            os.environ[k] = v
        else:
            missing.append(k)
    if missing:
        log(f"  state persistence OFF, missing secret(s): {', '.join(missing)}")
        log("     Add-ons -> Secrets -> add them from your kaggle.json, and turn")
        log("     the toggle on for this notebook. Results still appear in the")
        log("     version output; only state across versions is lost.")
        return False
    return True

KAGGLE_OK = _kaggle_ready()

def _slug():
    if STATE_DATASET:
        return STATE_DATASET
    return f"{os.environ.get('KAGGLE_USERNAME','unknown')}/{KAGGLE_DATASET.lower()}"

def state_files():
    """Every file the push should carry, newest copy of each name.

    Local first, since this run may have just written it. Then whatever is in
    the attached dataset and not local, so another notebook's output survives a
    push from this one."""
    chosen, source = {}, {}
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(pat)):
            chosen[os.path.basename(f)] = f
            source[os.path.basename(f)] = "this run"
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(f"/kaggle/input/**/{pat}", recursive=True)):
            b = os.path.basename(f)
            if b not in chosen:
                chosen[b] = f
                source[b] = "carried forward"
    return chosen, source

FRESH_DIR = "/kaggle/working/_fresh"

def _dataset_names():
    """Filenames actually IN the dataset, from the refreshed download.

    NOT from /kaggle/input. The mount is fixed when the session starts, so it
    can be older than the dataset, and comparing the mount against a push
    assembled from the mount compares a thing with itself and can never fire.
    That was the whole failure: the dataset held ctest_metrics.json, the mount
    did not, and a push assembled from the mount deleted it.

    If the refresh did not run or failed, this returns nothing and the guard
    stays quiet, because there is no trustworthy list to compare against.
    """
    out = set()
    if not os.path.isdir(FRESH_DIR):
        return out
    for pat in STATE_PATTERNS:
        for f in glob.glob(os.path.join(FRESH_DIR, pat)):
            out.add(os.path.basename(f))
    return out

STATE_REFRESH = True

def state_refresh():
    """Download the current dataset version into the working directory.

    Cheap, a couple of megabytes, and it removes a whole class of confusion:
    after this the notebook is reading what is IN the dataset rather than what
    happened to be mounted.

    Never raises. A failure leaves /kaggle/input in place, which is the old
    behaviour, and says so.
    """
    if not (STATE_REFRESH and KAGGLE_OK):
        return
    slug = _slug()
    d = FRESH_DIR
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    try:
        r = subprocess.run(["kaggle", "datasets", "download", "-d", slug,
                            "-p", d, "--unzip"],
                           capture_output=True, text=True, timeout=300)
        if r.returncode != 0:
            log(f"  could not refresh from {slug}: "
                f"{(r.stderr or r.stdout).strip()[:120]}")
            log(f"  falling back to whatever the session mounted, which may be "
                f"older than the dataset.")
            return
    except Exception as e:
        log(f"  could not refresh from {slug}: {str(e)[:100]}")
        return

    got, stale = [], []
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(os.path.join(d, pat))):
            b = os.path.basename(f)
            mounted = None
            for p in glob.glob(f"/kaggle/input/**/{b}", recursive=True):
                mounted = p; break
            if os.path.exists(b):
                continue
            shutil.copy(f, b)
            got.append(b)
            if mounted is None:
                stale.append(f"{b} (NOT mounted at all)")
            elif os.path.getsize(mounted) != os.path.getsize(f):
                stale.append(f"{b} (mounted copy is a different size)")
    log(f"  refreshed {len(got)} file(s) from {slug}: {', '.join(sorted(got))}")
    if stale:
        log(f"  the session's mount was OUT OF DATE for: {', '.join(stale)}")
        log(f"  That is why this matters: without the refresh those would have "
            f"been read stale or not at all.")
def state_save(msg="update"):
    """Push a new version of the state dataset. Never raises."""
    if not KAGGLE_OK:
        return
    chosen, source = state_files()
    if not chosen:
        log("  no state files to push yet"); return
    d = "/kaggle/working/_state"
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    for b, f in chosen.items():
        shutil.copy(f, os.path.join(d, b))
    _in_dataset = _dataset_names()
    if not _in_dataset:
        log(f"  note: no refreshed copy of the dataset, so the push cannot be "
            f"checked for deletions. Set STATE_REFRESH = True.")
    _would_lose = _in_dataset - set(chosen)
    if _would_lose:
        log(f"  REFUSING TO PUSH. These are in the dataset and not in this "
            f"push, so the push would DELETE them:")
        for b in sorted(_would_lose):
            log(f"      {b}")
        log(f"  Most likely the session mounted an old version of the dataset. "
            f"STATE_REFRESH pulls the current one at startup; if that failed,")
        log(f"  the log above says why. Nothing has been pushed and nothing is "
            f"lost.")
        return

    carried = [b for b, s in source.items() if s == "carried forward"]
    log(f"  staging {len(chosen)} files"
        + (f", {len(carried)} carried forward from the dataset: "
           f"{', '.join(sorted(carried)[:4])}"
           f"{' ...' if len(carried) > 4 else ''}" if carried else ""))
    slug = _slug()
    json.dump({"title": slug.split("/")[-1], "id": slug,
               "licenses": [{"name": "CC0-1.0"}]},
              open(f"{d}/dataset-metadata.json", "w"))
    try:
        r = subprocess.run(["kaggle", "datasets", "version", "-p", d, "-m", msg,
                            "-r", "zip", "--dir-mode", "zip"],
                           capture_output=True, text=True, timeout=900)
        out = (r.stdout + r.stderr).strip()
        last = out.split("\n")[-1][:160] if out else ""
        if r.returncode == 0 and "error" not in out.lower():
            log(f"  state push: {last}")
        else:
            log(f"  the version push did not go through. Its output was:")
            for _l in out.split("\n")[-6:]:
                if _l.strip(): log(f"    {_l.strip()[:150]}")
            # Only create when the dataset really is absent. "Already in use"
            # means it exists and the version push failed for some other
            # reason, so creating it cannot help.
            missing = ("404" in out or "not found" in out.lower()) and \
                      "already in use" not in out.lower()
            if missing:
                r2 = subprocess.run(["kaggle", "datasets", "create", "-p", d,
                                     "-r", "zip", "--dir-mode", "zip"],
                                    capture_output=True, text=True, timeout=900)
                o2 = (r2.stdout + r2.stderr).strip()
                if "already in use" in o2.lower():
                    log(f"  the dataset exists after all, so the version push "
                        f"failed for another reason. NOTHING WAS PUSHED: "
                        f"download the files from this version's output.")
                else:
                    log(f"  created dataset '{slug}'. Attach it as an input so "
                        f"the next version can read it back.")
                    log(f"  state push: {o2.split(chr(10))[-1][:160]}")
            else:
                log(f"  NOTHING WAS PUSHED. The files are in this version's "
                    f"output and can be downloaded from there.")
    except Exception as e:
        log(f"  state push failed ({str(e)[:80]}). The results are still in this"
            f" version's output.")

stage("PERSISTENT STATE")
if ENV != "kaggle":
    log(f"  not on Kaggle, so nothing to do. Outputs go to {OUTDIR}.")
elif KAGGLE_OK:
    subprocess.run(["pip", "install", "-q", "kaggle"], capture_output=True)
    log(f"  credentials found for '{os.environ['KAGGLE_USERNAME']}'")
    log(f"  state dataset: {_slug()}")
    log(f"  pushes" + (f" every {STATE_PUSH_EVERY} items and" if STATE_PUSH_EVERY
        else "") + " at the end of the run")
_attached = sorted({os.path.basename(os.path.dirname(f))
                    for f in glob.glob("/kaggle/input/*/*")})
if ENV == "kaggle":
    if _attached:
        log(f"  datasets attached: {', '.join(_attached)}")
    else:
        log(f"  NO dataset attached. Add Input -> Datasets -> {KAGGLE_DATASET},")
        log(f"  or the loaders will not find the battery.")
_ch, _src = state_files()
if _ch:
    log(f"  state would carry {len(_ch)} files: {', '.join(sorted(_ch)[:6])}"
        + (" ..." if len(_ch) > 6 else ""))

state_refresh()


## 1d · What the run cost

One row per run, appended to `ctest_runs.csv`.
Wall clock, CPU seconds, GPU
seconds and device, and tokens by model split into prompt, answer and thinking.

Nothing here is a measurement of energy.
It is the record from which energy and
CO2 can be worked out later, from published per-token figures and device power,
so those numbers can be revised without re-running anything.
Keeping the inputs
and deriving the rest is the only way that stays honest.

The numbers are approximate on purpose.
CPU time is process time, so it counts
this notebook and not the API's work.
GPU seconds are wall-clock while a model
is loaded, not utilisation.
Thinking tokens are separated because they are most
of the output on a reasoning model and are charged the same, and they are why a
call truncates with nothing usable returned.


In [ ]:
import csv, resource

RUNS_CSV = "ctest_runs.csv"

# Seed the local copy from wherever the file already lives, before anything is
# appended. Every other file is read back through find(), which globs
# /kaggle/input; this one was only ever appended, so on Kaggle it started empty
# each session and each run wrote a fresh one-row file. Worse, the dataset push
# prefers the local copy, so that one row REPLACED the accumulated history on
# every push. Rows were being lost, not merely not added.
_prev = find(RUNS_CSV)
if _prev and os.path.abspath(_prev) != os.path.abspath(RUNS_CSV):
    shutil.copy(_prev, RUNS_CSV)
    with open(RUNS_CSV) as _f:
        _n = max(0, sum(1 for _ in _f) - 1)
    print(f"run history       {_n} earlier run(s) carried in from {_prev}")
elif _prev:
    with open(RUNS_CSV) as _f:
        _n = max(0, sum(1 for _ in _f) - 1)
    print(f"run history       {_n} earlier run(s) already here")
else:
    print(f"run history       none yet, {RUNS_CSV} will be created")

_T_START = time.time()
_RUN_KEY = datetime.datetime.now().isoformat(timespec="seconds")
_UNITS_SO_FAR = [0]        # work finished, updated as the run goes
_RUN_NOTE = [""]           # what the row should say
_CPU_START = resource.getrusage(resource.RUSAGE_SELF).ru_utime + \
             resource.getrusage(resource.RUSAGE_SELF).ru_stime
_GPU_SECONDS = [0.0]      # wall clock with a model loaded; add to it when one is

def _gpu_name():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return ""

def account(units=0, note=""):
    """Record this run's cost. Never raises: accounting must not stop work.

    UPSERT, not append. The row is keyed on when the run STARTED, so calling
    this repeatedly updates one row instead of adding several, and the numbers
    only grow. It used to be called once at the end, so a run stopped by hand
    or by the session ending wrote nothing at all and its compute vanished from
    the record.
    """
    try:
        cpu = (resource.getrusage(resource.RUSAGE_SELF).ru_utime +
               resource.getrusage(resource.RUSAGE_SELF).ru_stime) - _CPU_START
        tk = dict(TOKENS) if "TOKENS" in globals() else {}
        row = dict(
            run_started=_RUN_KEY,
            run_ended=datetime.datetime.now().isoformat(timespec="seconds"),
            build=BUILD, env=ENV, note=note,
            wall_s=round(time.time() - _T_START, 1),
            cpu_s=round(cpu, 1),
            gpu_s=round(_GPU_SECONDS[0], 1),
            gpu=_gpu_name(),
            units=units,
            model=(COMPOSER if "COMPOSER" in globals()
                   else REF_MACHINE if "REF_MACHINE" in globals() else ""),
            api_calls=tk.get("calls", 0),
            prompt_tokens=tk.get("in", 0),
            answer_tokens=tk.get("answer", 0),
            thinking_tokens=tk.get("think", 0),
            output_tokens=tk.get("out", 0),
            forward_passes=(REF.calls if "REF" in globals()
                            and hasattr(REF, "calls") else 0),
        )
        old_rows, old_cols = [], []
        if os.path.exists(RUNS_CSV):
            with open(RUNS_CSV, newline="") as f:
                rd = csv.DictReader(f)
                old_cols = list(rd.fieldnames or [])
                old_rows = list(rd)
        old_rows = [r for r in old_rows
                    if r.get("run_started") != _RUN_KEY]
        cols = old_cols + [c for c in row if c not in old_cols]
        with open(RUNS_CSV, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols, restval="")
            w.writeheader()
            for r in old_rows + [row]:
                w.writerow({c: r.get(c, "") for c in cols})
        if OUTDIR != ".":
            shutil.copy(RUNS_CSV, OUTDIR)
    except Exception as e:
        log(f"  accounting failed ({str(e)[:70]}), which changes nothing else")


## 3 · The composer and the files

`ask` is the one metered call.
`parse_json` returns the head of the reply when
it fails, so a truncation, a refusal and prose-before-JSON are distinguishable.

All four files are created if absent, so an empty folder is a valid starting
point.


In [ ]:
import math
import logging
from google import genai
CLIENT = genai.Client()

# The SDK logs a warning about automatic function calling on every
# generate_content. It is emitted whether or not tools are configured, so it is
# not evidence that any are. Two token counts settle it: a probe multiplication
# came back with tool_use_prompt_token_count = None and no executable_code part,
# having spent 1,916 tokens thinking about it. There is no interpreter; the
# model simulates, and that is what makes long sequences expensive.
#
# Disabling AFC explicitly should be a no-op AND silence the warning, so the
# counter below is the confirmation. Passing tools=[] is NOT the way to do this:
# it made a clean answer come back with the reasoning leaking into the text.
class _AFCCount(logging.Handler):
    def __init__(self):
        super().__init__(); self.n = 0
    def emit(self, record):
        if "automatic function calling" in record.getMessage().lower():
            self.n += 1
_AFC = _AFCCount()
logging.getLogger("google_genai.models").addHandler(_AFC)
TOKENS = {"in": 0, "out": 0, "calls": 0, "empty": 0, "errors": 0, "truncated": 0,
          "think": 0, "answer": 0, "near_ceiling": 0, "tool_tokens": 0}
_TOK_LOCK, _SAVE_LOCK = threading.Lock(), threading.Lock()

def spend_usd():
    with _TOK_LOCK:
        return (TOKENS["in"]*USD_PER_MTOK_IN + TOKENS["out"]*USD_PER_MTOK_OUT)/1e6

def _text_of(r):
    """Thinking models often return the answer in parts rather than in .text."""
    if getattr(r, "text", None): return r.text
    out = []
    for c in (getattr(r, "candidates", None) or []):
        for p in (getattr(getattr(c, "content", None), "parts", None) or []):
            t = getattr(p, "text", None)
            if t and not getattr(p, "thought", False): out.append(t)
    return "".join(out)

def _finish(r):
    for c in (getattr(r, "candidates", None) or []):
        fr = getattr(c, "finish_reason", None)
        if fr is not None: return str(getattr(fr, "name", fr))
    return ""

def ask(prompt, temperature=1.0, max_out=None, timeout=None,
        model=None, think=None):
    """One metered call. Returns (text, note); note explains an unusual stop.

    `model` defaults to COMPOSER. The evaluator passes a different one per
    examinee, so this cell can stay identical across the notebooks rather than
    each keeping its own copy of the retry and metering logic.
    """
    model = model or COMPOSER
    max_out = max_out or OTHER_MAX_OUT
    if spend_usd() >= MAX_SPEND_USD: return None, "spend cap reached"
    with _TOK_LOCK:
        if TOKENS["in"] + TOKENS["out"] >= TOKEN_BUDGET:
            return None, "token budget reached"
    r = None
    for attempt in range(API_RETRIES):
        try:
            cfg = {"max_output_tokens": max_out, "temperature": temperature,
                   "http_options": {"timeout": int((timeout or API_TIMEOUT_S)*1000)}}
            if DISABLE_AFC:
                cfg["automatic_function_calling"] = {"disable": True}
            # THINKING, ASKED FOR RATHER THAN LEFT TO THE DEFAULT. Running a
            # seven-transformation rule for seventy terms is not a job a model
            # does in one pass, and the default differs by model: at identical
            # settings gemini-3.6-flash reproduced every item and
            # gemini-3.1-flash-lite failed at term 1 on almost all of them,
            # which is the flash model thinking and the lite model not.
            # Comparing two executors means asking both for it.
            if think is not None:
                cfg["thinking_config"] = {"thinking_budget": int(think)}
            r = None
            for drop in ([], ["thinking_config"],
                         ["thinking_config", "automatic_function_calling"],
                         ["thinking_config", "automatic_function_calling",
                          "http_options"]):
                try:
                    r = CLIENT.models.generate_content(
                        model=model, contents=prompt,
                        config={k: v for k, v in cfg.items() if k not in drop})
                    break
                except TypeError:
                    continue
            if r is None:
                raise TypeError("no accepted config shape")
            break
        except Exception as e:
            m = str(e).lower()
            transient = any(w in m for w in ("unavailable","overloaded","timeout",
                "deadline","timed out","rate limit","resource exhausted",
                "408","429","500","502","503","504"))
            with _TOK_LOCK: TOKENS["errors"] += 1
            if attempt == API_RETRIES-1 or not transient:
                return None, f"api error: {str(e)[:120]}"
            time.sleep(API_BACKOFF * (2**attempt))
    if r is None: return None, "no response"
    u, fr = getattr(r, "usage_metadata", None), _finish(r)
    with _TOK_LOCK:
        TOKENS["calls"] += 1
        if u:
            # Thinking is charged against max_output_tokens and is usually most
            # of it: one 9-digit multiplication cost 1,916 thinking tokens
            # against 18 of answer. That is why a call truncates with nothing
            # usable returned, and it is invisible unless counted apart.
            _think = getattr(u, "thoughts_token_count", 0) or 0
            _ans   = getattr(u, "candidates_token_count", 0) or 0
            TOKENS["in"]  += getattr(u, "prompt_token_count", 0) or 0
            TOKENS["out"] += _ans + _think
            TOKENS["think"] += _think
            TOKENS["answer"] += _ans
            if max_out and (_think + _ans) > 0.85 * max_out:
                TOKENS["near_ceiling"] += 1
            if getattr(u, "tool_use_prompt_token_count", None):
                TOKENS["tool_tokens"] += u.tool_use_prompt_token_count
        if fr and fr not in ("STOP", "FINISH_REASON_STOP"): TOKENS["truncated"] += 1
    t = _text_of(r)
    if not t:
        with _TOK_LOCK: TOKENS["empty"] += 1
        return None, f"empty reply (finish reason {fr or 'unknown'})"
    return t, ("" if fr in ("STOP","FINISH_REASON_STOP","") else f"finish reason {fr}")

def parse_json(text):
    """Close a truncated object before parsing, since the output budget covers
    the thinking too. Returns (obj, head) - head says why it failed."""
    if not text: return None, ""
    t = re.sub(r"```(?:json)?", "", text)
    i = t.find("{")
    if i < 0: return None, "no opening brace. reply begins: " + t.strip()[:250]
    frag = t[i:]
    for a in (frag, frag+'"]}', frag+"]}", frag+'"}', frag+"}"):
        try: return json.loads(a), ""
        except Exception: pass
    return None, "unparseable json. fragment begins: " + frag[:250]

def tokens_report():
    with _TOK_LOCK: tk = dict(TOKENS)
    extra = "".join([f", {tk['errors']} api errors" if tk["errors"] else "",
                     f", {tk['empty']} empty" if tk["empty"] else "",
                     f", {tk['truncated']} truncated" if tk["truncated"] else "",
                     f", {tk['near_ceiling']} near the output ceiling"
                     if tk["near_ceiling"] else ""])
    share = tk["think"] / max(1, tk["think"] + tk["answer"])
    log(f"  composer: {tk['calls']} calls, {tk['in']+tk['out']:,} tokens "
        f"(~${spend_usd():.2f}){extra}")
    if tk["think"]:
        log(f"            thinking {tk['think']:,} of {tk['think']+tk['answer']:,} "
            f"output tokens ({share:.0%}); the model has no interpreter, so a long "
            f"sequence is simulated step by step")
    if tk["tool_tokens"]:
        log(f"            {tk['tool_tokens']:,} TOOL tokens: something IS calling "
            f"a tool, which changes what these runs mean")

def parallel(fn, jobs, workers=None, label=""):
    """Run fn over jobs. Results follow jobs, not completion order."""
    workers = workers or MAX_WORKERS
    if workers <= 1 or len(jobs) <= 1:
        return [fn(j) for j in jobs]
    out = [None] * len(jobs)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(fn, j): i for i, j in enumerate(jobs)}
        done = 0
        for f in as_completed(futs):
            i = futs[f]
            try: out[i] = f.result()
            except Exception as e:
                out[i] = None
                log(f"      worker failed on job {i}: {str(e)[:110]}")
            done += 1
            if label and done % max(1, len(jobs)//10) == 0:
                log(f"      {label}: {done}/{len(jobs)}")
    return out

# ------------------------------------------------------------------ files --
def load(name, default):
    p = find(name)
    if not p:
        log(f"  {name}: absent, starting empty"); return default
    log(f"  {name}: loaded from {p}")
    return json.load(open(p))

stage("LOADING")
BAT = load(BATTERY, {"_README":
    "C-test items. Tentative: deterministic, long enough, not continuable by a "
    "dull predictor, with programs attached. Unquestionability is decided by a "
    "separate pricing pass and is not recorded here.", "items": []})
PROG = load(PROGRAMS, {"_README":
    "Programs per item. `equivalents` generate the same sequence and are what a "
    "pricing pass minimises over. `rivals` are grouped by the prefix they were "
    "asked at, and each records where execution says it diverges. The listing "
    "rival is not here: its cost depends on the reference machine.", "items": {}})
THM = load(THEMES, {"_README": "Themes, with a used flag.", "themes": []})
PRO = load(PROMPTS, {"_README": "Prompt text by hash. Every item names the hash "
                     "that produced it.", "prompts": {}})

# Coerce the shapes, in case a file was edited by hand. The battery holds a
# LIST of items and the programs a DICT keyed by id; getting those the wrong way
# round fails much later with an unhelpful message.
if not isinstance(BAT.get("items"), list): BAT["items"] = []
if not isinstance(PROG.get("items"), dict): PROG["items"] = {}
ITEMS = BAT.setdefault("items", [])
log(f"\n  {len(ITEMS)} items in the battery, "
    f"{len(PROG.get('items', {}))} with programs")
log(f"  {len(THM.get('themes', []))} themes, "
    f"{sum(1 for t in THM['themes'] if not t.get('uses') and not t.get('used'))}"
    f" unused, {sum(max(0, THEME_REUSE - _uses(t)) for t in THM['themes'])}"
    f" offers left counting the slack")

_renamed = 0
for _b in BAT.get("items", []):
    if "rule" in _b and "seed" not in _b:
        _b["seed"] = _b.pop("rule"); _renamed += 1
if _renamed:
    log(f"  renamed `rule` to `seed` on {_renamed} items from an earlier battery")

# ------------------------------------------------------- exclusions -------
# Nothing is ever deleted. An item that should not be used is MARKED, so the
# decision stays visible, reversible and auditable. Recovered from the v5
# pipeline, where this vocabulary was defined:
#   excluded = null         in use
#            = "duplicate"  another item measures the same thing
#            = "quota"      bucket full, held back for balance, nothing wrong
#            = "flawed"     defective: ambiguous rule, wrong ground truth
#            = "stale"      measured by an older pipeline, re-measure before use
# An exclusion already on an item is never overwritten, so a decision made by
# hand always beats an automatic one.
EXCLUSIONS = ("duplicate", "quota", "flawed", "stale", "questionable")

CALIBRATE_FROM = 50   # valid measured items before any of this applies. Below
CALIBRATE_CAP_R = 0
CALIBRATE_CAP_T = 2   # t is cheaper to ask for: the validity rate is flat
                      # from t = 0 to t = 9, between 45% and 67%, so the
                      # correction costs little there.
CALIBRATE_FLOOR = -1  # and no lower. t=-1 and r=-1 each add a paragraph to
                      # what t=0 and r=0 already say; there is no -2 to say.

def _fit_line(pairs):
    """Least squares through (asked, measured). Returns intercept and slope."""
    n = len(pairs)
    if n < 8:
        return 0.0, 1.0
    mx = sum(a for a, _ in pairs) / n
    my = sum(m for _, m in pairs) / n
    den = sum((a - mx) ** 2 for a, _ in pairs)
    if den <= 0:
        return 0.0, 1.0
    b = sum((a - mx) * (m - my) for a, m in pairs) / den
    if b < 0.2:            # a flat or inverted fit inverts the correction
        return 0.0, 1.0
    return my - b * mx, b

def calibration(items):
    """What to ask, per axis, from what past asks delivered.

    Only VALID items with a measurement count: an item that never reproduced
    its own sequence says nothing about what the composer delivers, and an
    unmeasured one says nothing at all. The measurement is averaged over the
    machines that have one, since they agree to about a fifth of a point and
    the drift being corrected is ten times that.
    """
    R, T = [], []
    for b in items:
        if not is_valid(b):
            continue
        rs = [b.get(f"r_hat_{m}") for m in MACHINE_TAGS]
        ts = [b.get(f"t_hat_{m}") for m in MACHINE_TAGS]
        rs = [x for x in rs if x is not None]
        ts = [x for x in ts if x is not None]
        if not rs or not ts:
            continue
        R.append((b.get("want_retrieval") or 0, sum(rs) / len(rs)))
        T.append((b.get("want_transformation") or 0, sum(ts) / len(ts)))
    live = len(R) >= CALIBRATE_FROM
    return dict(n=len(R), live=live,
                r=_fit_line(R) if live else (0.0, 1.0),
                t=_fit_line(T) if live else (0.0, 1.0))

def ask_for(target, fit, axis="t"):
    """The value to ask for, so that `target` is what comes back.

    Capped either side of the target and floored, then returned as a fraction:
    the caller draws an integer from it, so that over a run the asks average
    where they were aimed instead of at the nearest whole number.
    """
    a, b = fit
    want = (target - a) / b
    _cap = CALIBRATE_CAP_R if axis == "r" else CALIBRATE_CAP_T
    want = max(target - _cap, min(target + _cap, want))
    return max(CALIBRATE_FLOOR, want)

def draw(x, rng=random):
    """An integer from a fractional ask: 1.6 is 1 forty per cent of the time.

    Rounding would throw the fraction away every time and the measured grid
    would centre on the nearest integer rather than where it was aimed.
    """
    lo = math.floor(x)
    return lo + (1 if rng.random() < x - lo else 0)

def usable(b):
    """In use, unless something says otherwise."""
    return not b.get("excluded")

MACHINE_TAGS = ("qwen3-8b", "qwen3-14b")

def is_valid(b):
    """The seed, run cold a second time, reproduced what the first run made.

    Recorded two ways and both mean this: generation writes `valid` as it
    admits an item, the standalone extension pass writes
    `repro.seed_cold.verdict`.
    """
    return (b.get("valid") is True
            or (b.get("repro", {}).get("seed_cold") or {}).get("verdict")
            == "reproduced")

def measured_cell(b):
    """The cell an item LANDED in, averaged over the machines that measured it.

    None where it has not been measured, which is every item until it has been
    priced and then typed.
    """
    rs = [b.get(f"r_hat_{m}") for m in MACHINE_TAGS]
    ts = [b.get(f"t_hat_{m}") for m in MACHINE_TAGS]
    rs = [x for x in rs if x is not None]
    ts = [x for x in ts if x is not None]
    if not rs or not ts:
        return None
    return (int(round(sum(rs) / len(rs))), int(round(sum(ts) / len(ts))))

def asked_cell(b):
    """The cell an item was ASKED for. What the dedup compares on.

    cell_of used to mean this, and was changed to mean where the item landed,
    which silently broke the same-cell duplicate check: it began comparing a
    measured cell against an asked one, and returning None for invalid items
    so that nothing was ever compared against them at all.
    """
    return (b.get("want_retrieval"), b.get("want_transformation"))

def cell_of(b):
    """Where an item counts towards coverage.

    THE CELL IT LANDED IN, not the one it was asked for, and only if it is
    valid. Counting the ask counted the wrong thing twice over: an invalid item
    filled a gap it should not, and an item asked at r0t0 that came back at
    r1t2 was counted against r0t0. The whole r=2 column above t=2 read as
    covered with nothing valid measured in it, while r0t2 was asked for twice
    and held nine items that had drifted in.

    An item not yet measured counts where it was asked, since that is the best
    guess available and the alternative is that a run generating six items sees
    none of them and asks for the same cell again.
    """
    if not is_valid(b):
        return None
    return measured_cell(b) or (b.get("want_retrieval"),
                                b.get("want_transformation"))

def mark(b, value, note):
    """Mark an item, unless it already carries a decision. A mark made by hand
    always beats an automatic one, so an existing value is never overwritten."""
    if b.get("excluded"):
        return False
    b["excluded"] = value
    b["excluded_note"] = note
    b["excluded_by"] = BUILD
    b["excluded_on"] = datetime.date.today().isoformat()
    return True

def battery_counts():
    """Usable items per cell, and what was excluded and why."""
    have, gone = collections.Counter(), collections.Counter()
    for b in ITEMS:
        _c = cell_of(b)
        if _c is not None and (usable(b) or FILL_COUNTS_EXCLUDED):
            have[_c] += 1
        if b.get("excluded"):
            gone[b["excluded"]] += 1
    return have, gone

# ------------------------------------------------------ duplicates -------
_STOP = {"the", "each", "term", "terms", "subsequent", "starting", "start",
         "with", "next", "and", "for", "from", "list", "listed", "order",
         "their", "that", "this", "into", "then", "given", "using", "based"}

def rule_words(s):
    """Content words of a rule, exactly as v5 computed them.

    The stop list and the regex are v5's, unchanged, because the 0.35 and 0.80
    thresholds were measured with THIS signature. A stricter stop list looks
    like an improvement and silently breaks the calibration: dropping words such
    as "letters" or "previous" sent the overlap on a pair v5 measured at 0.40 to
    0.62 down to 0.00."""
    return {w for w in re.findall(r"[a-z]{3,}", str(s).lower()) if w not in _STOP}

def _overlap(a, b):
    return len(a & b) / len(a | b) if (a and b) else 0.0

def duplicate_of(rule, cell, pool=None):
    """The item this rule measures the same thing as, or None.

    The bar depends on the cell: DEDUP_SAME_CELL within one, DEDUP_ANY_CELL
    across. Both were measured on the v5 battery.

    The `mechanism` field is deliberately NOT used. It looks like the natural
    signal and is misleading: on the v5 battery it scored 0.50 to 1.00 on pairs
    that had to stay SEPARATE, because a generic phrase like 'ordered set
    lookup' describes half the battery, and only 0.12 to 0.33 on the real
    duplicates.

    Not caught: the same family with different constants. The fix for those is
    on the prompt side, in the rules already held that the composer is shown."""
    w = rule_words(rule)
    if not w:
        return None, 0.0
    near, near_jr = None, 0.0
    for b in (pool if pool is not None else ITEMS):
        # excluded items are compared too. If a new candidate repeats a shape
        # that was set aside, it should not come back under a new id; unmark
        # the original instead.
        jr = _overlap(w, rule_words(b.get("seed")))
        if jr > DEDUP_ANY_CELL:
            return b, jr
        if asked_cell(b) == cell and jr > DEDUP_SAME_CELL:
            return b, jr
        # remember the closest miss, so a candidate that came near the line is
        # visible in the log rather than silently reported as zero
        if jr > near_jr:
            near, near_jr = b, jr
    return None, near_jr

def build_slots():
    """Slots to run. With FILL, PLAN is a target and only the shortfall is
    asked for, so a run can be repeated until the grid is populated."""
    have, gone = battery_counts()
    if gone:
        log(f"  excluded and not counted: {dict(gone)}"
            + ("  (counted anyway: FILL_COUNTS_EXCLUDED)" if FILL_COUNTS_EXCLUDED else ""))
    slots, short, full = [], [], []
    for c in CELLS:
        n = N_PER_CELL - have.get(c, 0) if FILL else N_PER_CELL
        if n <= 0:
            full.append(c); continue
        short.append((c, have.get(c, 0), n))
        slots += [c] * n
    if FILL:
        log(f"  target {N_PER_CELL} per cell across {len(CELLS)} cells "
            f"= {len(CELLS)*N_PER_CELL} items")
        log(f"  battery already covers {len(full)} cells; "
            f"{len(short)} short by {len(slots)} items in total")
        if short:
            worst = sorted(short, key=lambda x: -x[2])[:8]
            log("  biggest gaps: " + ", ".join(f"r{c[0]}t{c[1]} has {h}, needs {n}"
                                               for c, h, n in worst))
        if not slots:
            log("  the target is met everywhere. Raise N_PER_CELL, widen PLAN, "
                "or set FILL = False.")
    if SHUFFLE_SLOTS:
        random.Random(0).shuffle(slots)
    if MAX_SLOTS and len(slots) > MAX_SLOTS:
        import collections as _c
        _before = _c.Counter(slots)
        slots = slots[:MAX_SLOTS]
        _after = _c.Counter(slots)
        if not globals().get("_SLOTS_LOGGED"):
            globals()["_SLOTS_LOGGED"] = True
            log(f"  MAX_SLOTS = {MAX_SLOTS}: attempting {MAX_SLOTS} of the "
                f"{sum(_before.values())} short. The rest keep their gap and "
                f"come back next run.")
            log(f"  cells covered tonight: {len(_after)} of {len(_before)}, "
                f"{min(_after.values())} to {max(_after.values())} slots each")
    return slots

def save_all(msg=""):
    with _SAVE_LOCK:
        for name, obj in ((BATTERY, BAT), (PROGRAMS, PROG),
                          (THEMES, THM), (PROMPTS, PRO)):
            json.dump(obj, open(name, "w"), indent=1)
            if OUTDIR != ".": shutil.copy(name, OUTDIR)
    if msg: log(f"   saved. {msg}")

SLOTS = build_slots() if DO_GENERATE else []

def register(name, text):
    """Record a prompt by hash and return the hash."""
    h = hashlib.md5(text.encode()).hexdigest()[:8]
    e = PRO.setdefault("prompts", {}).setdefault(
            h, dict(name=name, chars=len(text), builds=[], text=text))
    if BUILD not in e["builds"]: e["builds"].append(BUILD)
    return h


Optional, and normally skipped.
`STALE_BEFORE_HASH` names the generation prompt
you want to keep; every item made under any other one is marked `stale`.

It exists for a restart. Marking the old items reopens
every cell.

Nothing is deleted and nothing is lost.
`stale` means measured or made by an
older pipeline and worth re-doing, not wrong, and one line reverses it.

**And the old items still do useful work while set aside.** The three checks
that push the composer towards variety all read the whole battery rather than
only the usable part: `avoid_line` shows it rules already held, `worn_out_line`
counts which mechanisms keep recurring, and `duplicate_of` rejects a candidate
that repeats a shape.
Those were filtering on `usable` until now, which silenced
them exactly when a battery was being restarted, which is when they matter
most.

So after this cell: `FILL` sees an empty battery and plans every cell, while the
composer is still told about all 71 shapes it should not repeat.


In [ ]:
# Set to a prompt hash to keep, or None to skip.
# Check what is there first:
#   collections.Counter(b.get("provenance",{}).get("gen_prompt") for b in ITEMS)
STALE_BEFORE_HASH = None

if STALE_BEFORE_HASH:
    seen = collections.Counter(b.get("provenance", {}).get("gen_prompt")
                               for b in ITEMS)
    log(f"  prompt hashes in the battery: {dict(seen)}")
    if STALE_BEFORE_HASH not in seen:
        raise SystemExit(f"no item was made under {STALE_BEFORE_HASH!r}. "
                         f"Pick one of {sorted(k for k in seen if k)}.")
    n = 0
    for b in ITEMS:
        if b.get("provenance", {}).get("gen_prompt") != STALE_BEFORE_HASH:
            if mark(b, "stale", f"made under a prompt other than "
                                f"{STALE_BEFORE_HASH}; superseded"):
                n += 1
    save_all()
    log(f"  marked {n} items stale")
    log(f"  {sum(1 for b in ITEMS if usable(b))} of {len(ITEMS)} usable now, so "
        f"FILL will reopen the cells they occupied")
    log(f"  they are still shown to avoid_line, worn_out_line and duplicate_of,")
    log(f"  so the composer is told about every shape it should not repeat")
else:
    log("  STALE_BEFORE_HASH is None, so nothing was set aside")

# Recomputed here, after any staling, because setting items aside reopens the
# cells they occupied and that has to be reflected in what gets planned.
if DO_GENERATE:
    SLOTS = build_slots()
    if not SLOTS:
        log("\n  NOTHING TO PLAN. Every cell in PLAN already holds "
            f"{N_PER_CELL} usable item(s).")
        log("  Raise N_PER_CELL, widen PLAN, or set items aside with "
            "STALE_BEFORE_HASH.")


## 4 · Themes

A theme is two to five words of subject matter, no rule and no verb.
It says
what the item is about; the composer discards it once it has supplied a seed.

Themes are persisted with a `used` flag so a later run does not reinvent them.

Two properties are asked for and stored.
`generative` says the subject can go on
past a couple of dozen items, which a closed list of seven cannot.
`elementary`
marks the ones a child knows, which is where the easy anchors come from: the
weakest examinee being measured has to be able to solve something.

The finite-list guard uses `generative` when it is there and a small word list
when it is not, and skips such a theme for a cell needing $t \ge 4$ before any
generation call is spent.


In [ ]:
THEMES_PROMPT = """List {k} THEMES for sequence puzzles. A theme is two to five words naming
subject matter only - no rule, no instruction, no verb.

Good: "primes and ordinals" / "chess openings" / "Beaufort scale" /
"karate belt order" / "Greek letters and their names"
Bad: "Take the planets in order" - that is a rule, not a theme.

WELL-KNOWN TEXTS COUNT, and {text_share} of your list should be them. The
words of a famous passage, in order, are a sequence like any other, and a
reader who does not have the text cannot continue while one who does can,
which is what makes it retrieval:

  "opening words of A Tale of Two Cities"   it, was, the, best, of, times
  "the Gettysburg Address"                  four, score, and, seven
  "the first verse of Jabberwocky"          twas, brillig, and, the, slithy
  "Humpty Dumpty"     "the Lord's Prayer"    "To be or not to be"
  "the opening of Moby Dick"    "Mary Had a Little Lamb"

Only texts first published before 1900 and ORIGINALLY IN ENGLISH. A modern
translation is a new work whatever the age of what it translates. Nursery
rhymes, hymns, speeches, poems, proverbs and novel openings are all fair.

Spread them widely. No more than THREE from any one area. Reach for games,
crafts, cuisine, folklore, liturgy, sport, trades, law, farming, textiles,
navigation and children's rhymes at least as readily as for mathematics,
astronomy and notation systems, which are over-represented already.

Mark each one:
- "generative": true if the subject can keep going past thirty items (things
  counted, measured, graded, ranked, notated, built up in stages). False for a
  closed list of a handful, like the days or the rainbow. A text is generative
  when it runs to thirty words or more, which most passages do.
- "elementary": true if a child knows it ("counting", "the alphabet", "even
  numbers"). About one in six should be elementary: a battery needs an easy end.

Avoid these, already used:
{used}

Answer in strict JSON, nothing else:
{{"themes": [{{"theme": "...", "domain": "<one word>", "generative": true,
  "elementary": false}}, ...]}}"""
THEMES_HASH = register("THEMES_PROMPT", THEMES_PROMPT)

FINITE_WORDS = {"day","days","colour","colours","color","colors","rainbow",
  "season","seasons","finger","fingers","month","months","planet","planets",
  "vowel","vowels","compass","suit","suits","primary","sense","senses",
  "continent","continents","ocean","oceans"}

def _words(t):
    return {w for w in re.findall(r"[a-z]{4,}", t.lower())
            if w not in {"each","give","list","from","with","that","then","into",
                         "order","sequence","term","next","using","their"}}

_THEME_CALLS = [0]

def theme_pool(want, exclude=None):
    """`want` themes for this run, drawn in tiers and fetched only when short.

    THREE TIERS, in order.

      1  themes no item has used yet
      2  the SLACK: themes already used, up to THEME_REUSE items each
      3  new themes from the composer, and only when the first two are dry

    The slack is what makes a large battery reachable. The composer does not
    have an endless supply of subjects whose members are words: over fourteen
    calls on a cold start the near-duplicate rate climbed from 0% to 67% and the
    word yield fell from 19 a call to 3, so 500 items sourced one theme each
    would drag the store into obscure corners, and an examinee failing on FEPA
    P-scale grit ratings has failed a lookup rather than a reasoning step.

    A theme can carry several genuinely different items: the weekdays as a plain
    cycle at r1t0, under four operations at r1t4, paired with another table at
    r2t2. The duplicate check is on RULE WORDING, 0.35 within a cell and 0.80
    across, so it already guards the thing that matters, which is two items that
    ARE the same rather than two items that share a subject.

    THE SLACK IS OFF ON THE FIRST RUN. A run that starts from an empty store
    stays one item per theme, so the battery it produces can be scrapped and
    regenerated and will come out the same shape. The slack opens only once a
    store exists, which is the second run onwards.
    """
    exclude = exclude or set()
    first_run = len(THM["themes"]) < THEME_STORE_MIN

    def _tier(max_uses):
        out = [t for t in THM["themes"]
               if _uses(t) < max_uses and t["theme"] not in exclude]
        if THEME_PREFER_NEW:
            # newest first. The store is a file in append order, so without
            # this a top-up sits at the end and is handed out last: 60 new
            # themes joining 240 unused old ones would barely appear in a run
            # of 17 slots.
            out.sort(key=lambda t: str(t.get("added", "")), reverse=True)
        return out

    fresh = _tier(1)
    slack = [] if first_run else [t for t in _tier(THEME_REUSE) if _uses(t) >= 1]
    free = fresh + slack

    if first_run and THEME_REUSE > 1:
        log(f"  the store holds {len(THM['themes'])} themes, under "
            f"THEME_STORE_MIN = {THEME_STORE_MIN}, so this is a first run and "
            f"each theme carries ONE item.")
        log(f"  The slack of {THEME_REUSE} opens on the next run, once there is "
            f"a store to draw on. A first battery stays reproducible.")

    if len(free) >= want:
        log(f"  {len(fresh)} unused themes"
            + (f" and {len(slack)} with slack left" if slack else "")
            + f", enough for {want}, no call needed")
        return free[:want]

    rounds = 0
    while len(free) < want and rounds < THEMES_MAX_ROUNDS \
            and _THEME_CALLS[0] < THEMES_MAX_CALLS:
        rounds += 1
        _THEME_CALLS[0] += 1
        used = {t["theme"] for t in THM["themes"]}
        need = min(THEMES_PER_CALL, max(20, (want - len(free)) + 10))
        log(f"  {len(fresh)} unused"
            + (f", {len(slack)} with slack" if slack else "")
            + f", asking for {need} more (round {rounds} of at most "
              f"{THEMES_MAX_ROUNDS})")
        txt, note = ask(THEMES_PROMPT.format(k=need, text_share=TEXT_SHARE,
                        used=", ".join(sorted(used)[-60:]) or "(nothing yet)"),
                        temperature=1.4, max_out=PROG_MAX_OUT)
        d, head = parse_json(txt)
        if d is None:
            log(f"  themes call failed: {note or head}")
            break
        added, seen = 0, [_words(t) for t in used]
        for x in d.get("themes", []):
            if not isinstance(x, dict): continue
            t = str(x.get("theme","")).strip().lower()
            if not (3 <= len(t) <= 60) or t in used: continue
            w = _words(t) or {t}
            if any(len(w & p)/max(1, len(w | p)) > 0.5 for p in seen): continue
            THM["themes"].append(dict(theme=t,
                domain=str(x.get("domain","")).strip().lower()[:20],
                generative=bool(x.get("generative", True)),
                elementary=bool(x.get("elementary", False)),
                uses=0, added=datetime.date.today().isoformat()))
            seen.append(w); used.add(t); added += 1
        log(f"  {added} new themes kept after removing near-duplicates")
        if added and "judge_themes" in globals():
            globals()["judge_themes"](f"{added} new")
        save_all()
        if added and "state_save" in globals():
            try:
                globals()["state_save"](f"{BUILD}: +{added} themes")
                log(f"  pushed, so the themes survive an abort")
            except Exception as e:
                log(f"  could not push the themes: {str(e)[:70]}")
        if added == 0:
            log(f"  the call added nothing, so asking again would not help")
            break
        fresh = _tier(1)
        slack = [] if first_run else [t for t in _tier(THEME_REUSE)
                                      if _uses(t) >= 1]
        free = fresh + slack

    if len(free) < want:
        log(f"  {len(free)} themes against {want} asked for"
            + (f", and the run's budget of {THEMES_MAX_CALLS} theme calls is "
               f"spent" if _THEME_CALLS[0] >= THEMES_MAX_CALLS else "")
            + ". A slot that finds no theme is skipped, so some cells will be "
              "thinner. That is a theme shortage, not a generation failure.")
        if first_run and THEME_REUSE > 1:
            log(f"  The slack would have added "
                f"{sum(THEME_REUSE - 1 for t in THM['themes'])} more offers, "
                f"and opens on the next run.")
    return free[:want]

THEME_SKIPS = collections.Counter()   # why themes were passed over

def theme_fits(th, r, t):
    """False when the theme cannot carry the cell, so no call is spent on it.

    Both tests are now off by default. They rejected 45 candidates against 6
    planned in one run, which left half the slots empty and two frontier cells
    with nothing.

    The length test existed to stop a seven-element theme being asked for 24
    terms. That no longer applies: the terms come from executing the rule, so
    the world does the work and the theme only supplies a subject. A short
    theme can sit on a deep mechanism perfectly well.

    The elementary test existed so the easy corner got material a child knows.
    It also threw away most of the pool at (0,0). The prompt already says
    trivial items are wanted there, which is the better place for it.

    Set SKIP_LONG_THEMES or SKIP_HARD_THEMES back to True if short themes start
    producing short sequences, or if the easy corner stops being easy."""
    if SKIP_LONG_THEMES and t >= 4:
        if th.get("generative") is False: return False
        if _words(th["theme"]) & FINITE_WORDS: return False
    if SKIP_HARD_THEMES and r == 0 and t == 0 and not th.get("elementary", False):
        THEME_SKIPS["not elementary, and r0t0 needs it"] += 1
        return False

    if THEME_NUMERIC_SKIP and th.get("terms_are") == "numbers":
        if random.random() < THEME_NUMERIC_SKIP:
            THEME_SKIPS["members are numbers"] += 1
            return False
    return True


## 5 · The dials, in words

$r$ and $t$ become instructions.

In [ ]:
# The two exemptions. Both rest on the same principle as the `control` category
# in DECOMPOSE: what every reader already has, or what is needed merely to HAVE
# r sources at all, is not part of the description.
#
# FREE ORDERINGS are never knowledge. Counting, the letters of the alphabet in
# order, the digits. Universally available, so naming one retrieves nothing.
# Everything else nameable is knowledge: the zodiac, NATO, the planets, the
# weekdays. Until now retrieval_text(0) offered "counting, or the letters" as
# r=0 material and retrieval_text(1) offered "'counting', 'the alphabet'" as
# r=1 material, so one thing satisfied two cells and the bottom of the axis
# measured nothing.
#
# FREE JOINS are never operations. Concatenating or alternating between the r
# sources IN THEIR NATURAL ORDER, at most r-1 times. That is the minimum glue
# needed to have r sources at all, so charging for it puts every cell with
# r > t+1 out of reach and leaves the grid a triangle: r2t0 would need two
# tables joined by nothing, which is barely constructible, and it measured 2.5
# operation spans where none were asked. A join stays free only while it
# computes nothing. The moment WHICH source to draw from depends on the index,
# a value or a condition, it is an operation again.
FREE_ORDERINGS = ("the counting numbers", "the letters of the alphabet in "
                  "order", "the digits")

def retrieval_text(r, t=None):
    """What the item should be built from, at retrieval depth r.

    It takes t as well, because at r = 0 the old wording asked for a situation
    that RUNS, and that fired at every r=0 cell including t=0 where the
    transformation dial says none.
    """
    # BELOW ZERO, as for t: the r=0 text with a paragraph added, so a cell
    # asked for 0 or more reads exactly as it did before.
    _none = ("\nAND NOTHING TO LOOK UP AT ALL, not even a short list named in "
             "the rule. The term is built from counting or from the letters "
             "and from nothing else."
             if r < 0 else "")
    if r <= 0:
        if t is not None and t <= 1:
            return ("NO memorised table. Counting, the letters and the digits "
                    "are free here: everyone has them, so using one keeps this "
                    "item at r=0. At this depth the item IS the plain thing, "
                    "so do not reach for machinery to fill the cell: a shallow "
                    "item is the anchor the difficulty scale needs at the "
                    "bottom") + _none
        return ("NO memorised table, and nothing that has to be remembered. "
                "Counting, the letters and the digits are free"
                + (", and so is a world you define in the rule itself, which "
                   "at this depth is usually the better answer"
                   if (t or 0) >= WORLD_FROM else "")
                + ". Arithmetic on the index is the "
                "obvious way to fill a cell that forbids knowledge, and it is "
                "why so much of this battery is sequences of numbers") + _none
    # The table's part in a world, said only where a world is on offer. The
    # suggestion itself lives in world_line, which is keyed on t alone.
    _world_at_r = ("\nIF YOU BUILD A WORLD HERE, the table supplies what its "
                   "cells or tokens are CALLED, and it costs characters the "
                   "update also needs. So keep the world SIMPLER than you "
                   "would without one: one moving part, one named step. A "
                   "queue, a sort and a tie-break on top of a table will not "
                   "fit the budget and will not survive the determinism "
                   "check.\n" if (t or 0) >= WORLD_FROM else "")

    if r == 1:
        return _world_at_r + ("EXACTLY ONE memorised table, so standard that naming it takes "
                "VERY FEW WORDS: 'the weekdays', 'the planets', 'the NATO "
                "alphabet', 'the chemical elements', 'the opening words of "
                "Moby Dick'. A well-known text counts, and needs a few more "
                "words to name than a list does; what matters is that naming "
                "it is enough, so do not qualify it and do not spell it "
                "out.\n"
                "COUNTING, THE ALPHABET AND THE DIGITS DO NOT COUNT. They are "
                "free to everyone, so a rule built on them retrieves nothing "
                "and belongs at r=0. It must also be REALLY NEEDED: a rule that "
                "could be followed by someone who had never heard of the theme "
                "is an r=0 rule wearing a costume, and nine candidates were "
                "thrown away on one run for exactly that. Arithmetic on the "
                "index is not retrieval")
    if r == 2:
        return _world_at_r + ("EXACTLY TWO memorised tables, each nameable in a few words, "
                "and BOTH really needed: remove either and the sequence should "
                "become uncomputable. Counting, the alphabet and the digits do "
                "not count as either of them.\n"
                "JOINING THEM IS FREE. Running one after the other, or "
                "alternating between them in their natural order, costs no "
                "transformation and leaves this item at whatever t was asked "
                "for, including t=0. What is NOT free is choosing between them "
                "by the index, by a value, or by any condition: that is an "
                "operation and belongs on the other dial")
    return (f"EXACTLY {r} memorised tables, combined. Name each as briefly as "
            f"it can be named. Counting, the alphabet and the digits do not "
            f"count. Running them in sequence or alternating between them in "
            f"natural order is free")

FLOORS = [
    "the whole numbers, counting up from any start",
    "the whole numbers, counting down from some start above two hundred",
    "the whole numbers, counting up in steps of the same size each time",
    "the days of the calendar, running on through the months so that no date "
    "comes twice, written however you like",
    "the chemical elements by atomic number, as symbols, starting somewhere "
    "past the tenth",
    "the chemical elements by atomic number, as names, starting somewhere "
    "past the tenth",
    "the words of a very well known song or nursery rhyme, one word a term, "
    "in the order they are sung, long enough not to run out",
    "the letters of a very well known passage or title, one letter a term, "
    "long enough not to run out",
    "the years, counting up from some year in the last century",
]

def transform_text(t, floor=None):
    """What "t transformations" asks for.

    Rewritten in 2.23.0. The old wording ended, at t >= 4, with "or ONE RULE
    APPLIED REPEATEDLY TO ITS OWN OUTPUT", which offers recursion as a
    SUBSTITUTE for composition. They are not the same quantity: a recursive
    rule has iteration depth, while the annotator counts operation SPANS, so a
    rule that runs one update a hundred times scores one or two however deep it
    goes. The escape hatch let the composer satisfy a deep cell with a shallow
    rule, and it did: measured operations across the battery run 1, 2 or 3 with
    a median of 2 whatever the dial says, and t predicts them at +0.29 against
    +0.82 for r and knowledge.

    Recursion is still welcome AS ONE of the operations. It is not a way of
    reaching a count without the others.

    The count is now something the composer must be able to LIST, and the reply
    carries that list, so what was asked and what was delivered can be compared
    before anything is executed.
    """
    # BELOW ZERO. The calibration asks for less than none when less than none
    # is what it takes to land at none: asking for t=0 has produced a measured
    # 1.06 on average. The text is the t=0 text with a paragraph added, so a
    # cell asked for 0 or more reads exactly as it did before and the items
    # already generated stay comparable.
    _floor = ("\nAND ONE THING FURTHER: THE TERM IS THE THING ITSELF, with "
              "nothing accompanying it. No fixed word in front of it, no "
              "label, no unit: if every term would begin with the same "
              "characters, those characters should not be there. The term "
              "advances ONE step each time through THIS order and no other:\n"
              "    " + (floor or FLOORS[0]) + "\n"
              "and nothing else in the item changes or is added. Not two "
              "things advancing together, not a count that carries, not a "
              "cycle. The theme you have been given IS this order: there is "
              "nothing else to be about."
              if t < 0 else "")
    if t <= 0:
        return ("NO transformation at all. The nth term IS the thing, given as "
                "it stands, in its natural order. Nothing is computed from it, "
                "nothing is rearranged, nothing carries from one term to the "
                "next. A shallow cell is not a lesser item, it is the anchor "
                "the difficulty scale needs at the bottom, and an item that "
                "quietly does two or three operations here is not that anchor. "
                "If the cell also asks for two tables, running one after the "
                "other or alternating between them in natural order is FREE "
                "and leaves you at t=0. That is how a cell with tables and no "
                "transformation is built") + _floor
    if t == 1:
        return ("EXACTLY ONE transformation, of your own choosing, and nothing "
                "else")
    # The count in words, for every value the calibration can ask for. It
    # stopped at SIX, so a cell asked for seven read "EXACTLY SIX (7)", which
    # is what the seven items at t=7 were told.
    plural = {2: "TWO", 3: "THREE", 4: "FOUR", 5: "FIVE", 6: "SIX",
              7: "SEVEN", 8: "EIGHT", 9: "NINE", 10: "TEN"}.get(t, str(t))
    return (f"EXACTLY {plural} ({t}) DISTINCT transformations. Distinct means "
            f"you can name them one by one and each does something the others "
            f"do not: squaring, summing digits, reversing, taking a remainder, "
            f"sorting, filtering, rendering as symbols. "
            f"JOINING THE RETRIEVED TABLES DOES NOT COUNT: running one after "
            f"another, or alternating between them in natural order, is free "
            f"and is not one of the {t}. Choosing between them by the index or "
            f"by a condition IS one of them.\n"
            f"IF YOU BUILD A SMALL WORLD, COUNT THE STEPS INSIDE ITS UPDATE. A "
            f"world is not one transformation. An automaton with a single named "
            f"rule is one step, but a world where tokens move, test whether a "
            f"cell is taken, jump past it and hand over to the next token is "
            f"four. To reach {t}, write an update with {t} steps and name each "
            f"of them. Do NOT take a one-step automaton and wrap it in extra "
            f"arithmetic to make the count: that is what happened on the last "
            f"run, the wrapper is what two runs then disagreed about, and 36 "
            f"candidates died there. Naming each step is also how you close the "
            f"ties, which is what the determinism check is testing. "
            f"Recursion counts as ONE of the {t}, not as a way of reaching {t} "
            f"without the rest, so \"apply this step over and over\" is one "
            f"transformation however many times it runs. Naming a table is "
            f"RETRIEVAL and does not count here at all. If you cannot list {t} "
            f"of them, this item does not belong in this cell")

# Shapes for a little world, one picked at random per slot.
#
# Rebalanced. The first list had twelve entries and FIVE of them were queues:
# a shop with a queue, a tournament, an assembly line, a queue split and
# rejoined, a ledger settled in order. Two more were rings. So a run came back
# with 21 queues in 62 seeds and the composer looked unimaginative when it was
# picking faithfully from what it had been handed.
#
# Grouped so the spread is visible and stays visible when anything is added.
# At most two of any shape.
WORLDS = [
    # a line or a row, where position is the state
    "tokens on a small board, blocked by each other",
    "a walk along a row that leaves a mark where it has been",
    # a ring, where the ends join
    "a row of lamps switched by a rule about their neighbours",
    "a ring of cells where each takes its value from its neighbours",
    # a stack, last in first out, which behaves nothing like a queue
    "a stack where each item may push two smaller ones before it leaves",
    "a pile that topples when it grows past a height, spilling into its "
    "neighbours",
    # a queue, first in first out. TWO, not five
    "a shop with stock and a queue of customers",
    "a queue split and rejoined by a rule on its length",
    # a string rewritten in place, which grows without any container
    "a word rewritten by a fixed substitution on every symbol",
    "a growing structure, each stage built from the one before",
    # counters and carries, where the state is numbers rather than positions
    "a row of counters where one overflowing adds to the next",
    "an odometer whose wheels turn at different rates",
    # containers and amounts
    "vessels of different sizes poured between each other",
    "a boat with a capacity and some pairs that cannot be left alone together",
    # ordering, where the work is comparison rather than movement
    "a hand of cards sorted by repeated passes, one swap a term",
    "a tournament where winners advance and losers are placed",
    # a grid, two dimensions rather than one
    "a small grid where a marker turns according to what it lands on",
]


## 5 · The two screens on the terms

Both are arithmetic on the terms alone.
They cost nothing, they need no model,
and they do not shift when the reference machine changes.

### The repeat test

`periodicity` finds the shortest cycle the terms settle into, allowing a
transient before it, and counts the terms beyond the first cycle that confirm
it.
A candidate is rejected when a whole second copy is present, since the terms
then contain a verbatim repeat and the sequence has stopped saying anything new.
Anything weaker is recorded as `period` and `period_start` and not judged: the
threshold would be arbitrary, and the place it matters is the administration,
where an item with period $p$ from term $s$ can only ask something if
$n_{\text{admin}} < s + p$.

### Collapse, extended

The annotator already records `collapse_at`, the index from which a constant, a
cycle or an arithmetic run reproduces the rest. That check reads whole terms, so
it misses an item whose *parts* are predictable.

| item | predictor | share of continuations it got |
|---|---|---|
| `v11-0027` ordinals | tail-split with a vocabulary | 94% |
| `v11-0030` shotgun gauge | run-cycle | 83% |
| `v11-0024` gas giants | none, but 84% of each term is scaffolding | — |

So four predictors run, all of them dull by design.
None looks for a rule; if
one of them can continue the sequence, the item is not asking a question.

**vocabularies** lets a field counting in words be recognised, using ordered
lists of ordinals, cardinals and letters.
Without this, `fifth, sixth, seventh`
looks like unrelated strings.

**tail-split** cuts each term at its last separator and extends the two halves
independently, which survives a head that is itself hyphenated.
This is what
catches `twenty-eighth-second`.

**run-cycle** handles a leading token repeated a growing number of times with a
cycling remainder, allowing terms where the count is zero, which is why
`c c c c d` is caught although its first five terms have no `c` at all.

Arithmetic is fitted on a trailing window rather than the whole prefix, so one
odd opening term does not hide a steady trend.

`free_share` is separate from all of this.
It is the fraction of a term's
characters sitting in fields a dull model can fill.
It does not reject anything,
because a high share is not fatal: `galileo jupiter (0)` at 84% still carries an
integer that no dull predictor gets.
It is recorded so a sampling pass can
prefer items that are mostly signal, and so the paper can say how much of an
item's apparent width is scaffolding.

A self-test runs at import, covering both directions, and stops the notebook if
the detectors misbehave.


In [ ]:
# --------------------------------------------------------- the repeat test --
def periodicity(terms):
    """The cycle the terms settle into, allowing a leading transient.
    Returns None, or {period, start, extra} where extra counts the terms beyond
    the first cycle that confirm it. extra >= period means a whole second copy."""
    n = len(terms)
    found = []
    for s in range(n):
        for p in range(1, n - s):
            block = terms[s:s+p]
            if all(terms[i] == block[(i - s) % p] for i in range(s + p, n)):
                found.append((p, s, n - (s + p)))
                break                          # shortest p for this transient
    if not found:
        return None
    found.sort(key=lambda x: (-x[2], x[0], x[1]))   # most confirmed, then shortest
    p, s, extra = found[0]
    return dict(period=p, start=s, extra=extra)

# ------------------------------------------------------ collapse, extended --
ORD_WORDS = ("first second third fourth fifth sixth seventh eighth ninth tenth "
    "eleventh twelfth thirteenth fourteenth fifteenth sixteenth seventeenth "
    "eighteenth nineteenth twentieth twenty-first twenty-second twenty-third "
    "twenty-fourth twenty-fifth twenty-sixth twenty-seventh twenty-eighth "
    "twenty-ninth thirtieth thirty-first thirty-second").split()
CARD_WORDS = ("one two three four five six seven eight nine ten eleven twelve "
    "thirteen fourteen fifteen sixteen seventeen eighteen nineteen twenty "
    "twentyone twentytwo").split()
VOCABS = [ORD_WORDS, CARD_WORDS, list("abcdefghijklmnopqrstuvwxyz")]
SEP_RE = re.compile(r"(\s+|-|/)")

def _split(term):
    return [p for p in SEP_RE.split(term) if p != ""]

def _numeric(vals):
    """(numbers, formatter) when the values are integers or an ordered word list."""
    if all(re.fullmatch(r"\d+", v) for v in vals):
        return [int(v) for v in vals], (lambda i: str(i) if i >= 0 else None)
    for vocab in VOCABS:
        if all(x in vocab for x in vals):
            return ([vocab.index(x) for x in vals],
                    (lambda i, _v=vocab: _v[i] if 0 <= i < len(_v) else None))
    return None, None

def _fit(vals):
    """Next value under the dullest model that fits, or None."""
    if not vals: return None
    if len(set(vals)) == 1:
        c = vals[0]; return lambda: c
    nums, fmt = _numeric(vals)
    if nums is not None:
        w = nums[-GUESS_WINDOW:]
        d = {b - a for a, b in zip(w, w[1:])}
        if len(d) == 1:                                   # arithmetic run
            nxt = w[-1] + d.pop()
            return lambda: fmt(nxt)
        runs, cur = [], 1                                 # a step every L terms
        for a, b in zip(nums, nums[1:]):
            if a == b: cur += 1
            else: runs.append((a, cur)); cur = 1
        runs.append((nums[-1], cur))
        if len(runs) >= 3:
            lens = {L for _, L in runs[:-1]}
            deltas = {b - a for (a, _), (b, _) in zip(runs, runs[1:])}
            if len(lens) == 1 and len(deltas) == 1:
                L, dl = lens.pop(), deltas.pop()
                val, run = runs[-1]
                nxt = val + dl if run >= L else val
                return lambda: fmt(nxt)
    for p in range(1, 8):                                 # short cycle
        if len(vals) >= 2*p and all(vals[i] == vals[i % p] for i in range(len(vals))):
            nxt = vals[len(vals) % p]; return lambda: nxt
    return None

def _p_fields(prefix):
    fs = [_split(t) for t in prefix]
    if len({len(f) for f in fs}) != 1: return None
    out = []
    for j in range(len(fs[0])):
        fn = _fit([f[j] for f in fs])
        if fn is None: return None
        v = fn()
        if v is None: return None
        out.append(v)
    return "".join(out)

def _p_tailsplit(prefix):
    """Cut at the last separator; extend head and tail on their own."""
    for sep in ("-", " "):
        heads, tails, ok = [], [], True
        for t in prefix:
            i = t.rfind(sep)
            if i <= 0: ok = False; break
            heads.append(t[:i]); tails.append(t[i+len(sep):])
        if not ok: continue
        fh, ft = _fit(heads), _fit(tails)
        if fh is None or ft is None: continue
        h, tl = fh(), ft()
        if h is None or tl is None: continue
        return h + sep + tl
    return None

def _p_runcycle(prefix):
    """A leading token repeated a growing number of times, then a cycling tail."""
    firsts = {t.split()[0] for t in prefix if t.split()}
    for tok in firsts:
        counts, tails = [], []
        for t in prefix:
            parts = t.split()
            k = 0
            while k < len(parts) and parts[k] == tok: k += 1
            if k == len(parts) and k > 0: k -= 1     # nothing but the token
            counts.append(k); tails.append(" ".join(parts[k:]))
        if len(set(counts)) < 2: continue
        fc, ft = _fit([str(c) for c in counts]), _fit(tails)
        if fc is None or ft is None: continue
        c, tl = fc(), ft()
        if c is None or tl is None: continue
        return (" ".join([tok] * int(c)) + " " + tl).strip()
    return None

def _p_lastdiff(prefix):
    """Term k+1 is term k with a fixed edit at one end."""
    if len(prefix) < 4: return None
    edits = []
    for a, b in zip(prefix[-4:], prefix[-3:]):
        if b.startswith(a):   edits.append(("suf", b[len(a):]))
        elif b.endswith(a):   edits.append(("pre", b[:len(b)-len(a)]))
        else: return None
    if len(set(edits)) != 1: return None
    kind, e = edits[0]
    return prefix[-1] + e if kind == "suf" else e + prefix[-1]

PREDICTORS = [("field", _p_fields), ("tail", _p_tailsplit),
              ("run", _p_runcycle), ("grow", _p_lastdiff)]

def guessability(terms, start=None):
    """How far a dull predictor gets. Returns {best, by, scores, tries}."""
    start = GUESS_START if start is None else start
    if len(terms) <= start + 1:
        return dict(best=0.0, by=None, scores={}, tries=0)
    hits = {k: 0 for k, _ in PREDICTORS}
    tries = 0
    for k in range(start, len(terms)):
        pre, truth = terms[:k], terms[k]
        tries += 1
        for label, fn in PREDICTORS:
            try:
                if fn(pre) == truth: hits[label] += 1
            except Exception:
                pass
    by = max(hits, key=lambda k: hits[k])
    return dict(best=hits[by]/tries, by=(by if hits[by] else None),
                scores={k: round(v/tries, 3) for k, v in hits.items()}, tries=tries)

def free_share(terms):
    """Fraction of each term's characters in fields a dull model can fill."""
    fs = [_split(t) for t in terms]
    if len({len(f) for f in fs}) != 1: return 0.0
    free = total = 0
    for j in range(len(fs[0])):
        col = [f[j] for f in fs]
        n = sum(len(v) for v in col); total += n
        if _fit(col) is not None: free += n
    return round(free / total, 3) if total else 0.0

def collapse_at(terms):
    """The pipeline's original check, kept: index from which a constant, cycle or
    arithmetic run reproduces the rest. Returns (index, kind) or (None, None)."""
    def simple(seq):
        if len(seq) < 4: return None
        if len(set(seq)) == 1: return "constant"
        for p in range(1, len(seq)//2 + 1):
            if all(seq[i] == seq[i-p] for i in range(p, len(seq))):
                return f"period-{p}"
        try:
            v = [float(x) for x in seq]
            if len({round(b-a, 9) for a, b in zip(v, v[1:])}) == 1:
                return "arithmetic"
        except (ValueError, TypeError):
            pass
        return None
    for k in range(max(0, len(terms) - 4)):
        lab = simple(terms[k:])
        if lab: return k, lab
    return None, None

def guess_reject(load):
    """The share of continuations a dull predictor may get, by load = r + t.

    An item at load 0 is meant to be nearly guessable; one at load 8 must not
    be. Squared, so the relaxation concentrates at the very bottom rather than
    loosening cells that already work."""
    slack = max(0.0, (GUESS_EASY_LOAD - load) / GUESS_EASY_LOAD) ** 2
    return GUESS_REJECT + (GUESS_REJECT_EASY - GUESS_REJECT) * slack

def screen_terms(terms, min_terms=None, load=None):
    """Both screens. Returns (ok, reason, info) with info recorded either way.

    `min_terms` is the length floor, and there is no sensible default now that
    it varies by cell: the caller passes `accept_floor(t, r)`. Zero means do not
    check the length here, which is what the generation phase wants when the
    sequence has not been produced yet. The self-test passes its own, because
    its fixtures exist to exercise the DETECTORS and are shorter than any real
    item needs to be."""
    floor = 0 if min_terms is None else min_terms
    info = {"n_terms": len(terms), "n_distinct": len(set(terms))}
    if len(terms) < floor:
        return False, f"only {len(terms)} terms, below the {floor} worth verifying", info

    per = periodicity(terms)
    # A period is only RECORDED when the evidence covers a full period, which
    # is the same bar the rejection below uses. Recording it on less was
    # misleading in the one direction that matters: an item whose run lengths
    # grow, so that it cannot be periodic at all, was reported as "period 30
    # from term 4" on the strength of seven comparisons, and the summary told
    # the reader to stop administering it at term 34. With a handful of
    # distinct terms and long runs, seven consecutive matches at the tail of a
    # sequence happen by chance.
    if per and per["extra"] >= max(REPEAT_EVIDENCE, per["period"]):
        info.update(period=per["period"], period_start=per["start"],
                    period_confirmed_by=per["extra"])
        if True:
            return False, (f"verbatim repeat: period {per['period']} from term "
                           f"{per['start']+1}, confirmed by {per['extra']} terms"), info

    if len(terms) < GUESS_START + 4:
        # too short for the collapse predictors to mean anything. With
        # TERMS_FROM = "extend" this is the sample, and the real screen runs
        # once the sequence has been produced.
        return True, "", info
    g = guessability(terms)
    info["guess_best"] = round(g["best"], 3)
    info["guess_by"] = g["by"]
    info["guess_scores"] = g["scores"]
    info["free_share"] = free_share(terms)
    ca, ck = collapse_at(terms)
    info["collapse_at"], info["collapse_kind"] = ca, ck

    bar = GUESS_REJECT if load is None else guess_reject(load)
    info["guess_bar"] = round(bar, 2)
    if g["best"] > bar:
        return False, (f"continuable without the rule: the '{g['by']}' predictor "
                       f"gets {g['best']:.0%} of the continuations"), info
    return True, "", info

# ------------------------------------------------------------- self-test ----
def _screen_selftest():
    meal = (["dessert","soup","salad","soup","main","soup","salad","soup","soup",
             "salad","soup","main","soup","salad","soup"] * 2)
    ordinals = ("first-second fifth-fourth sixth-second seventh-fourth "
        "eighth-second ninth-fourth tenth-second eleventh-fourth twelfth-second "
        "thirteenth-fourth fourteenth-second fifteenth-fourth sixteenth-second "
        "seventeenth-fourth eighteenth-second nineteenth-fourth twentieth-second "
        "twenty-first-fourth twenty-second-second twenty-third-fourth "
        "twenty-fourth-second twenty-fifth-fourth twenty-sixth-second "
        "twenty-seventh-fourth twenty-eighth-second").split()
    shotgun = [(("c " * (n // 5)) + "dosvh"[n % 5]).strip() for n in range(26)]
    def tm(n):
        s = "0"
        while len(s) < n: s += "".join("1" if c == "0" else "0" for c in s)
        return list(s[:n])
    zeck = ["1","2","3","3+1","5","5+1","5+2","8","8+1","8+2","8+3","8+3+1","13",
            "13+1","13+2","13+3","13+3+1","13+5","13+5+1","13+5+2","21","21+1",
            "21+2","21+3","21+3+1","21+5"]
    cases = [
        ("verbatim repeat",        meal,      False),
        ("ordinals, tail-split",   ordinals,  False),
        ("shotgun, run-cycle",     shotgun,   False),
        ("strict alternation",     ["a","b"]*15, False),
        ("counting to 30",  [str(i) for i in range(1, 31)], False),  # arithmetic
        ("thue-morse binary",      tm(30),    True),
        ("zeckendorf",             zeck,      True),
    ]
    bad = []
    for name, seq, want in cases:
        ok, reason, _ = screen_terms(seq, min_terms=12, load=8)
        if ok != want:
            bad.append(f"{name}: got {ok}, wanted {want}" + (f" ({reason})" if reason else ""))
    log(f"  screens: {len(cases)-len(bad)} of {len(cases)} self-test cases correct")
    for b in bad: log(f"    FAILED  {b}")
    if bad: raise SystemExit("the term screens are not behaving, fix before running")
_screen_selftest()


## 7 · Generation

Most of what
was cut was negative instruction: do not repeat yourself, do not put state
fragments in the terms, do not label the equivalents, do not do arithmetic on
character codes.

What is left is what the composer cannot be told by a rejection: the theme, the
two dials, determinism, the term format, and one worked contrast showing that a
term should mean something.

Two fields are new in the reply.

`gloss` is one English sentence saying what the rule does.
The rule itself is
compact code because that is what gets priced, and the component pass needs
English to read.
Asking for both in the same call costs nothing.

`answer_hint` is where the composer says which term first becomes impossible to
guess without the rule.
It is advice, not a decision, and the pricing pass may
disagree.
It is recorded because it is free and because it is the composer's
own view of where its item starts asking something.


In [ ]:
def _used_line(n=28):
    """The subjects the battery already draws on, for the prompt.

    The composer returns to the same few unless it is told which they are:
    Mary Had a Little Lamb, the NATO alphabet and the Greek letters each came
    back after the battery already held them.
    """
    import collections, re as _re
    # OPERATIONS ARE NOT SUBJECTS. A list built from word counts is mostly
    # verbs -- square, reverse, multiply -- and banning those would constrain
    # the mechanism when what repeats is the CONTENT: the NATO alphabet, the
    # chemical elements, the rainbow.
    STOP = set("the a an of for each and or to in is are with then take its "
               "their nth n th by at as from that this it on into every term "
               "terms number numbers next start first second word words list "
               "order sum add value letter letters step steps emit "
               "digits digit reverse reversed count square squared multiply "
               "cycle cycling append appended modulo mod remainder replace "
               "length name names based followed left right plus minus "
               "index indexed position running total alternate join".split())
    c = collections.Counter()
    # BAT["items"], not BAT: BAT is the whole document and iterating it gives
    # the top-level keys, which are strings.
    for b in (BAT.get("items") or []):
        for w in _re.findall(r"[a-z][a-z-]{3,}", (b.get("seed") or "").lower()):
            if w not in STOP: c[w] += 1
    top = [w for w, k in c.most_common(n) if k >= 2]
    if not top: return ""
    return ("\nALREADY IN THE BATTERY, so do not build on these: "
            + ", ".join(top) + ".\n")

GEN_PROMPT = """THEME: {theme}

Invent one sequence puzzle around it. The theme says what the item is about; how
the sequence is built is yours.

USE THE THEME. Its words, its list, its ordering must appear in the sequence
and a reader must be able to see them there. If the theme will not carry a
puzzle, say so in one line and stop; do not write about something else. Four
of the last six items arrived with a theme they had nothing to do with --
asked for rowing boat seats and given the Academy Awards, asked for Catholic
holy orders and given a nursery rhyme -- and an item that ignores its theme
comes back to the same handful of subjects every time.
{used_line}

HARD REQUIREMENT: one rule must fix every term, so anyone who knows the rule
computes the same sequence with no ambiguity. Ties broken by a stated convention
(alphabetical, lowest first) so every step is forced.

BUILD IT FROM: {retrieval_txt}
COMBINED WITH: {transform_txt}
{world_line}{low_t_example}
THE TERMS MUST MEAN SOMETHING. A reader should be able to check a step. Compare:

  GOOD  a boat carries two; the fox may not be left with the goose, nor the
        goose with the grain; each term names the alphabetically first legal
        load: goose, empty, fox, goose, grain, empty, goose
  BAD   s = s[1:] + chr(97 + (ord(s[0]) + ord(s[1])) % 26)
        giving: redyellowblue, edyellowbluev, dyellowbluevh

The second is deterministic and deep and still worthless: nobody can check a
step or recognise the answer.

HOW TO WRITE THE RULE

Plain English. Reach for notation only where it makes that part shorter AND more
precise: a modular index, an arithmetic expression, a fraction. It is not a
program and nobody runs it, so it needs no imports, no `def`, no `yield`, no
`while True:`, no variable declared only so a later line can use it.
{shape_line}
  GOOD  count from 1; each term is n/64 reduced
  BAD   from fractions import Fraction; n = 1
        while True: yield str(Fraction(n, 64)); n += 1
{shape_example}

The English is SHORTER than the Python in both, once the scaffolding an
interpreter needs is gone. But plain English is not a licence to ramble: forty
words where six would do is just as bad.

HOW LONG THE RULE MAY BE: about {char_budget} characters.

This is not a style note. An item is only usable if its rule is CHEAPER than
writing out the terms it is shown; otherwise "just copy the {n_show} terms above
and carry on" is as good an answer as the rule, the item has two defensible
continuations, and it is thrown away. At {n_terms} terms that line sits at about
{char_budget} characters.

The trap is describing a MACHINE. A machine takes a paragraph to set up and the
paragraph is the cost:

  BAD   Start with a row of 5 lamps in state [off, off, on, off, off]. At each
        step, a lamp turns on if it was off and had exactly one on neighbour,
        turns off if it was on and both neighbours were on, and otherwise
        stays as it was. Emit the number of lamps that are on.
  GOOD  rule 22 on 5 cells from one lit centre; each term counts the lit cells

Same sequence, 270 characters against 74. The first spells out what the second
names, and naming is free because the reader already knows it.

Depth is not length, and depth is not arithmetic. A rule can be short and still
take real work to follow. Every rule below is inside the budget, and each is
shown WITH ITS TERMS, because a rule for a small world is not readable without
them: it has to say what a term IS.

  GOOD  n o's in a row of n+3 dashes slide one per term;              (82)
        at each end gain an o and reverse
          o---, -o--, --o-, ---o, ---oo, --oo-, -oo--, oo---, ooo--- ...
  GOOD  all balanced bracket strings, shortest first then             (78)
        alphabetical with ( before )
          (), (()), ()(), ((())), (()()), (())(), ()(()), ()()() ...
  GOOD  towers of hanoi with 5 discs by the shortest solution;        (82)
        each term is the three pegs
          12345|-|-, 2345|-|1, 345|2|1, 345|12|-, 45|12|3, 145|2|3 ...
  GOOD  a knight from a1 always takes the alphabetically first        (88)
        legal move to an unvisited square
          a1, b3, a5, b7, c5, a4, b2, c4, a3, b1, c3, a2, b4 ...
  GOOD  the planets outward, each followed by its first letter        (93)
        repeated as many times as its position
          mercury m, venus vv, earth eee, mars mmmm ...
  GOOD  a(1)=1; a(n) = a(n-1) + the digit sum of a(n-1)               (47)
          1, 2, 4, 8, 16, 23, 28, 38, 49, 62 ...

NOT EVERYTHING SHOULD BE NUMBERS. A term can be a pattern, a position, a board,
a word, a state of a small world. Bare integers are the easiest thing to invent
and the least interesting to look at, and counting is tempting because counting
never runs out. If your terms are integers, ask whether they could be the THING
instead of a count of it.

  BAD   a(1)=2; a(n) = a(n-1) + n + the number of ones in binary n
          2, 5, 9, 14, 20, 27, 35 ...

Short, deterministic, deep, and still a poor item: the terms carry no meaning, a
reader cannot check a step by eye, and the theme has vanished.

A SMALL WORLD IS A GOOD SHAPE AND IT HAS THREE WAYS TO FAIL. All three of these
are deterministic, and none of them works:

  BAD   an ant on white turns right, on black turns left, flips the cell, and steps
        It never says what a TERM is. The grid? The ant's square? Its heading?
        A rule for a world must name the string each step produces.

  BAD   cars on a ring .>>.>. all advance one cell if the cell ahead is empty
          .>>.>., .>.>.>, >.>.>., .>.>.>, >.>.>. ...
        It settles into a two-term cycle after one step. A sequence that has
        closed its cycle carries no more information and is thrown away. Most
        tidy little simulations do this. The world must GROW, or its period
        must be longer than the sequence.

  BAD   n from 1; n marks written as i, in groups of five separated by a space
          i, ii, iii, iiii, iiiii, iiiii i ...
        The terms grow without bound and pass {max_len} characters before the
        sequence is long enough. Each term has to stay short.

So a small world earns its place when a term is a fully specified string, the
terms stay short, and the world grows or has a long period.

A CELLULAR AUTOMATON is the reliable way to get all three, and it needs no
knowledge at all, so it fits the cells that forbid a table:

  GOOD  the rule-30 cellular automaton on a ring of 15 cells,          (104)
        starting from one x in the middle and o elsewhere; each
        term is the row
          oooooooxooooooo, ooooooxxxoooooo, oooooxxooxooooo,
          ooooxxoxxxxoooo, oooxxooxoooxooo, ooxxoxxxxoxxxoo ...

Sixty terms, no repeat, fifteen characters each, and the rule fits in a hundred
characters because naming the automaton REPLACES describing it. Any two symbols
will do, x and o or # and . or v and m, and the choice is yours; say which,
since the reader cannot guess it.

USE RULE 30, and a ring of NINE CELLS OR MORE. This is not taste. On a ring
started from one lit cell, rule 30 comes back round only after hundreds of
terms at every width from 9 up, while rule 90 returns after 8 terms at width 9
and 16 at width 15, and rule 110 after 9 at width 9 and 41 at width 17. A ring
of 7 closes in 4 terms whichever rule you pick. A sequence that has repeated is
thrown away, so the wrong rule or a ring two cells too narrow loses the item.

Naming an automaton is ONE transformation, however many ticks it runs for. So
it fills a cell asking for one, and in a deeper cell you may apply further
transformations to what it emits, up to the number that cell asks for.

THE TWO SCREENS THAT KILL WORLDS. These are not advice, they are the checks
your candidate is put through, and on the last run they threw away five worlds
for every one that survived.

DETERMINISM. Your rule is handed to three separate readers who see nothing but
the rule, and their sequences must agree on ALL {n_terms} terms. Any case you
leave open, they will settle differently. In a world that means: which token
moves first, what happens when the target cell is taken, what happens at the
edge or the wrap, what happens when two things could move at once, what happens
when nothing can move. Write the answer to each into the rule. "Tokens block on
collision" is not an answer; "if the target is taken, the token stays and the
next token moves" is. This is what 36 candidates failed on last time, the
largest single cause of death.

REPETITION. Your sequence is rejected if any block of it repeats, so a world
that settles into a cycle is thrown away however good it is. Last run lost
candidates at period 12, period 10 and period 1. A finite ring ALWAYS cycles
eventually: {n_terms} terms of a fifteen-cell automaton is usually safe, ten
cells is not, and a handful of tokens shuffling on a short track will come back
round in twenty. Either make the ring wide, or let the world GROW, or check by
hand that nothing has repeated by term {n_terms}.

The same shape works for anything with a small state and a named update: a
sandpile toppling, a queue where the front item moves back by its own size, a
row of tokens sliding and reversing. What makes them affordable is naming the
update rather than spelling it out.

At the easy end the same budget is generous, not tight:

  GOOD  the days of the week in order, repeating                               (40)
  GOOD  a, a, b, b, c, c and so on through the alphabet                        (47)

Those are correct answers to a shallow cell, not failures. An item that a weak
solver can finish is as necessary as one that defeats a strong one.

So: SHORT TO STATE, and NOT CONTINUABLE WITHOUT DOING THE WORK, where the work
may be simulating a small world, recalling a table, or arithmetic. Length buys
nothing. If your rule needs a paragraph, you have described machinery instead of
naming it, or built something too fiddly to make a good item.

The test is whether two careful readers would compute the same sequence, in as
few characters as you can manage.

NAME WHAT THE READER ALREADY KNOWS. A standard table is not part of your state.
Anyone reading this knows the NATO alphabet, the chemical elements, the planets
in order. Writing one out is like defining "seven" before using it: it makes the
rule enormous and measures nothing, because it is knowledge the reader brings.

  GOOD  the nato alphabet in order, each word followed by the square of its
        position
  BAD   w = ['alpha','bravo','charlie','delta','echo','foxtrot','golf','hotel',
             'india','juliett', ... 16 more]; term = w[i] + ' ' + str(i*i)

Three cautions on naming.
  Say WHICH FORM, since a name is only free if it pins the exact strings: "the
    US presidents by SURNAME in term order", and then use `juliett` not
    `juliet`. A rule naming a table but using a private variant of it cannot be
    reproduced by anyone.
  Name DATA, not OPERATIONS. `best_load(a, capacity=2, safe=...)` is not a
    shorter rule, it is an incomplete one: no such function exists and nobody
    can follow it. Write the operation out.
  Spell out only a table nobody could know: an arbitrary substitution you
    invented, a made-up set of capacities. Those are part of the rule. The
    alphabet is not.

WHERE DIFFICULTY COMES FROM: not opacity. An item is good when a reader shown
the rule says "of course" and a reader shown only the terms cannot get there.
Small state, obvious step, enough interaction that the sequence cannot be
continued by pattern-matching.

Difficulty is DEPTH, not WIDTH. A harder item is not one with more printed in
each term; it is one where the next term is harder to predict from the ones
before. Items asked to be hardest came back with four or five fields per term
and were the easiest to continue, because most fields held a counter or a fixed
label. The ones that worked print one or two things.

Before answering, say the rule to yourself a second way in one line. If you
cannot, it is too tangled and the item will be thrown away later when nobody
else can restate it either.

CONSTRAINTS
- The sequence must run to at least {min_terms} terms without repeating. This is the single most
  common reason an item is thrown away, and it is not a stylistic preference.
  An item is only worth asking if writing the shown prefix out costs more than
  stating the rule, and a prefix earns about seven bits a term. A rule taking
  200 bits to state needs about thirty terms before the item is usable at all,
  and a rule twice that needs sixty. Short sequences are wasted work however
  good the rule is.
  So pick a situation that can keep going, and keep going. If you find yourself
  running out before {min_terms}, the situation was too small: start again with
  one that accumulates.
- The sequence must NOT run out and start repeating itself: prefer a state that
  grows or accumulates over one that only moves.
- Each term under {max_len} characters and lowercase. Any characters you like
  except a COMMA, a DOUBLE QUOTE or a BACKSLASH. Terms are read separated by
  commas, so a comma inside one leaves the reader unable to tell where it ends.
  So a term may be drawn: -------o-------, ....x...., |--o--o--|, [--o--] and
  ---> are all fine, and drawing the row is often the clearest way to show a
  small world.
  A term is still what the examinee READS. Do not emit a piece of your own
  state: s[3] and state = {{...}} are not terms, however few characters they
  take.
- KEEP THE TERM NARROW. One or two things, not four. Do not print:
    the position in the sequence      (n1, n2, n3 ... the reader can count)
    a label that never changes        (3b 1 4b 0 5b 0 6b 0 -> 1 0 0 0)
    anything already derivable        (a running total of what has been shown)
  Every one of those is free to guess, so it pads the item without making it
  harder, and it also makes the item impossible to find rivals for, because the
  padding pins the sequence down.
  Compare, from real items:
    WORKS   1, 3, 2, 5, 7, 6, 4        a knockout bracket; the term IS the state
    WORKS   1908, 1912, 1924, 1916     tokens on a track; the term is where one landed
    WORKS   1 white, 2 white, 1 green 1 red   a stack of chips; it grows and shrinks
    FAILS   3b 1 4b 0 5b 0 6b 0        94% of the characters never change
    FAILS   n1 2.00mm us0 fine         the index twice, and a threshold on the third
    FAILS   20 beats 10 rank 1         "beats" and "rank" every time, and rank counts
- If you write the rule as a function or a generator, make what it returns or
  yields the term itself, already a string. A rule whose output is a list, a
  dict or an index expression cannot be checked and will be thrown away.
- The first five terms must not give the answer away.
- Simple items are wanted too. When the dials ask for little of both, answer
  with a rule of two to five words ("count from one", "powers of two"). Those
  are the anchors the scale needs, not failures.
LIST WHAT YOU DID. `operations` names each transformation, one entry each, and
its length must match the number asked for above. `tables` names each memorised
set or ordering the rule needs, and its length must match the retrieval asked
for. These are the two dials this item is being built to, so if you cannot fill
the lists to the right lengths, the rule is in the wrong cell and should be
rebuilt rather than relabelled.

  seed        n cubed, digits summed, times 4, reversed, last digit
  operations  ["cube", "sum digits", "multiply by 4", "reverse", "last digit"]
  tables      []

  seed        the planets outward, each followed by its first letter repeated
              as many times as its position
  operations  ["repeat the initial by position"]
  tables      ["the planets in order"]

Counting is not a transformation: "count from 1" is how the terms are indexed,
not something done to them. Naming a table is retrieval and belongs in
`tables`. Applying one step over and over is ONE operation however many times
it runs.
{avoid_line}
{terms_note}

Answer in strict JSON, nothing else:
{{"seed": "<the rule>",
  "gloss": "<one English sentence saying what it does>",
  "mechanism": "<short generic phrase for how the sequence is built>",
  "operations": ["<each transformation, two or three words>", ...],
  "tables": ["<each memorised table or ordering the rule needs>", ...],
  "answer_hint": <the 1-based index of the first term that cannot be guessed
                  without the rule>,
  "sequence": ["a few terms, to show you checked", ...]}}"""

_KEYS = {"theme","retrieval_txt","transform_txt","world_line","min_terms",
         "max_len","avoid_line","terms_note","shape_line","shape_example",
         "low_t_example",
         "char_budget","n_terms","n_show",
         "used_line"}
_found = set(re.findall(r"(?<!\{)\{([a-z_]+)\}(?!\})", GEN_PROMPT))
if _found != _KEYS:
    raise SystemExit(f"GEN_PROMPT placeholders do not match: "
                     f"unfilled {_found - _KEYS}, unused {_KEYS - _found}")
GEN_HASH = register("GEN_PROMPT", GEN_PROMPT)
log(f"generation prompt {GEN_HASH}, {len(GEN_PROMPT)} chars")
log(f"  output budget {gen_max_out_for(0)} at t=0, "
    f"{gen_max_out_for(max(t for _, t in CELLS))} at the deepest cell")

TERM_BANNED = set(',"\\')

def clean_terms(seq):
    """Keep the longest leading run of plausible terms. The note names the term
    it stopped on, so a rejected first term and an empty reply differ."""
    out = []
    for i, t in enumerate(seq or []):
        s = re.sub(r"\s+", " ", str(t)).strip().lower().strip('"\'')
        if not s: return out, f"term {i+1} is empty"
        if len(s) > MAX_TERM_LEN:
            return out, f"term {i+1} is {len(s)} chars: {s[:40]!r}"
        bad = sorted(TERM_BANNED & set(s))
        if bad:
            return out, (f"term {i+1} contains {bad}, which a term may not: "
                         f"a comma makes it unreadable as one term, and a "
                         f"quote or a backslash is an escape. {s[:40]!r}")
        out.append(s)
    return out, ""

TABLE_STOP = {"the", "and", "of", "in", "to", "a", "an", "for", "by", "with",
              "order", "standard", "list", "sequence", "names", "name", "table",
              "each", "its", "their", "from", "into", "per", "all", "one",
              "two", "three", "first", "next", "letter", "letters", "word",
              "words", "number", "numbers", "count", "counts", "value",
              "values", "rank", "ranks", "size", "sizes", "grade", "grades",
              "type", "types", "class", "classes", "code", "codes"}

def worn_tables_line(top=4, floor=3):
    """Tables the battery keeps reaching for, read off the items.

    Same idea as worn_out_line and for the same reason, one level down. On one
    run the NATO alphabet appeared in 12 of 62 seeds, usually welded onto a
    theme it had nothing to do with, because a cell asking for two tables needs
    a second one and NATO is the shortest to name, so it wins on the character
    budget every time. Nothing told the composer it had already used it.

    Counted on DISTINCTIVE WORDS rather than whole spans, because the span is
    phrased afresh each item: "nato phonetic word", "nato alphabet" and "nato
    words" are the same table and match on nothing. The stoplist removes the
    words that describe any table at all, which is most of what a span says.
    """
    if not ITEMS:
        return ""
    counts, seen = collections.Counter(), collections.defaultdict(set)
    for b in ITEMS:
        words = set()
        for c in (b.get("components") or []):
            if isinstance(c, dict) and c.get("type") == "knowledge":
                for w in re.findall(r"[a-z]{3,}", str(c.get("text", "")).lower()):
                    if w not in TABLE_STOP: words.add(w)
        for w in words:
            counts[w] += 1; seen[w].add(b["id"])
    hot = [(k, v) for k, v in counts.most_common() if v >= floor][:top]
    if not hot:
        return ""
    return ("\nTABLES THIS BATTERY ALREADY LEANS ON, with how many items use "
            "each:\n"
            + "\n".join(f"  - {k} ({v})" for k, v in hot)
            + "\nIf this cell needs a table, prefer one not on that list. A "
              "second table bolted on because it is short to name, and having "
              "nothing to do with the theme, makes the item a worse one and "
              "the battery a narrower one.\n")

def avoid_line(cell=None):
    """Rules already held, so the composer is pushed off their shapes.

    Two changes from v2.2.1. It shows the RULE, not the `mechanism`: v5 measured
    mechanism overlap at 0.50 to 1.00 across pairs that had to stay separate,
    because a generic phrase like "ordered set lookup" describes half the
    battery, so showing it teaches the composer nothing about what to avoid.
    And it prefers rules from the SAME cell, since that is where a duplicate is
    both most likely and hardest to tell apart."""
    # Every item, including excluded ones. An item marked stale or flawed is
    # still a perfectly good example of a shape not to repeat, and filtering
    # them out silenced this exactly when a battery was being restarted, which
    # is when the pressure to differ matters most.
    pool = [b for b in ITEMS if b.get("seed")]
    if not pool: return ""
    same = [b for b in pool if cell is not None and cell_of(b) == cell]
    other = [b for b in pool if b not in same]
    random.shuffle(same); random.shuffle(other)
    picked = (same[:4] + other[:4])[:6]
    if not picked: return ""
    lines = []
    for b in picked:
        one = " ".join(str(b["seed"]).split())[:110]
        here = " (same dials)" if cell is not None and cell_of(b) == cell else ""
        lines.append(f"  - {one}{here}")
    worn = worn_out_line() + worn_tables_line()
    return (worn + "\nRules already in the battery. Invent something structurally "
            "DIFFERENT, not the same shape with different names or different "
            "constants:\n" + "\n".join(lines) + "\n")

WORN_STOP = set(
    "the each with from into then that this and for its one two based order "
    "sequence rule new set list value values using their step steps next "
    "number numbers item items where while over per term terms".split())

def worn_out_line(top=WORN_TOP, floor=WORN_FLOOR):
    """Mechanism phrases that keep recurring, read off the battery.

    Adaptive on purpose. A hard-coded list is wrong twice: it bans good
    mechanisms forever, and it does not reproduce from an empty battery, where
    nothing has recurred yet and the list should be silent.

    The `mechanism` field is the right input. v5 measured it as a poor signal
    for DUPLICATES because it is generic, scoring 0.50 to 1.00 on pairs that had
    to stay separate. That genericity is what makes it a good FAMILY label:
    a phrase describing several items is exactly what has been overused.

    Two-word phrases, not single words: alone, the words come out as `table`,
    `state`, `combined`, which say nothing. On a battery of 52 the phrases were
    `cyclic table`, `state machine`, `cellular automaton`.
    """
    if not ITEMS:
        return ""
    counts, who = collections.Counter(), collections.defaultdict(set)
    for b in ITEMS:
        # excluded items count here too: a mechanism that recurred is worn out
        # whether or not those particular items are still in use
        t = [w for w in re.findall(r"[a-z]{3,}", str(b.get("mechanism", "")).lower())
             if w not in WORN_STOP]
        for a, c in zip(t, t[1:]):
            counts[f"{a} {c}"] += 1
            who[f"{a} {c}"].add(b["id"])
    hot = [(k, v) for k, v in counts.most_common() if v >= floor][:top]
    if not hot:
        return ""
    return ("\nALREADY WORN OUT in this battery, with how many items use each:\n"
            + "\n".join(f"  - {k} ({v})" for k, v in hot)
            + "\nA theme does not make a mechanism new: it has to work "
              "differently, not be about something different. Prefer something "
              "outside this list, unless you can make it behave in a way none "
              "of those items does.\n")

def terms_note():
    """What the generation call is asked for, which depends on TERMS_FROM."""
    if TERMS_FROM != "produce":
        return ""
    return (
      "\n\nYOU DO NOT WRITE THE TERMS. Your rule will be handed, on its own, to "
      "several readers who have never seen this conversation, and they will "
      "produce the sequence from it. If they disagree, the item is thrown away.\n"
      "So the rule must fix the sequence COMPLETELY, including what a term looks "
      "like. If a term reads `100 shu` and your rule only says to add the running "
      "mean, one reader writes `100`, another `100 shu`, and the item dies. Say "
      "what a term is: `each term is the total followed by ' shu'`.\n"
      "Give a handful of terms below only to show you have checked this. They are "
      "not the sequence and are not kept.")

def shape_line(t):
    """How the answer should be shaped, which depends on depth.

    "Give the starting state once and the update step once" is an instruction
    for a MACHINE, and it was given at every cell. At t = 0 the composer read it
    as saying that an answer HAS a state and a step, and built a queue
    simulation: every r0t0 seed in one run was a state machine, and the gate
    then correctly rejected it for doing three or four transformations where
    none were asked. Nothing here tells it not to build a machine; the
    instruction to build one simply stops at t = 2."""
    if t >= 2:
        return "Give the starting state once and the update step once.\n"
    return ("Say what the nth term IS. At this depth there is nothing to carry "
            "from one term to the next, so a starting state and an update step "
            "are more machinery than the item needs.\n")

def shape_example(t):
    """The worked English-against-code pair. At t <= 1 the usual one is a queue
    machine, and shape gets copied even when the surrounding text is about
    something else, so a non-process pair is shown instead."""
    if t >= 2:
        return """
  GOOD  a queue holding 1. Take the front number m and emit it; if m is odd,
        put 2m and 2m+1 at the front of the queue
  BAD   line = [1]
        while True:
            m = line.pop(0)
            yield str(m)
            if m % 2 == 1: line = [2*m, 2*m+1] + line
"""
    return """
  GOOD  the alphabet, each letter twice over
  BAD   s = ''; i = 0
        while True: s = chr(97+i); yield s; yield s; i += 1
"""

def low_t_example(r, t):
    """A worked pair for the shallow cells, chosen by RETRIEVAL depth.

    Gated on r as well as t, because a single example would teach the wrong
    lesson at the other end of the retrieval axis: showing "the alphabet, each
    letter twice" at r=2 would collapse two tables into none. Each GOOD below
    uses exactly the number of tables its cell asks for.

    The r=1 example is a real seed from a finished run, accepted and measured at
    exactly r1t0."""
    if t > 1:
        return ""
    good = {
        0: "the alphabet, each letter twice over",
        1: "the standard wool quality counts in ascending order, each "
           "followed by s",
        2: "for each letter of the alphabet in order, the chemical element "
           "whose atomic number is that letter's position",
    }.get(min(r, 2))
    bad = {
        0: "a queue holding one letter; take the front, emit it, and append "
           "the next two letters",
        1: "a queue of wool counts; take the front, emit it, then append the "
           "next count and twice it",
        2: "a queue of letters and a running total; take the front, add its "
           "position to the total, emit the element at the total",
    }.get(min(r, 2))
    return (f"\nAT THIS DEPTH the answer is a description, not a process:\n"
            f"  GOOD  {good}\n"
            f"  BAD   {bad}\n"
            f"Both are deterministic. The second builds a machine the cell did "
            f"not ask for, and the gate counts its transformations and throws "
            f"the item away.\n")

HIDDEN_FROM  = 3     # the branch is offered from this t up
HIDDEN_FULL  = 6     # and is the only branch offered from this t up
                     # in between, the chance rises linearly, so the middle of
                     # the grid keeps the diversity the free composer gives it

def hidden_share(t):
    """How often the deep branch is taken at this t, from 0 to 1."""
    if t < HIDDEN_FROM: return 0.0
    if t >= HIDDEN_FULL: return 1.0
    return (t - HIDDEN_FROM + 1) / (HIDDEN_FULL - HIDDEN_FROM + 1)

def hidden_line(t, n_show):
    """The deep branch of world_line: tables and transformations used, not shown."""
    return (
"THE TERM SHOWS ONLY WHAT COMES OUT OF THE LAST TRANSFORMATION.\n"
f"The {t} transformations are applied in turn, each to what the one before it "
"produced. The tables they consult and the results they pass along are used "
"and not printed. A reader given the rule can check any step, table by table, "
"even if it takes a while. A reader given only the terms cannot work back to "
"the tables, because the transformations do not undo.\n"
"\n"
"THE TERM IS SMALL AND THE WORLD IT SHOWS MUST REACH MANY STATES. One mark "
"on six cells reaches six, however deep the rule that moves it: the terms "
"repeat with period six and a reader sees it in two rows. Two marks on those "
"cells reach fifteen, six cells each holding one of three arrows reach seven "
"hundred, and counters that keep growing never repeat at all. Count the "
"states your world can be in before you choose it.\n"
"\n"
"Written as table, then transformations, then what is printed:\n"
"\n"
"  no table -> a number grows by the sum of its own digits; which cell it "
"picks turns that cell's arrow one quarter turn, and a gap moves with it -> "
"the four cells.  ^--<, ^-<-, ->-<, v>-<\n"
"\n"
"  no table -> five counters start at nine; each step your transformations "
"pick one and add the digit sum of the step number to it -> the counters.  "
"9-9-12-9-9, 9-9-12-11-9, 9-14-12-11-9\n"
"\n"
"  a list of common nouns -> walk it by a fixed stride; report how long the "
"word is and where it would fall in a grid -> a clue.  4 across: 7, 9 down: 5\n"
"\n"
"  a list of common nouns -> walk it by a stride your rule fixes; two of the "
"six cells are lit and each step moves one of them -> the ring.  oo*o*o, "
"o*oo*o, o*o*oo\n"
"\n"
"  months -> the month's position in the year decides whether a filled or an "
"empty block is added; trim the pile from the bottom once it passes five -> "
"the pile.  [#], [#.], [#.#], [.#.#]\n"
"\n"
"  two decks in an order your rule fixes -> each player draws by one rule and "
"discards by another -> the two cards on top.  7S KH, 2D KH, 2D 9C\n"
"\n"
"Marks, blocks, arrows, suits and boxes are all fine in a term, and a term "
"made of them is often better than one made of words: it is compact and there "
"is no second way to spell it. A comma, a double quote and a backslash are the "
"only characters a term may not contain.\n"
"\n"
"THE RULE MUST BE PRECISE ENOUGH THAT TWO READERS GET THE SAME TERMS. Each "
"transformation should say what it acts on. Write it however reads best, as "
"numbered steps or semicolons or a sentence, but a reader must never have to "
"decide what a step applies to or which order two of them come in.\n"
"\n"
"Clear, and the term is still a picture:\n"
"  kings of England in order; the position of the current king in the list "
"decides which of four cells to act on; that cell's arrow turns a quarter "
"turn clockwise; print the four cells\n"
"  ^--<, ^-<-, ->-<, v>-<\n"
"\n"
"Also clear, and no longer:\n"
"  d starts at 7; each term d becomes d plus the sum of the digits of d; add "
"d to whichever of five counters d itself picks out; print the counters\n"
"  9-16-9-9-9, 9-16-9-31-9, 9-16-9-31-40\n"
"\n"
"CLEAR AND STILL NOT WORTH BUILDING, for two reasons worth knowing:\n"
"  d starts at 7; each term d becomes d plus the sum of its own digits; the "
"lit cell is d mod 6; print a ring of six cells with that one lit\n"
"  oo*ooo, oooo*o, o*oooo\n"
"  Six cells and one mark is six terms whatever the rule does, and a "
"remainder on division by six repeats with period six. Depth that ends in a "
"small index is depth a reader never has to follow.\n"
"\n"
"  kings of England; count the letters of the current king's name; step an "
"arrow that many cells along a track\n"
"  A reader may recall how long a word is rather than working it out, so "
"this is the step that goes wrong first, and everything after it inherits "
"the error. Use the position of a thing in its list, not the length of its "
"name.\n"
"\n"
"Not clear:\n"
"  append n, replace vowels with *, interleave -, reverse, shift digits +3 "
"mod 10, double the first char\n"
"\n"
"The last one has the same words and is not a rule: it does not say what is "
"reversed, whether the dashes go in before or after, or which digits are "
"shifted. Two runs of it produced a term and its mirror image. Notice that "
"the two clear ones are no longer than it, and that both print something to "
"look at.\n"
"\n"
"WHAT HAS GONE WRONG BEFORE WITH ITEMS OF THIS KIND:\n"
"\n"
"  A transformation that throws its own work away. Keeping only the last few "
"characters of something, or reducing everything to a handful of values. What "
"the earlier steps worked out should still be there, in some form, for the "
"later ones to use.\n"
"\n"
f"  A table or a transformation that sits idle at the start. All {t} of them, "
"and every table, should be doing something from the first terms onward, not "
"waiting for a condition that first arises well into the sequence. Anything "
"that has not yet made a difference is not yet part of the item, and the early "
"terms are explained without it.\n"
"\n"
"  Printing the table entry, the counter or the position. Then the "
"transformations are on the page and there is nothing left to work out. Print "
"what came out of the last step, not what went into it.\n"
"\n"
"  A table that can be read two ways. Say which: regnal names as commonly "
"written, months in English, elements by symbol. The rule is run cold more "
"than once and has to give the same terms every time.\n"
"\n"
"  Too many things printed. A term with four fields, most of them counters, is "
"easy to carry on. One thing moving, with a couple of properties, is the "
f"target. Over the {n_show} terms you write, a reader should see every "
"transformation making a difference and still not be able to name one.\n")

def world_line(t, n_show=20):
    """The world suggestion, from WORLD_FROM up, at every r.

    Two words are kept apart here. A STEP is one tick of the world's clock, and
    the world takes one step per term. A TRANSFORMATION is what t counts, and a
    step may contain several. Earlier wording used "step" for both and the
    count became unreadable.

    The threshold is 1. A named automaton is ONE transformation, so it sits at
    t = 1; a band starting at 3 excluded it and kept only the compound worlds,
    which lost 22 of 24 on the last run.
    """
    # the deep branch, taken with a chance that rises with t. Above
    # HIDDEN_FULL it is the only branch: nobody can hold seven transformations
    # in their head and keep them all doing something.
    if random.random() < hidden_share(t):
        return hidden_line(t, n_show)
    if t < WORLD_FROM: return ""
    if t <= 2:
        line = ("A LITTLE WORLD suits this cell: a small state, and one tick "
                "of its clock per term.\n"
                "  the rule-30 cellular automaton on a ring of 15 cells, "
                "starting from one x in the middle and o elsewhere; each term "
                "is the row\n"
                "Naming the automaton is ONE transformation, however many "
                "ticks it runs for. So at t = 1 that rule is the whole item, "
                "and at t = 2 you may add one more transformation to it.\n"
                "USE RULE 30, AND NINE CELLS OR MORE. From one lit cell, rule "
                "30 comes back round only after hundreds of terms at every "
                "width from 9 up, while rule 90 returns after 8 terms at width "
                "9 and 16 at width 15, and rule 110 after 9 at width 9. A ring "
                "of 7 closes in 4 terms whichever rule you pick, and a "
                "repeated sequence is thrown away. Any two symbols will do; "
                "say which you are using.\n")
    else:
        line = ("At this depth the best items are LITTLE WORLDS: a small "
                "state, and one tick of its clock per term. For this one, "
                "build it on: " + random.choice(WORLDS) + ".\n"
                f"The {t} transformations have to be IN the world: the tick "
                "itself does that many distinct things, or a named automaton "
                f"does one of them and {t - 1} more are applied to what it "
                "emits. Both are fine and both count the same.\n"
                "Whatever you add has to be written as precisely as the named "
                "automaton already is. \"the rule-30 cellular automaton\" "
                "leaves nothing open. \"take the lit count and combine it "
                "with the index\" leaves the reader to guess, and three "
                "readers guess differently.\n"
                "Adding a transformation can also FIX a world that repeats. A "
                "finite state comes back round, but if each term also carries "
                "the term number, or a running total, no two terms are alike "
                "even when the state returns.\n")
    if WORLD_PUSH:
        line += (
            "\nFOR THIS RUN, PREFER A WORLD. Two earlier batteries lost almost "
            "every world they attempted, 22 of 24 on the last one, to a repeat "
            "or to a case the rule left open. The battery is short of them. "
            "Unless the theme cannot carry one, make this item a world rather "
            "than a chain of arithmetic on the index.\n"
            "Spend the characters on being exact rather than on being "
            "elaborate. A small world stated completely is worth more than a "
            "large one stated loosely.\n")
    return line


## 8 · Determinism

The composer is trusted to know its own rule and not to have applied it without
slipping. So the rule is executed again in a fresh call that sees only the rule,
and the terms are compared.

The passes run together.
That gives up an early exit worth at most one call and
saves about a minute per candidate.


In [ ]:
RUN_PROMPT = """Follow this rule and list the sequence it produces, {k} terms.

Rule: {rule}

Apply it exactly as written. Do not explain, do not comment, do not correct it.

Give the TERMS THE RULE EMITS, not the state it holds: what it says to write
down, once per step, each as a plain string. Never a list, a dict, an index
expression or a variable name. A term looks like "waxing crescent" or "13+5" or
"c c c d", never like "s[3]" or "[1, 2]" or "state = {{...}}".

If the rule does not say what a term should look like, that is the rule's
problem and not yours: produce what it does say, and no more.

Answer in strict JSON, nothing else: {{"sequence": ["term1", "term2", ...]}}"""
RUN_HASH = register("RUN_PROMPT", RUN_PROMPT)

# Every candidate runs under RUN_PROMPT and nothing else: the item's own rule
# when its terms are produced, each equivalent when it is checked, each rival
# when it is executed. Under different prompts a disagreement caused by the
# harness would be indistinguishable from a real one, and unquestionability
# compares exactly those disagreements.

def execute(rule, want):
    """Run one candidate under RUN_PROMPT, the same prompt the item's own terms
    were produced under. Returns (terms, note)."""
    txt, note = ask(RUN_PROMPT.format(rule=rule, k=want), temperature=0.0,
                    max_out=OTHER_MAX_OUT)
    if txt is None: return None, note
    d, head = parse_json(txt)
    if d is None: return None, head or note
    terms, tnote = clean_terms(d.get("sequence", []))
    return terms, ("unreadable output: " + tnote) if tnote else note

def agreement(produced, target, want):
    n = 0
    for a, b in zip(produced or [], target[:want]):
        if a != b: break
        n += 1
    return n

def produce_terms(rule, want, iid="", reps=None, wins=None, model=None,
                  think=None):
    """Produce the sequence from the rule ALONE, several times, cold.

    Nothing is shown but the rule. No sample terms, no hint of the format. If a
    rule does not say whether a term reads `100` or `100 shu`, independent runs
    disagree and the item dies, which is correct: an underspecified rule cannot
    be reproduced by anyone.

    EVERY produced term is compared, not a prefix of them. A term the runs
    never agreed on cannot be shown or asked for, so leaving it unchecked only
    invites a rewriting to be judged against a term nobody verified.

    Checking the whole thing also catches a rule that is merely AMBIGUOUS. The
    executor-depth test found failure to be bimodal: a rule either reproduces
    all 60 terms or falls apart inside three, and what kills it is being
    unclear rather than being long. Two items already in this battery, accepted
    on a 20-term check, are wrong from term 0 when asked for 60. A check at the
    full length would have rejected both, and it costs no extra calls.

    Returns (terms, votes, checked_to)."""
    reps = reps or EXTEND_REPS
    wins = wins or EXTEND_WINS
    t0 = time.time()
    _WHY = []

    def one(i):
        # Temperature 0 three times is one computation repeated, which proves
        # little. A spread makes agreement evidence rather than an echo.
        txt, note = ask(RUN_PROMPT.format(rule=rule, k=want),
                        model=model, think=think,
                        temperature=EXTEND_TEMPS[i % len(EXTEND_TEMPS)],
                        max_out=GEN_MAX_OUT)
        if txt is None:
            _WHY.append(str(note)[:140]); return None
        d, head = parse_json(txt)
        if d is None:
            _WHY.append(f"reply was not usable json: {str(txt)[:60]}"); return None
        got, _ = clean_terms(d.get("sequence", []))
        return got or None

    runs = [r for r in parallel(one, list(range(reps)),
                                workers=min(MAX_WORKERS, reps)) if r]
    if not runs:
        log(f"      {iid} produce: no run returned anything usable "
            f"({time.time()-t0:.0f}s)"
            + (f"\n        why: {_WHY[0]}" if _WHY else ""))
        return None, 0, 0

    check = min(len(s) for s in runs)
    best, votes = None, 0
    for cand in runs:
        v = sum(1 for other in runs if agreement(other, cand, check) >= check)
        if v > votes or (v == votes and best is not None and len(cand) > len(best)):
            best, votes = cand, v
    worst = min((agreement(s, best, check) for s in runs if s is not best),
                default=check)
    log(f"      {iid} produce: {len(runs)} of {reps} replied, {votes} agree on "
        f"the first {check}, longest {len(best)} ({time.time()-t0:.0f}s)")
    if votes < wins:
        log(f"      {iid} produce: only {votes} agreed, needed {wins}. They part "
            f"company after term {worst}, so the rule does not fix its own "
            f"sequence"
            + ("" if worst > 2 else
               ". Parting at once means a case was left open.")
            + ("" if worst <= 2 else
               f". They agreed on {worst} terms first, so the rule is clear "
               f"and too heavy to carry."))
        _lo = max(0, worst - 2)
        _hi = worst + 3
        if _lo:
            log(f"        (terms {_lo+1} to {_hi} of each run; they agree "
                f"before that)")
        for k, sq in enumerate(runs[:3]):
            _seg = sq[_lo:_hi]
            log(f"        run {k+1}: "
                + ("... " if _lo else "")
                + ", ".join(_seg)
                + (" ..." if len(sq) > _hi else ""))
        return None, votes, worst
    return best, votes, check

def deterministic(rule, terms, reps=None, wins=None, indent="      "):
    """Does the rule reproduce the sequence, reliably?
    Returns (ok, votes, best_agreement, why)."""
    reps, wins = reps or VERIFY_REPS, wins or VERIFY_WINS
    # Compare enough terms to cover the longest prefix that will ever be
    # administered. At 20 the tail of a 60-term sequence was never checked.
    want = len(terms)          # every term, see the note above
    t0 = time.time()
    def one(_):
        produced, note = execute(rule, want)
        return agreement(produced, terms, want), note
    res = [r for r in parallel(one, list(range(reps)),
                               workers=min(MAX_WORKERS, reps)) if r]
    votes = sum(1 for r in res if r[0] >= want)
    best = max([r[0] for r in res] or [0])
    notes = [r[1] for r in res if r[1]]
    unread = sum(1 for n in notes if n.startswith("unreadable output"))
    why = "unreadable output" if (votes == 0 and res and unread == len(res)) else ""
    line = (f"{indent}determinism {votes}/{reps}, best {best}/{want} terms "
            f"({time.time()-t0:.0f}s)")
    if why: line += "  [output was state notation, not terms]"
    elif notes: line += f"  [{notes[0][:60]}]"
    log(line)
    return votes >= wins, votes, best, why


### 9b · Stripping notation labels

Only prose is stripped: the prefix must contain
a space, carry no code punctuation, and use a word the prompt itself used.


In [ ]:
# Words a notation label is made of. The composer prefixes its rewritings with
# these because the prompt names the notations, and the prefix is then charged
# as part of the program. Stripping it is not cosmetic: the cheapest equivalent
# is what gets priced.
LABEL_WORDS = {
    "python", "expression", "notation", "phrase", "assignment", "update",
    "recurrence", "arrow", "generator", "relation", "terse", "step", "form",
    "version", "style", "way", "alternative", "rewriting", "variant",
    "oneliner", "one-liner", "lambda", "code", "comprehension", "functional",
    "imperative", "iterative", "recursive", "compact", "short", "verbose",
    "restated", "equivalent", "rule", "method", "approach", "formulation",
}

def strip_label(p):
    """Remove a leading notation label such as 'Python expression: '.
    Returns (stripped, label). A label must be prose: it has a space, carries no
    code punctuation, and uses a word from the vocabulary the prompt itself
    used. So 'pos: (pos+1)%9' and 'state={a:0}; step: toggle' are left alone."""
    m = re.match(r"^\s*([^:\n]{3,44}):\s+(\S.*)$", p, flags=re.S)
    if not m:
        return p.strip(), ""
    label, rest = m.group(1), m.group(2)
    if " " not in label.strip():
        return p.strip(), ""
    if re.search(r"[=({\[<>%*/+]", label):
        return p.strip(), ""
    if not (set(re.findall(r"[a-z\-]+", label.lower())) & LABEL_WORDS):
        return p.strip(), ""
    return rest.strip(), label.strip()

def _label_selftest():
    keep = ["pos: (pos + behind(pos)) % 9", "state={a:0,b:1}; step: toggle a",
            "count from one", "levers={S:0}; next: A", "x = (2x+3) mod 29"]
    drop = [("Python expression: lambda n: n*(n+1)//2", "lambda n: n*(n+1)//2"),
            ("Arrow notation: (q_n, c_n) -> (q_n+1, c_n+1)",
             "(q_n, c_n) -> (q_n+1, c_n+1)"),
            ("Assignment and update step: q=0; q=(q+3)%9", "q=0; q=(q+3)%9")]
    bad = 0
    for p in keep:
        if strip_label(p)[0] != p:
            log(f"  SELFTEST FAILED: stripped a real rule: {p!r}"); bad += 1
    for p, want in drop:
        if strip_label(p)[0] != want:
            log(f"  SELFTEST FAILED: did not strip {p!r}"); bad += 1
    log(f"  label stripper: {len(keep)+len(drop)-bad} of "
        f"{len(keep)+len(drop)} self-test cases correct")
    if bad: raise SystemExit("the label stripper is not behaving")
_label_selftest()


## 9b · Does the item retrieve anything?

One call per candidate, after determinism and before any of the expensive work.

The retrieval dial does not land on its own. The theme's
vocabulary survives, the retrieval does not.

There is no free test for this.
Requiring non-numeric terms fails, because `mm`
and `spi` are units rather than knowledge.
It has to be asked.

So `draws_on` puts the SEED to the composer and asks what it actually needs to
know.
Asking about the seed rather than the bare rule is sound and cheap: the
seed is what generated the sequence, and naming a table in an equivalent can
only make explicit what the seed already inlined.
It cannot invent retrieval
that was never there.

When `DOMAIN_GATE` is on and a cell asks for $r \ge 1$ but the seed draws on
nothing, the candidate is rejected before its programs are bought: one call
instead of thirty-eight.
The next candidate for that slot is tried, and with
`DOMAIN_RETRY` a slot that runs out gets one more generation call which is told
what went wrong, the same shape as the rival retry.

At $r = 0$ "none" is the correct answer and nothing is rejected.


In [ ]:
DOMAIN_PROMPT = """Here is a rule that generates a sequence.

{seed}

TWO questions about it.

ONE. LIST every memorised table, ordering or set of facts WITHOUT WHICH THIS
SEQUENCE COULD NOT BE CONTINUED.

The test is necessity, and nothing else. Take someone who can see the terms and
knows arithmetic. For each candidate table, ask: if they did not know that
table, could they still work out what comes next? If they could not, it counts.

Two things follow, and both matter.

It does NOT matter whether the rule spells the table out. This rule lists all
twelve animals: "cycle through the 12 Chinese zodiac animals in standard order
(rat, ox, tiger, rabbit, dragon, snake, horse, goat, monkey, rooster, dog,
pig)". Its terms are `rat 3, ox 5, tiger 10`. Nobody continues that without
knowing the zodiac, so the zodiac COUNTS. Whether the rule happens to include
the list is a question about how long the rule is, not about what solving the
item requires.

It does NOT matter whether the table shows in the terms. "The proton count of
each metal in alphabetical order" has bare numbers for terms and needs the
periodic table and alphabetical order, neither of them stated. Both COUNT.

What does not count:
  - units. `mm`, `spi`, `shu` are notation.
  - arithmetic, counting, divisibility, sorting, and the letters of the
    alphabet in their ordinary order. Everyone has those.
  - a table the rule INVENTS. Three vessels of capacity 3, 4 and 5 are the
    rule's own furniture; nobody had to learn them.
  - words that are only labels. "cow 1, cow 2, cow 3" needs nothing about
    cows: a solver who has never seen a cow continues it perfectly.
  - a mapping that is really arithmetic wearing a name. Awarding 0 to 3 stars
    by the number of prime factors needs no knowledge of restaurant ratings.

TWO. COUNT the distinct TRANSFORMATIONS the rule performs on its way to a term.
A transformation changes something: an arithmetic step, a lookup by a computed
index, a comparison that branches, a state that carries forward, a wrap or
modulo, a sort, a concatenation that depends on what came before.

What does NOT count as a transformation:
  - reading the next entry of a table and printing it. That is zero.
  - formatting: adding a unit, lowercasing, padding.
  - counting up by one, if nothing is done with the count.

Count carefully. "Print the next NATO word" is 0. "Print the letter n places
along, wrapping" is 1. "Take the running total, add the digit sum, look up the
result in a table" is 3.

Answer in strict JSON, nothing else. An empty list and a zero are both normal.
{{"knows": ["...", "..."],
  "transformations": <a whole number>,
  "why": "<a dozen words at most>"}}"""
DOMAIN_HASH = register("DOMAIN_PROMPT", DOMAIN_PROMPT)

def draws_on(seed, iid=""):
    """What the seed needs and how much it does. Returns (list, count, why).

    A LIST, not one subject, because the dial asks for a NUMBER of tables and
    the answer has to be comparable with it. An earlier version returned a
    single name, so an r=2 item needing one table passed the gate and the
    second table the cell asked for was silently missing."""
    txt, note = ask(DOMAIN_PROMPT.format(seed=seed), temperature=0.0,
                    max_out=OTHER_MAX_OUT)
    d, head = parse_json(txt)
    if d is None:
        log(f"      {iid} draws_on: no answer ({(note or head)[:60]})")
        return None, -1, "the question could not be answered"
    raw = d.get("knows", [])
    if isinstance(raw, str):
        raw = [] if raw.strip().lower() in ("", "none", "-", "n/a") else [raw]
    out, seen = [], set()
    for x in raw:
        t = str(x).strip().lower()[:40]
        if t and t not in ("none", "-", "n/a", "nothing") and t not in seen:
            seen.add(t); out.append(t)
    try:
        n_t = int(d.get("transformations", -1))
    except (TypeError, ValueError):
        n_t = -1
    return out, n_t, str(d.get("why", "")).strip()[:90]

GEN_RETRY_NOTE = """
A previous attempt at this item was thrown away. Its rule was:

{seed}

The trouble: {why} The cell asks for {r} memorised table(s) and that rule needed
{got}.

Two ways this usually goes wrong, and the fix is the same for both.

If the rule SPELLED THE TABLE OUT, a reader needs to know nothing, because it is
all there in the rule. Name it instead. Say "the NATO alphabet in standard
spellings" and let the reader supply the words. Naming it is also far shorter,
which is what the item is scored on.

If the rule used the theme only for WORDS, the theme was decoration. Three
counters called root, stem and ear are three counters. The table has to be
needed: a reader who does not know it should not be able to continue the
sequence.

The table does NOT have to appear in the terms. "The proton count of each metal
in alphabetical order" is a fine item with bare numbers for terms, because it
needs the periodic table and alphabetical order, and neither is stated.
"""

REPAIR_PROMPT = """Here is a rule you wrote for a sequence puzzle. It was thrown
away, and the fault is fixable. KEEP THE IDEA. Repair it.

The rule:
{seed}

{problem}

{history}
What is wanted is the SAME world with the fault removed, not a different
subject. If the idea cannot survive the repair, say so by writing a rule that
keeps as much of it as you can.

{world_txt}
{constraints}

Answer in strict JSON, nothing else:
{{"seed": "<the repaired rule>",
  "gloss": "<one English sentence saying what it does>",
  "mechanism": "<short generic phrase for how the sequence is built>",
  "operations": ["<each transformation, two or three words>", ...],
  "tables": ["<each memorised table or ordering the rule needs>", ...],
  "fixed": "<one sentence: what you changed and why it removes the fault>",
  "sequence": ["<term 1>", "<term 2>", ...]}}"""

# This call carries no memory of the one that wrote the rule, so anything the
# generation prompt said has to be said again. Without it a rule is asked to be
# exact with no example of what exact looks like, which is the one thing it
# most needs at that moment.
REPAIR_WORLD = """Two things about worlds, in case they help.

NAMING AN AUTOMATON IS CHEAP AND EXACT. "the rule-30 cellular automaton on a
ring of 15 cells, starting from one x in the middle and o elsewhere; each term
is the row" fits in a hundred characters and leaves nothing for a reader to
guess. Naming one counts as a single transformation, however many ticks it
runs for, so there is room to apply more transformations to what it emits.
Use rule 30 and nine cells or more: rule 90 comes back round after 8 terms at
width 9 and 16 at width 15, and rule 110 after 9 at width 9, so the wrong rule
is itself a repeat waiting to happen.

A TERM MAY CARRY MORE THAN THE STATE. If each term shows the term number, or a
running total, alongside the state, then no two terms are alike even when the
state itself comes back round. That is often the shortest way out of a repeat.
"""

PROBLEM_REPEAT = """THE FAULT: the sequence repeats itself. {detail}

A sequence that has closed its cycle carries no more information, so it is
thrown away however good the idea is. A finite state space ALWAYS comes back
round; the only questions are how soon and whether you noticed.

Three ways out, in the order worth trying:

  GROW THE STATE. Let the world get bigger as it runs, so it cannot return to
  where it was. One more cell each time something reaches the edge; a queue that
  gains an item; a row that lengthens. This is the reliable fix and it usually
  costs a handful of characters.

  WIDEN IT. A fifteen-cell automaton is usually safe over {n_terms} terms; ten
  is not, and a few tokens on a short track come back round in twenty. If the
  state is a ring, make the ring bigger.

  CARRY A COUNTER. If the world itself must stay finite, let each term also show
  something monotone, the step number or a running total, so no two terms are
  identical even when the world returns.

Do not simply relabel the terms. The screen compares terms, so a cycle dressed
in different words is still a cycle."""

PROBLEM_OPEN = """THE FAULT: three readers were given this rule, saw nothing
else, and produced different sequences. {detail}

They parted almost at once, so the rule leaves a case open and each reader
closed it their own way. The rule is not wrong; it is incomplete.

Where a world usually leaves a gap:

  ORDER. When several things could move, which moves first? "Tokens move" is
  not an answer; "tokens move in index order, lowest first" is.
  COLLISION. What happens when the target is already taken: stay, swap, push,
  or skip to the next free one?
  EDGE. What happens at the end of the row or at the wrap?
  SIMULTANEITY. Are the updates computed from the OLD state all at once, or
  applied one after another so the second sees the first?
  STANDSTILL. What happens when nothing can move?
  START. Is the initial state given exactly, every cell of it?

Answer any of those your rule does not already answer."""

PROBLEM_HEAVY = """THE FAULT: three readers were given this rule, saw nothing
else, and produced different sequences. {detail}

They agreed for a while and then drifted apart, so the rule is CLEAR. What
went wrong is that it is too much work to carry. A reader has no calculator
and no notes: they hold the state in their head and update it {n_terms} times.
One slip early on changes everything after it, and two readers slip in
different places.

So this is not repaired by saying more. Adding clauses makes it heavier and it
will fail again. MAKE IT LIGHTER:

  FEWER MOVING PARTS. Two tokens instead of four. One counter instead of three.
  A single rule applied to one thing.
  A SMALLER STATE. A ring of 9 cells rather than 20. A queue of 5 rather than
  12. The reader has to re-read the whole state every term, so its width is
  what costs.
  LESS PER TERM. If a term needs a lookup, then a sum, then a modulo, then a
  format, that is four chances to slip. Two is usually enough to be
  interesting.
  NO LONG ARITHMETIC. Adding letter counts is fine. Squaring, then summing
  digits, then taking a remainder is where readers diverge.

Keep the idea. Make it smaller. A simple world that survives is worth more
than an elaborate one that nobody can reproduce."""

REPAIR_HISTORY = """Earlier attempts at this same idea, and how each failed:
{lines}
Do not repeat a fix that has already been tried."""



## 9 · Equivalents and rivals

Both are proposed in one call each and checked by execution rather than trusted.

An **equivalent** must produce the same sequence. It is what a pricing pass
minimises over, so the cheapest becomes $h$.

A **rival** must reproduce $x_{1:L}$ and then differ.
Rivals are asked for at
every length in `RIVAL_PREFIXES` that the sequence can support, and stored under
the length they were asked at.
Each also records where execution says it
actually diverges, which is often not where it was asked to.

This is the fix for the gap in v1.x.
Rivals existed only at $L_0 = 5$, so an
item certifying at $L = 9$ had a certificate nothing had ever been tested
against.
`no_rival_at` records the lengths where the search came back empty, so
a pricing pass can tell a prefix that survived from one that was never tried.

The listing rival is not here.
Its price depends on the reference machine's own
next-token distribution, so it has to be built where the machine is.

The composer prefixes its rewritings with the
notation name from the prompt, giving `Python expression: lambda n: ...`, and
since the cheapest equivalent is what gets priced, that label is charged as
part of the program. It only strips prose, so `pos: (pos + behind(pos)) % 9`
survives.


In [ ]:
EQUIV_PROMPT = """Here is a rule and the sequence it generates.

Rule: {rule}
Sequence: {terms}

Write {k} DIFFERENT ways of stating the SAME rule, each producing exactly that
sequence. Vary the notation between them: a compact expression, an assignment
plus an update step, arrow notation, a terse phrase, a closed form if one
exists.

MAKE THEM SHORT. This is the point of the exercise, not a nicety. The item is
scored by the SHORTEST of these, so one genuinely terse rewriting is worth more
than nine careful ones. Look for:
  - a closed form for the nth term, which usually beats any loop
  - a standard name for what the rule does, if there is one
  - dropping the names: single letters for variables, no comments, no function
    wrapper
  - dropping anything the terms already show

  - AND ABOVE ALL: NAME ANY STANDARD TABLE, DO NOT SPELL IT OUT. If the rule
    above writes out the NATO alphabet, the chemical elements, the books of the
    Bible or the planets, replace that list with its name. The reader already
    knows it, so listing it costs a great deal and says nothing. This is usually
    the single largest saving available, and on the items built so far it is
    nearly half of every rule at the crystallised end.
      instead of  w = ['alpha','bravo','charlie', ... 23 more]
      write       w = the nato alphabet in order
    Keep a table spelled out only when nobody could be expected to know it: an
    arbitrary substitution, a made-up set of capacities.

    Two cautions, both learnt the hard way. SAY WHICH FORM: a name is only
    useful if it pins the exact strings, so write "the US presidents by SURNAME
    in term order" rather than "the US presidents". Rewritings have been thrown
    away for producing "john adams" where the item says "adams". And NAME DATA,
    NOT OPERATIONS: you may refer to a table anyone could look up, but not to a
    function you invented. `best_load(a, capacity=2)` is not shorter, it is
    incomplete.

  - PREFER PLAIN ENGLISH, and use notation only where it makes that part
    shorter AND more precise: a modular index, an arithmetic expression, a
    fraction. These are rewritings for a reader, not programs to be run, so no
    imports, no `def`, no `yield`, no `while True:`. Dropping the scaffolding an
    interpreter needs often makes the sentence shorter than the code:
        "count from 1; each term is n/64 reduced"
      beats
        "from fractions import Fraction; n=1; while True: yield str(Fraction(n,64)); n+=1"
    A mixture is normal and often best. At least two of your answers should be
    mostly words.
Aim for something a person could write on one short line. If two of your answers
are the same length, replace the longer one with something shorter.

GIVE THE RULE AND NOTHING ELSE. No label in front of it: not "Python
expression:", not "Arrow notation:". Each of these is measured by how many
characters it takes to state, so a label is charged as part of the rule.

When followed, each term produced must be a plain string of letters, digits,
spaces and . - + / ' & ( ). A rule emitting brackets or index notation states
the machine's state rather than the term and cannot be checked.

EACH ONE MUST STAND ALONE. It is handed, by itself, to a reader who has never
seen this conversation, the rule above or the terms. Everything needed to
reproduce the sequence exactly must be inside it: the constants, the starting
values, and the FORM a term takes. If a term reads `100 shu` and your rewriting
only says to add the running mean, that reader writes `100` and your answer is
thrown away.

Do not change what the sequence is.

Answer in strict JSON, nothing else: {{"rules": ["...", "..."]}}"""

RIVAL_PROMPT = """Here are the first {L} terms of a sequence:
{prefix}

The next term is actually {intended}.

Find {k} DIFFERENT rules that produce those {L} terms EXACTLY and then continue
DIFFERENTLY - anything but {intended}. You want genuine alternative readings: a
solver who saw only this prefix could reasonably arrive at one and give the
wrong continuation.

EACH ONE MUST STAND ALONE. It is handed, by itself, to a reader who has never
seen this conversation or these terms. Everything needed to reproduce the prefix
exactly must be inside it: the constants, the starting values, and the FORM a
term takes.

SAY WHAT A TERM LOOKS LIKE, INSIDE THE RULE. This is not advice. On the last run
79 rival candidates failed to match their prefix and 46 of them matched NOTHING
AT ALL, median agreement zero terms: not because the arithmetic was wrong but
because the reader could not tell whether to write `100`, `100 shu` or `one
hundred`. You are looking at the terms; that reader is not.

Every rival must produce `{first}` as its first term, character for character,
and the rule has to make that unambiguous by itself: the unit, the separator,
the capitalisation, whether numbers are digits or words, whether there is
padding. Write it into the rule, for instance "..., each written as a number
followed by a space and `shu`".

WHAT THIS SEQUENCE'S OWN RULE LOOKS LIKE WHEN RUN COLD:
{cold}
That is not a rival, it is the working rule for these very terms, shown so you
can see the level of detail a rule needs before a stranger reproduces it. Match
that level.

WRITE IT AS THE REWRITINGS ARE WRITTEN, since a rival only matters if it is
cheap:
  - plain English, with notation only where notation is shorter AND more
    precise: a modular index, an arithmetic expression, a fraction
  - not a program to be run, so no imports, no `def`, no `yield`, no
    `while True:`
  - name any standard table rather than spelling it out, and say which form:
    "the nato alphabet, standard spellings", not the 26 words
  - name DATA, never an operation you invented
  - no label in front of it, and as short as you can make it

If you cannot honestly find one, return an empty list rather than inventing
something that does not fit the prefix.

Answer in strict JSON, nothing else:
{{"rivals": [{{"rule": "...",
              "form": "<how a term is written, in a few words>",
              "first": "<the first term YOUR rule produces, exactly>",
              "next": "<the term IT gives instead>"}}, ...]}}

`first` must come out identical to `{first}`. If it does not, the rival is
wrong about the form and will be thrown away, so fix the rule before answering.
"""

EQUIV_HASH = register("EQUIV_PROMPT", EQUIV_PROMPT)
RIVAL_HASH = register("RIVAL_PROMPT", RIVAL_PROMPT)

EQUIV_RETRY_NOTE = """
An earlier attempt at this went badly. {n_ok} of {n_tried} rewritings could be
used. Here is what happened to the others when each was run on its own:

{feedback}

Most of those are not disagreements about the sequence. They are answers that
could not be read: a reply that was not valid JSON, or that described a STATE
rather than listing the terms the rule emits. Fix that first.

  - Answer with nothing but the JSON object. No prose before it, no fence.
  - A term is what gets written down once per step, as a plain string. Never a
    list, a dict, an index expression or a variable name.
  - If the sequence has a unit or a separator, every rewriting has to say so,
    because whoever runs it sees only your rewriting.
"""

def equivalents_for(rule, terms, iid="", gloss="", extra=""):
    """Rewritings that reproduce the sequence. Each is executed before it counts."""
    t0 = time.time()
    txt, note = ask(EQUIV_PROMPT.format(rule=rule, k=N_EQUIVALENTS,
                    terms=", ".join(terms[:20])) + extra, temperature=0.9,
                    max_out=PROG_MAX_OUT)
    d, head = parse_json(txt)
    if d is None:
        log(f"      {iid} equivalents: nothing usable. {note or head}")
        return []
    cands, seen, stripped = [], {rule}, 0
    # The gloss is a candidate like any other. It is one English sentence
    # describing the same rule, it costs nothing to include since it already
    # exists, and it is checked by execution exactly like the rest. If it wins,
    # that is a finding: words were cheaper than the code.
    if gloss and 3 <= len(gloss) <= 400 and gloss not in seen:
        seen.add(gloss); cands.append(gloss)
    for q in d.get("rules", []):
        q, label = strip_label(str(q))
        if label: stripped += 1
        if 3 <= len(q) <= 400 and q not in seen:
            seen.add(q); cands.append(q)
    log(f"      {iid} equivalents: {len(cands)} to check ({time.time()-t0:.0f}s)"
        + (", the gloss is one of them" if gloss and gloss in cands else "")
        + (f", {stripped} labels removed before pricing" if stripped else ""))
    if not cands: return []
    def one(q):
        ok, votes, agreed, why = deterministic(q, terms, reps=2, wins=1,
                                               indent=f"        {iid} [{q[:20]}..] ")
        return dict(p=q, verified=ok, votes=votes, agreed_terms=agreed, note=why)
    out = [r for r in parallel(one, cands, workers=max(1, MAX_WORKERS//2)) if r]
    nv = sum(1 for e in out if e["verified"])
    unread = sum(1 for e in out if e["note"] == "unreadable output")
    log(f"      {iid} equivalents: {nv} of {len(out)} reproduce the sequence"
        + (f"; {unread} failed on output format, not on the rule" if unread else ""))
    return out

RIVAL_FAILURES = collections.Counter()
EQUIV_RESCUED = [0]      # equivalents found only by the informed second round
RIVALS_SKIPPED = [0]     # items whose rivals were skipped for having no program
RIVALS_UNAFFORDABLE = [0]  # items whose shortest rewriting is too long to
                           # certify inside the terms they will ever be shown
RIVAL_RESCUED = [0]      # rivals found only by the informed second round
RIVAL_NEAR = []          # (item, L, agreed, of) for the near misses

RIVAL_RETRY_PROMPT = """Here are the first {L} terms of a sequence:
{prefix}

The next term is actually {intended}.

You were asked for rules that produce those {L} terms EXACTLY and then continue
differently. {n_tried} were tried and {n_ok} worked. Here is what happened to the
others, with what each actually produced when it was run on its own:

{feedback}

Try again, {k} more, fixing what went wrong. Note especially:
- A rule that produced NOTHING like the prefix has almost always lost the FORM
  of a term. You can see the terms; the reader who runs your rule cannot. If a
  term reads {example!r}, your rule has to say so.
- A rule that matched some terms and then broke was close. Look again at where
  it broke rather than starting over.
- A rule that gave {intended} is not a rival at all; it has to differ there.

Same requirements as before: each stands alone, plain English with notation only
where it is shorter, no imports or `def`, name standard tables and say which
form, no label, and as short as you can.

Your first term is `{first}`. Whatever else changes, every rival must produce
exactly that, character for character, and the rule must say how a term is
written: the unit, the separator, whether numbers are digits or words. Of the
candidates that missed the prefix last time, most matched NOTHING, which is the
form being lost rather than the arithmetic being wrong.

Answer in strict JSON, nothing else:
{{"rivals": [{{"rule": "...", "next": "<the term IT gives instead>"}}, ...]}}"""
RIVAL_RETRY_HASH = register("RIVAL_RETRY_PROMPT", RIVAL_RETRY_PROMPT)

def _run_rival_batch(cands, terms, L, iid, tag=""):
    """Execute each candidate and say why it failed. No judging, no pricing."""
    def one(q):
        produced, note = execute(q, min(len(terms), L + 4))
        if not produced:
            return dict(ok=False, why="produced nothing",
                        detail=(note or "")[:60], p=q, got=None)
        if len(produced) <= L:
            return dict(ok=False, why="too short",
                        detail=f"{len(produced)} terms, needs more than {L}",
                        p=q, got=produced)
        agree = agreement(produced, terms, len(terms))
        if agree < L:
            return dict(ok=False, why="agreed on fewer than L", agreed=agree,
                        detail=f"agreed on {agree} of {L}"
                               + (f", gave {produced[agree]!r} for "
                                  f"{terms[agree]!r}" if agree < len(produced) else ""),
                        p=q, got=produced)
        if produced[L] == terms[L]:
            return dict(ok=False, why="did not diverge",
                        detail=f"gave the intended {terms[L]!r}", p=q, got=produced)
        return dict(ok=True, p=q, asked_at=L, diverges_at=agree,
                    gives=produced[agree] if agree < len(produced) else None)

    res = [r for r in parallel(one, cands) if r]
    good = [r for r in res if r["ok"]]
    bad = [r for r in res if not r["ok"]]
    log(f"      {iid} rivals at L={L}{tag}: {len(good)} of {len(cands)} confirmed")
    if bad:
        by = collections.Counter(r["why"] for r in bad)
        RIVAL_FAILURES.update(by)
        for r in bad:
            if r["why"] == "agreed on fewer than L":
                RIVAL_NEAR.append((iid, L, r.get("agreed", 0)))
        log(f"        discarded: " + "; ".join(f"{v} {k}" for k, v in by.most_common()))
        for r in bad[:N_RIVALS]:
            log(f"        - {r['why']:22s} {r['detail'][:56]:58s} {r['p'][:34]}")
    return good, bad

def _ask_rivals(prompt, iid, L, tag=""):
    """One rival call. Returns the cleaned candidate list.

    v2.24.0 asked each candidate to report the first term its rule produces and
    dropped the ones that disagreed. It never fired once in 282 candidates, and
    46 of 79 lost rivals still matched nothing, so it is gone. The composer
    cannot predict what a fresh reader will produce: it predicts what IT would
    produce. Only running the thing tells you, which is why the informed second
    round works and a self-report does not."""
    t0 = time.time()
    txt, note = ask(prompt, temperature=1.1, max_out=RIVAL_MAX_OUT,
                    timeout=API_TIMEOUT_RIVALS)
    d, head = parse_json(txt)
    if d is None:
        why = note or head
        if "MAX_TOKENS" in why:
            why = (f"reply truncated at {RIVAL_MAX_OUT} output tokens. The search "
                   f"runs inside the reply, so raise RIVAL_MAX_OUT or lower "
                   f"N_RIVALS")
        RIVAL_FAILURES["call gave nothing usable"] += N_RIVALS
        log(f"      {iid} rivals at L={L}{tag}: nothing usable. {why}")
        return []
    cands, seen = [], set()
    for x in d.get("rivals", []):
        q = str(x.get("rule", "")) if isinstance(x, dict) else str(x)
        q, _ = strip_label(q)
        if 8 <= len(q) <= 400 and q not in seen:
            seen.add(q); cands.append(q)
    log(f"      {iid} rivals at L={L}{tag}: {len(cands)} to check "
        f"({time.time()-t0:.0f}s)")
    return cands

def _cold_example(rule):
    """The item's own rule, shown to the rival search as a worked example.

    Round two beats round one because it is TOLD what happened, and round one
    cannot be. This is the nearest thing round one can have for free: the rule
    that actually produced these terms was executed at generation time and
    agreed with itself across three runs, so it is a rule known to survive a
    cold reading. Showing it says what that costs in detail, without spending a
    call and without proposing anything."""
    if not rule:
        return "  (not available for this item)"
    r = " ".join(str(rule).split())
    return "  " + (r[:400] + (" ..." if len(r) > 400 else ""))

def rivals_at(terms, L, iid="", cold_rule=None):
    """Rules matching terms[:L] that then differ, verified by execution.

    Two rounds at most. The second only happens when the first found almost
    nothing, and it is TOLD what went wrong: which candidates produced nothing
    like the prefix, which were near misses and where they broke. That is fair
    to give the composer, because the composer writes the rival with the terms
    in front of it while the reader who runs it sees only the rule; the format
    ambiguity is real and it accounted for 15 of 33 discards.

    Only the COMPOSER prompt carries this. RUN_PROMPT never changes, so every
    candidate is still executed under identical conditions."""
    cands = _ask_rivals(RIVAL_PROMPT.format(L=L, prefix=", ".join(terms[:L]),
                        intended=terms[L], k=N_RIVALS,
                        first=terms[0], cold=_cold_example(cold_rule)),
                        iid, L)
    if not cands:
        RIVAL_FAILURES["none proposed"] += N_RIVALS
        return []
    good, bad = _run_rival_batch(cands, terms, L, iid)

    if len(good) <= RIVAL_RETRY_IF and bad:
        fb = "\n".join(
            f"  - {r['p'][:70]}\n      {r['detail'][:90]}" for r in bad[:N_RIVALS])
        cands2 = _ask_rivals(RIVAL_RETRY_PROMPT.format(
            L=L, prefix=", ".join(terms[:L]), intended=terms[L], k=N_RIVALS,
            n_tried=len(cands), n_ok=len(good), feedback=fb,
            first=terms[0], example=terms[0]), iid, L,
            tag=" (second round)")
        if cands2:
            seen = {r["p"] for r in good} | {r["p"] for r in bad}
            cands2 = [q for q in cands2 if q not in seen]
            g2, _ = _run_rival_batch(cands2, terms, L, iid, tag=" (second round)")
            good += g2
            if g2:
                RIVAL_RESCUED[0] += len(g2)

    if good:
        best = min(good, key=lambda r: r["diverges_at"])
        log(f"      {iid} rivals at L={L}: earliest divergence at term "
            f"{best['diverges_at']+1}, giving {best['gives']!r}")
    return [dict(p=r["p"], asked_at=r["asked_at"], diverges_at=r["diverges_at"],
                 gives=r["gives"]) for r in good]

def programs_for(item, old=None):
    """Everything an item needs before a pricing pass can look at it.

    `old` is any record already held. It is MERGED, not overwritten: a program
    that verified once does not stop being valid, and a rival confirmed by
    execution is a fact about the sequence. Overwriting meant a repair pass paid
    to re-derive equivalents that had already passed, and could lose a rival
    when this run's call truncated where the earlier one had not."""
    iid, terms, rule = item["id"], item["terms"], item["seed"]
    old = old or {}
    old_eqs = list(old.get("equivalents", []))
    old_rivals = list(old.get("rivals", []))
    kept_eqs = sum(1 for e in old_eqs if e.get("verified"))

    # Only ask for equivalents when the record is short of them. Six that
    # verified is enough to minimise over.
    if kept_eqs >= N_EQUIVALENTS:
        log(f"      {iid} equivalents: {kept_eqs} already verified, not asking again")
        eqs = old_eqs
    else:
        fresh = equivalents_for(rule, terms, iid=iid,
                                gloss=item.get("gloss", ""))
        seen = {e["p"] for e in old_eqs}
        eqs = old_eqs + [e for e in fresh if e["p"] not in seen]
        if old_eqs:
            log(f"      {iid} equivalents: kept {kept_eqs} verified from before, "
                f"added {len(eqs) - len(old_eqs)} new")

        n_ok = sum(1 for e in eqs if e.get("verified"))
        bad = [e for e in eqs if not e.get("verified")]
        if EQUIV_RETRY and n_ok < EQUIV_RETRY_IF and bad:
            fb = "\n".join(
                f"  - {e['p'][:60]}\n      "
                + (e.get("note") or f"reproduced {e.get('agreed_terms', 0)} terms "
                   f"and then diverged")[:80]
                for e in bad[:N_EQUIVALENTS])
            log(f"      {iid} equivalents: only {n_ok} usable, asking again with "
                f"the failures fed back")
            more = equivalents_for(rule, terms, iid=iid,
                                   gloss=item.get("gloss", ""),
                                   extra=EQUIV_RETRY_NOTE.format(
                                       n_ok=n_ok, n_tried=len(eqs), feedback=fb))
            seen = {e["p"] for e in eqs}
            gained = [e for e in more if e["p"] not in seen]
            eqs = eqs + gained
            n2 = sum(1 for e in gained if e.get("verified"))
            if n2:
                EQUIV_RESCUED[0] += n2
                log(f"      {iid} equivalents: the second round added {n2} that "
                    f"reproduce the sequence")

    # Rivals: only the prefixes that have none yet, so a retry after a truncated
    # call costs one call rather than all of them.
    # An item with no verified equivalent cannot be priced at all, so its
    # rivals would be compared against nothing. On one run four such items
    # bought 40 rivals between them, about twenty minutes of entirely wasted
    # work. A later repair pass buys rivals if an equivalent ever verifies.
    _nv = sum(1 for e in eqs if e.get("verified"))
    _skip = SKIP_RIVALS_IF_NO_EQUIV and _nv == 0
    if _skip:
        log(f"      {iid}: no equivalent reproduces the sequence, so rivals are "
            f"skipped. Nothing could be priced against them.")
        RIVALS_SKIPPED[0] += 1

    # ---- the affordability gate, before the expensive half -----------------
    # Rivals are about 70% of what an item costs, and they are bought AFTER the
    # equivalents, which is exactly when the shortest one is known. So this is
    # the first moment the certification arithmetic can be run, and the last
    # moment before the money is spent.
    #
    # Prop 4.1: an item certifies at L only if K + delta < beta*L + c0. With L
    # capped at this item's own length less K_ADMIN, a rule too long to beat
    # that prefix cannot certify at any prefix it will ever be shown at. Buying
    # rivals for it is buying evidence against a verdict that cannot be reached.
    #
    # This does not delete or mark anything. The item is kept with its estimate
    # recorded, so a later run with a shorter rewriting, or a machine with a
    # larger beta, can pick it up. It only declines to spend now.
    _short = min((len(e["p"]) for e in eqs if e.get("verified")), default=None)
    # the TEXT of the shortest verified rewriting, for the rival search to see
    # as a worked example. Falls back to the authored rule, which also passed
    # determinism, when nothing verified.
    _short_p = min((e["p"] for e in eqs if e.get("verified")), key=len,
                   default=None) or rule
    _Lmax = min(RIVAL_PREFIX_MAX, len(terms) - 2, len(terms) - K_ADMIN)
    _unaffordable = False
    if not _skip and _short:
        _need = prefix_needed(_short, beta=BETA_GATE)   # optimistic on purpose
        if _need > _Lmax:
            _unaffordable = True
            log(f"      {iid}: the shortest rewriting is {_short} chars, about "
                f"{bits_of(_short):.0f} bits, which needs a prefix of {_need} "
                f"terms to beat the listing.")
            log(f"      {iid}: the most that can be shown is {_Lmax}, so no "
                f"prefix certifies and rivals are not bought, even taking "
                f"beta at the top of its measured range. A rewriting under "
                f"{chars_affordable(len(terms), beta=BETA_GATE)} chars would "
                f"change that.")
            RIVALS_UNAFFORDABLE[0] += 1

    Ls = ([] if (_skip or _unaffordable)
          else rival_prefixes_for(_short, len(terms)))
    if Ls and _short:
        log(f"      {iid}: asking for rivals at {Ls}"
            + (f" (expected certification near L={prefix_needed(_short)})"
               if RIVAL_ADAPTIVE else ""))
    have_at = {r.get("asked_at") for r in old_rivals}
    rivals = list(old_rivals)
    empty, skipped = [], []
    todo = [L for L in Ls if L not in have_at]
    skipped = [L for L in Ls if L in have_at]
    # The prefixes are independent: a rival at L=5 has no bearing on one at
    # L=38. Asking them in sequence made the item cost the SUM of the asks, and
    # an ask grows with the prefix, roughly 90s at L=5 and ten minutes at L=38.
    # Run together it costs the longest instead of the total, which is most of
    # the difference between a four hour run and a fourteen hour one.
    if todo:
        got_all = parallel(lambda L: rivals_at(terms, L, iid=iid, cold_rule=_short_p), todo,
                           workers=min(len(todo), MAX_WORKERS))
        for L, got in zip(todo, got_all):
            got = got or []
            seen = {r["p"] for r in rivals}
            rivals += [r for r in got if r["p"] not in seen]
            if not got: empty.append(L)
    if skipped:
        log(f"      {iid} rivals: already had some at L={skipped}, not asked again")
    if rivals:
        best = min(rivals, key=lambda r: r["diverges_at"])
        log(f"      {iid}: {len(rivals)} rivals in total, earliest divergence at "
            f"term {best['diverges_at']+1} (gives {best['gives']!r} not "
            f"{terms[best['diverges_at']]!r})")
    else:
        log(f"      {iid}: still no rival at any of {Ls}")
    verified = [e for e in eqs if e.get("verified")]
    bare = min((e["p"] for e in verified), key=len, default=None)
    shortest = len(bare) if bare else None
    gl = item.get("gloss", "")
    gloss_ok = any(e["p"] == gl and e.get("verified") for e in eqs)
    if gl:
        log(f"      {iid}: the gloss "
            + ("reproduces the sequence" if gloss_ok else "does not reproduce it")
            + (f" and is the shortest verified program"
               if gloss_ok and len(gl) == shortest else ""))
    need = terms_needed(shortest) if shortest else None
    L_need = prefix_needed(shortest) if shortest else None
    if shortest:
        _afford = chars_affordable(len(terms))
        log(f"      {iid}: shortest verified equivalent is {shortest} chars "
            f"(about {bits_of(shortest):.0f} bits), so it needs a prefix of "
            f"{L_need} and {need} terms in all; the item has {len(terms)}"
            + ("" if need <= len(terms) else "  <- SHORT"))
        log(f"      {iid}: the budget at {len(terms)} terms is a rule of "
            f"{_afford} chars"
            + ("" if shortest <= _afford else
               f", so this one is {shortest - _afford} chars over"))
    return dict(
        prefix_needed_estimate=L_need,
        chars_affordable=chars_affordable(len(terms)),
        calibration=CALIB.get("source"),
        equivalents=eqs, rivals=rivals,
        asked_at=sorted(set(Ls) | have_at - {None}),
        no_rival_at=sorted(set(empty) |
                           (set(old.get("no_rival_at", [])) - {r.get("asked_at")
                            for r in rivals})),
        # what the pricing pass needs to know about the shape of this record.
        # Changing either count invalidates a verdict: more equivalents lower K
        # and can rescue a questionable item, more rivals can break a certified
        # one. So a pricing pass can compare and reprice only what moved.
        bare=bare,
        gloss_verified=gloss_ok,
        gloss_chars=len(gl) if gl else None,
        n_verified_equivalents=len(verified),
        n_rivals_total=len(rivals),
        shortest_equivalent_chars=shortest,
        terms_needed_estimate=need,
        terms_available=len(terms))


## 10 · The gauge on the dials

One call per item.
It splits the gloss into spans typed `knowledge`,
`operation` or `control` and counts them.

The dials say what was asked for.
This says what was made.
Asking for $r{=}0$
and getting three knowledge spans means the cell did not land, and without this
there is no way to know.

`gc_count` is $n_\text{knowledge} / (n_\text{knowledge} + n_\text{operation})$.
It is unweighted, so it is not the `gc_share` of the old monolith, which
weighted each span by its cost in context under a reference machine.
That
weighting is what separates `for each`, which is nearly free in context, from
`the western zodiac`, which stays dear because nothing else says which table is
meant.
The bits version needs a machine and can be added later for whatever you
price.

What the count version has instead is machine independence.
It does not move
when $\mathcal{M}$ changes, which for a manipulation check on the dials is the
property you want.

The spans are stored, so the same items can be repriced into a bits version
later without asking the composer again.


In [ ]:
# FROZEN. This prompt is the ADMISSION TEST, not the measurement.
#
# It decides whether a candidate lands close enough to its cell to be kept, so
# every item in the battery should have been admitted under the same rule. If
# it changes, the composition of the battery changes and no amount of
# re-measuring afterwards puts that right. Improvements to how r and t are
# COUNTED belong in Ctest-Measure, which runs after pricing and can be improved
# as often as it needs to be.
#
# Change it only when starting a new battery prefix, and record which version
# admitted which items.
DECOMPOSE_VERSION = "filter-v1"

DECOMPOSE = """Parse this instruction into components. Use exactly these types:

- "knowledge": anything that must be REMEMBERED rather than worked out. Both
  naming a memorised set or ordering ("the western zodiac", "US states by
  admission") AND reading a stored attribute off its members ("release year of",
  "atomic number of"). If answering needs a learnt table, it is knowledge, even
  when phrased as an action.
  EXCEPT the free orderings, which are NOT knowledge and are "control":
  the counting numbers, the letters of the alphabet in order, and the digits.
  Everyone has these, so naming one retrieves nothing. "count from 1", "the
  alphabet", "a, b, c ..." and "0 to 9" are control, not knowledge. Anything
  else nameable still is knowledge.
- "operation": anything DONE to the sequence or its terms that needs no
  memorised table. Functions of a term ("square", "count the letters of", "first
  letter of") and rearrangements ("sorted alphabetically", "reversed", "every
  second one", "keeping only those that"). If it changes which terms appear, in
  what order, or what each becomes, it is an operation.
  EXCEPT a plain JOIN between the sources the rule already draws on, which is
  "control": running one after another, or alternating between them in their
  NATURAL order. That is the least that having two sources requires, so it is
  not something extra done to them. But choosing WHICH source by the index, by
  a value, or by any condition IS an operation, and so is interleaving under a
  rule rather than by simple alternation.
- "control": the scaffolding that introduces or joins the output: "list", "for
  each", "then", "in order" when that means nothing more than the natural
  order. Also the two exemptions above: a free ordering, and a plain
  alternation or concatenation of the sources. Sorting, filtering and reversing
  are NOT control.

Quote each component as a span of the instruction. Do not invent components that
are not there.

Instruction: {p}

Then, separately: what human subject does this rule ACTUALLY draw on? Not what
it is dressed as. A rule that names three counters `root`, `stem` and `ear` and
adds them up draws on nothing: the words are decoration and the answer is
"none". A rule that indexes the NATO alphabet draws on that alphabet. Answer in
one or two words, or "none".

Answer in strict JSON, nothing else:
{{"components": [{{"text": "<span>", "type": "knowledge|operation|control"}}],
  "domain": "<one or two words, or none>"}}"""
DECOMPOSE_HASH = register("DECOMPOSE", DECOMPOSE)

def components_for(item):
    """Type the spans, and name the domain the rule really draws on.

    Read from the BARE rule when there is one, since that is the description
    with the decoration stripped out. An item whose theme was `cereal grain
    growth stages` and whose rule adds three counters called root, stem and ear
    should come back "none", and its theme domain of `agriculture` is then known
    to be a label rather than a fact."""
    text = (item.get("bare") or item.get("gloss") or item.get("mechanism")
            or item.get("seed"))
    txt, note = ask(DECOMPOSE.format(p=text), temperature=0.0, max_out=OTHER_MAX_OUT)
    d, head = parse_json(txt)
    if d is None:
        return dict(components=[], components_source="failed",
                    components_note=(note or head)[:120])
    comps = []
    for c in d.get("components", []):
        if not isinstance(c, dict): continue
        ty = str(c.get("type", "")).strip().lower()
        sp = str(c.get("text", "")).strip()
        if ty in ("knowledge", "operation", "control") and 0 < len(sp) <= 120:
            comps.append(dict(text=sp, type=ty))
    n = collections.Counter(c["type"] for c in comps)
    k, o = n["knowledge"], n["operation"]
    dom = str(d.get("domain", "")).strip().lower()[:24] or "none"
    if dom in ("n/a", "-", "", "nothing"): dom = "none"
    return dict(domain_real=dom, components=comps,
                decompose_version=DECOMPOSE_VERSION,
                components_source=("gloss" if item.get("gloss") else "mechanism"),
                n_knowledge=k, n_operation=o, n_control=n["control"],
                gc_count=(round(k / (k + o), 3) if (k + o) else None))


## 11 · Run

Five phases, ordered so nothing expensive happens before the free checks.

1. **Plan.** A theme per candidate, skipping ones the cell cannot carry.
2. **Generate.** One call per candidate, all in flight.
3. **Screen.** Term count, repeat test, collapse screen, and a clash check
   against items already held.
No calls.
4. **Verify.** Determinism on the best survivor per slot.
5. **Programs and components.** Equivalents, rivals at each prefix, and the
   component pass.

Saved after every accepted item, so an interrupted run keeps what it had.


## 6b · What a theme's members look like

Written onto each theme as `terms_are`, and read by `theme_fits`.
Judged once
and cached, so this cell does nothing on later runs.

The theme is chosen before the prompt is built, so a theme whose members are
numbers gives an item made of numbers however the prompt is worded.
About half
the store is that kind.
That, and not the prompt, is why so much of the battery
counts.

The judge lists three members before answering, because the name misleads in
both directions: "chinese zodiac year cycle" reads numeric and gives rat, ox,
tiger, while "electrical wire gauge standards" reads harmless and gives 40, 36,
18.
And it strips units first, since `10 m, 20 m, 30 m` is a numeric sequence
wearing a unit.


In [ ]:
# ---------------------------------------------------------------------------
# WHAT DOES A THEME'S MEMBERS LOOK LIKE?
#
# Written onto each theme as `terms_are`, one of "numbers" or "words", and read
# by theme_fits. Judged once, cached, and skipped entirely when every theme
# already has a verdict.
#
# The name is not a good guide, in either direction. "chinese zodiac year cycle"
# reads numeric and gives rat, ox, tiger. "electrical wire gauge standards"
# reads harmless and gives 40, 36, 18. So the judge is asked to LIST three
# members first and answer about those.
#
# The trap is decoration. 10 m, 20 m, 30 m is a numeric sequence wearing a unit,
# and so are cone 06, size 12 and 0000 AWG. Strip the unit, the prefix and the
# suffix; if a number is left, it is a number.

THEME_JUDGE      = True    # off skips this cell entirely
THEME_JUDGE_MAX  = 0       # 0 judges every theme that has no verdict
THEME_JUDGE_TAG  = "terms_are"

JUDGE_PROMPT = """For each subject below, name three of its members, then say
whether those members are NUMBERS or WORDS.

Strip any unit, prefix or suffix first. If what is left is a number, the answer
is numbers.

  "10 m, 20 m, 30 m"        -> strip the unit -> 10, 20, 30 -> numbers
  "cone 022, cone 06"       -> strip the word -> 022, 06    -> numbers
  "0000 awg, 36 awg"        -> numbers
  "third, fourth, fifth"    -> written as words, but they are positions -> numbers
  "rat, ox, tiger"          -> words
  "do, re, mi"              -> words
  "alpha, bravo, charlie"   -> words
  "pale straw, gold, amber" -> words

Answer about the MEMBERS, not about the subject's name. A subject called
"... year cycle" whose members are animals is words.

Subjects:
{subjects}

Answer in strict JSON, nothing else:
{{"verdicts": [{{"subject": "<exactly as given>",
                "members": ["<three of them>"],
                "terms_are": "numbers" or "words"}}, ...]}}"""

def judge_themes(reason=""):
    """Give every unjudged theme a `terms_are` verdict.

    Called wherever themes appear, not at one fixed point in the
    notebook. On a cold start the file is empty and the themes are
    fetched inside PHASE 1, so a judging step that ran before it found
    nothing to do and the numeric filter was inert for the whole run:
    an unjudged theme is never skipped. That is the run the filter
    exists for.
    """
    if not THEME_JUDGE:
        return
    if reason:
        stage(f"JUDGING THE THEMES ({reason})")
    else:
        stage("JUDGING THE THEMES")
    _todo = [t for t in THM["themes"] if not t.get(THEME_JUDGE_TAG)]
    if THEME_JUDGE_MAX:
        _todo = _todo[:THEME_JUDGE_MAX]
    _have = len(THM["themes"]) - len([t for t in THM["themes"]
                                      if not t.get(THEME_JUDGE_TAG)])
    log(f"  {len(THM['themes'])} themes, {_have} already judged, "
        f"{len(_todo)} to do")

    if not _todo:
        log("  nothing to judge.")
    else:
        _BATCH = 25
        _chunks = [_todo[i:i+_BATCH] for i in range(0, len(_todo), _BATCH)]
        log(f"  {len(_chunks)} call(s) of up to {_BATCH} subjects each")

        def _judge(chunk):
            subj = "\n".join(f"- {t['theme']}" for t in chunk)
            txt, note = ask(JUDGE_PROMPT.format(subjects=subj),
                            temperature=0.0, max_out=OTHER_MAX_OUT)
            if txt is None:
                return {}
            d, _ = parse_json(txt)
            if not d:
                return {}
            out = {}
            for v in d.get("verdicts", []):
                if not isinstance(v, dict):
                    continue
                s = str(v.get("subject", "")).strip().lower()
                a = str(v.get("terms_are", "")).strip().lower()
                if s and a in ("numbers", "words"):
                    out[s] = (a, [str(x)[:24] for x in (v.get("members") or [])][:3])
            return out

        _res = parallel(_judge, _chunks, workers=min(len(_chunks), MAX_WORKERS),
                        label="judging")
        _all = {}
        for r in _res:
            if r: _all.update(r)

        _n = collections.Counter()
        for t in THM["themes"]:
            hit = _all.get(t["theme"].strip().lower())
            if not hit: continue
            t[THEME_JUDGE_TAG] = hit[0]
            t["members_seen"] = hit[1]
            _n[hit[0]] += 1
        save_all()
        log(f"  judged {sum(_n.values())} of {len(_todo)}: "
            f"{_n['numbers']} numbers, {_n['words']} words")
        _miss = len(_todo) - sum(_n.values())
        if _miss:
            log(f"  {_miss} came back without a usable verdict and are left "
                f"unjudged, so they are never skipped")
        # the ones judged in THIS round, not the first six in the file. It
        # printed the same six every round, which reads as though the judge
        # were going over the same themes again and again.
        _just = [t for t in _todo if t.get("members_seen")]
        for t in _just[:6]:
            if t.get("members_seen"):
                log(f"    {t['theme'][:42]:44s} {t[THEME_JUDGE_TAG]:8s} "
                    f"{', '.join(t['members_seen'])[:44]}")

    _num = sum(1 for t in THM["themes"] if t.get(THEME_JUDGE_TAG) == "numbers")
    _tot = len(THM["themes"])
    if _num:
        _share = (1-THEME_NUMERIC_SKIP)*_num / \
                 max(1e-9, (1-THEME_NUMERIC_SKIP)*_num + (_tot-_num))
        log(f"\n  {_num} of {_tot} themes have numeric members. At "
            f"THEME_NUMERIC_SKIP = {THEME_NUMERIC_SKIP} that is about "
            f"{_share:.0%} of what gets used,")
        log(f"  against {_num/_tot:.0%} if nothing were skipped. "
            f"{(1-THEME_NUMERIC_SKIP)*_num + (_tot-_num):.0f} themes stay usable.")

judge_themes()   # whatever is on file at the start


In [ ]:
RUN_SHOWN_PROMPT = """Here are the first {n} terms of a sequence.

{shown}

They were produced by this rule.

Rule: {rule}

Continue the sequence. Give terms {start} to {k}, applying the rule exactly as
written. Do not explain, do not comment, do not correct it.

Give the TERMS THE RULE EMITS, not the state it holds: what it says to write
down, once per step, each as a plain string. Never a list, a dict, an index
expression or a variable name.

Where the rule leaves a choice open, follow the terms above.

Answer in strict JSON, nothing else: {{"sequence": ["term{start}", ...]}}"""

SHOWN_N = 5          # terms shown before the rule. Five because that is the
                     # shortest prefix any item certifies at, and because it is
                     # available at generation: n_M comes from the pricing pass
                     # and needs a GPU.

def produce_shown(rule, want, shown_terms, iid="", reps=None, wins=None,
                  model=None, think=None):
    """Run the rule with the first terms shown, and check the CONTINUATION.

    The shown terms are not checked. They are given, so agreeing on them proves
    nothing, and counting them would make every item look more reproducible by
    the length of its own prefix. Agreement is measured from the first term
    after them.

    Returns (terms, votes, checked) with the same meaning as produce_terms,
    where `terms` is the whole sequence, shown terms included, so the caller
    can treat the two the same way.
    """
    reps = reps or EXTEND_REPS
    wins = wins or EXTEND_WINS
    n = len(shown_terms)
    t0 = time.time()

    def one(i):
        txt, note = ask(RUN_SHOWN_PROMPT.format(
                            n=n, shown=", ".join(shown_terms), rule=rule,
                            start=n + 1, k=want),
                        model=model, think=think,
                        temperature=EXTEND_TEMPS[i % len(EXTEND_TEMPS)],
                        max_out=GEN_MAX_OUT)
        if txt is None: return None
        d, head = parse_json(txt)
        if d is None: return None
        got, _ = clean_terms(d.get("sequence", []))
        return got or None

    runs = [r for r in parallel(one, list(range(reps)),
                                workers=min(MAX_WORKERS, reps)) if r]
    if not runs:
        log(f"      {iid} shown: no run returned anything usable "
            f"({time.time()-t0:.0f}s)")
        return None, 0, 0

    def agree_from(a, b):
        """How many terms of the CONTINUATION the two share."""
        m = 0
        for x, y in zip(a, b):
            if x != y: break
            m += 1
        return m

    best = max(runs, key=len)
    check = min((agree_from(best, r) for r in runs if r is not best),
                default=len(best))
    votes = sum(1 for r in runs if agree_from(best, r) >= check)
    log(f"      {iid} shown: {len(runs)} of {reps} replied, {votes} agree on "
        f"the first {check} continued terms ({time.time()-t0:.0f}s)")
    if votes < wins:
        worst = min((agree_from(best, r) for r in runs if r is not best),
                    default=0)
        log(f"      {iid} shown: only {votes} agreed, needed {wins}. They part "
            f"company {worst} terms after the {n} shown, so the rule does not "
            f"fix its own sequence even with the opening given")
        for k, sq in enumerate(runs[:3]):
            log(f"        run {k+1}: " + ", ".join(sq[:6])
                + (" ..." if len(sq) > 6 else ""))
        return None, votes, worst
    return list(shown_terms) + best[:check], votes, check

# ============================================================================
#  EXTENDING THE TERMS
#
#  A pass on its own: run each item's verified rule further and keep whatever
#  the runs agree on. No generation, no pricing, no GPU. It exists because the
#  listing has to be priced at N_# (Section: how long a listing to compare
#  against), and N_# reaches 82 terms while the sequences hold 72. Fifty more
#  terms covers every item in the battery.
#
#  It is also worth having anyway. Longer sequences give the executor-depth
#  test more to work with, and an item whose rule falls apart at term 80 is one
#  worth knowing about before it is used.

                          # explicit list of ids.

def _verdict(have, got, votes, checked, asked, build, shown=0):
    """One trial, in the shape every repro block uses.

    agreed_to and matched_to are independent and both matter: v17-0080 agrees
    with itself for a hundred terms and matches the stored ones for nine.
    """
    rec = dict(asked=asked, build=build, votes=votes, shown=shown or None,
               agreed_to=checked, matched_to=None, parted_at=None)
    if got is None:
        rec.update(verdict="runs disagreed", parted_at=checked)
        return rec
    m = 0
    for x, y in zip(have, got):
        if x != y: break
        m += 1
    rec["matched_to"] = m
    rec["verdict"] = ("reproduced" if m >= len(have)
                      else "diverged from stored")
    return rec

def second_opinion():
    """Run each item's seed under a SECOND executor, from another family.

    Gemini composes the rules and then judges whether they work, so a rule that
    survives generation has only been checked by the model that wrote it. That
    is not a neutral test, and it is not the test the reference machine will
    apply either: Gemini executes `sum_digits(len(c)^2 + n) % 5` without
    difficulty and Qwen3-14B does not. Every one of v17-0089, v17-0090 and
    v17-0091 passed generation with ten or more verified equivalents and then
    failed to certify, each after hours of GPU.

    So a second executor, weak and from another family, run on the SEED. It
    predicts nothing about accuracy and is not part of any definition. It is a
    filter: an item this model cannot execute is one to price last, or not at
    all.

    The verdict is [V2(p)]^{1@3}: three runs, and the item passes if ANY of
    them reproduces the stored terms. Both machines are stochastic and one bad
    trajectory is not evidence, so this is a test against RELIABLE failure.
    """
    want = [b for b in ITEMS
            if (SECOND_WHICH is None
                or (isinstance(SECOND_WHICH, str) and b["id"].startswith(SECOND_WHICH))
                or (not isinstance(SECOND_WHICH, str) and b["id"] in set(SECOND_WHICH)))]
    if SECOND_ONLY_MISSING:
        want = [b for b in want if not (b.get("second") or {}).get(SECOND_TAG)]
    stage(f"SECOND OPINION UNDER {SECOND_MODEL}")
    log(f"  {len(want)} item(s), {SECOND_REPS} run(s) each, "
        f"passing if {SECOND_WINS} reproduces the stored terms")
    log(f"  thinking budget {SECOND_THINK if SECOND_THINK is not None else 'the model default'}"
        f", answer budget {GEN_MAX_OUT}")
    if not want:
        log("  nothing to do: every item already has a verdict from this "
            "model. SECOND_ONLY_MISSING = False to run them again."); return

    done = ok = 0
    for b in want:
        # THE SEED, as the extension pass does: it is what generation ran and
        # what a description length means.
        i, rule = b["id"], (b.get("seed") or "")
        have = [str(x) for x in b["terms"]]
        if not rule:
            log(f"  {i}: no seed to run"); continue

        _L = b.get(f"n_M_{SECOND_PREFIX_TAG}") or SHOWN_N
        _L = min(_L, max(0, len(have) - SECOND_K))
        want = min(len(have), _L + SECOND_K)
        if want <= _L:
            log(f"  {i}: only {len(have)} terms, nothing to ask for after "
                f"a prefix of {_L}"); continue
        got, votes, checked = produce_shown(rule, want, have[:_L], iid=i,
                                            reps=SECOND_REPS, wins=1,
                                            model=SECOND_MODEL,
                                            think=SECOND_THINK)
        # agreement is measured on the CONTINUATION only
        agree = 0
        for a, bb in zip(have[_L:want], (got or [])[_L:want]):
            if a != bb: break
            agree += 1
        passed = bool(got) and agree >= (want - _L)
        b.setdefault("second", {})[SECOND_TAG] = dict(
            verdict="reproduced" if passed else "diverged",
            shown=_L, asked=want - _L, agreed=agree, votes=votes,
            model=SECOND_MODEL, think=SECOND_THINK, build=BUILD)
        done += 1; ok += passed
        log(f"  {i}: shown {_L}, asked {want - _L}: "
            + (f"reproduced all {want - _L}" if passed else
               f"DIVERGED at term {_L + agree + 1}"))
        if SECOND_PUSH_EVERY and done % SECOND_PUSH_EVERY == 0:
            save_all()
    save_all()
    state_save(f"{BUILD}: second opinion on {done} items")
    log(f"\n  {ok} of {done} reproduced under {SECOND_MODEL}")
    _bad = [b["id"] for b in want
            if (b.get("second") or {}).get(SECOND_TAG, {}).get("verdict")
            == "diverged"]
    if _bad:
        log(f"  a second executor could not run these, so price them last:")
        log(f"    " + ", ".join(_bad[:20]) + (" ..." if len(_bad) > 20 else ""))

def extend_terms():
    """Run each item's rule out to EXTEND_TO and keep what the runs agree on.

    THE RULE IS THE SEED, not the bare, because that is what generation ran.
    The first version of this pass used the bare where there was one, and it
    reported twelve items as diverging from their stored terms; ten of those
    twelve had run the bare. The two are different objects. The seed is the
    composer's rule and the terms were produced FROM it; the bare is the
    shortest rewriting later VERIFIED against those terms, which is a
    different test, and on v17-0011 it drops the words "in order of" and the
    run parts company at term 12.

    Uses produce_terms, which runs the rule COLD several times at different
    temperatures and keeps only terms every run produced. That matters more
    here than at generation: these terms are ground truth for pricing a
    listing, and a term the runs disagreed on is one nobody verified.

    Nothing already in the battery is overwritten. The existing terms were
    checked when the item was accepted; a fresh run could disagree with them
    for reasons that have nothing to do with the extension, and silently
    replacing them would invalidate every measurement made since. New terms
    are appended only where the first ones match.
    """
    want = [b for b in ITEMS
            if (not EXTEND_WHICH
                or (isinstance(EXTEND_WHICH, str) and b["id"].startswith(EXTEND_WHICH))
                or (not isinstance(EXTEND_WHICH, str) and b["id"] in set(EXTEND_WHICH)))]
    todo = [b for b in want
            if len(b.get("terms_ext") or b.get("terms") or []) < EXTEND_TO]
    stage(f"EXTENDING TERMS TO {EXTEND_TO}")
    log(f"  {len(want)} items, {len(todo)} shorter than {EXTEND_TO}")
    log(f"  state is pushed every 5 items and at the end, so a run that stops "
        f"part way keeps what it did.")
    log(f"  the rule run is {'the BARE, the shortest verified rewriting' if RULE_FROM == 'bare' else 'the SEED, which is what generation ran'}"
        f", trials: {TRIALS}")
    log(f"  the rule is run cold {EXTEND_REPS} times and only terms every run "
        f"agreed on are kept, so a rule that drifts yields fewer terms and "
        f"says where.")
    log(f"  existing terms are never overwritten: they were checked when the "
        f"item was accepted, and a fresh run disagreeing with them would "
        f"invalidate\n  every measurement made since. New terms are appended "
        f"only where the first ones match.\n")

    done = grew = drifted = failed = 0
    for n, b in enumerate(todo, 1):
        i = b["id"]
        if RULE_FROM == "bare":
            rule = b.get("bare") or ""
            src = "the verified equivalent"
        else:
            rule = b.get("seed") or ""
            src = "the seed"
        if not rule:
            log(f"  [{n}/{len(todo)}] {i}  no rule at all, skipped")
            failed += 1
            continue
        have = list(b.get("terms") or [])
        R = b.setdefault("repro", {})
        if TRIALS == "shown" and len(have) <= SHOWN_N:
            log(f"  [{n}/{len(todo)}] {i}  only {len(have)} terms, not more "
                f"than the {SHOWN_N} shown, so there is nothing to check")
            failed += 1
            continue

        # TRIAL 1, cold: the rule alone, which is what generation ran and what
        # a description length means. Skipped when only the shown trial is
        # wanted, which is the quick way to get a signal we do not have yet;
        # the cold verdicts for this battery are already recorded from earlier
        # runs and repeating them says nothing new.
        got = votes = checked = None
        if TRIALS in ("cold", "both"):
            got, votes, checked = produce_terms(rule, EXTEND_TO, iid=i)
            R[f"{RULE_FROM}_cold"] = _verdict(have, got, votes, checked,
                                      EXTEND_TO, BUILD)

        # TRIAL 2, shown: the first SHOWN_N terms given, then the rule. Weaker
        # by design, and closer to what the item asks. An item that fails cold
        # and passes here has a loose rule and a sound item.
        if TRIALS in ("shown", "both") and len(have) > SHOWN_N:
            sgot, svotes, schecked = produce_shown(
                rule, EXTEND_TO, have[:SHOWN_N], iid=i)
            R[f"{RULE_FROM}_shown"] = _verdict(have, sgot, svotes, schecked,
                                       EXTEND_TO, BUILD, shown=SHOWN_N)
            # the shown trial can extend the terms too, and reaches further on
            # items whose rule needed the opening pinned down
            if got is None or (sgot and len(sgot) > len(got or [])):
                _m = 0
                for x, y in zip(have, sgot or []):
                    if x != y: break
                    _m += 1
                if sgot and _m >= len(have):
                    got, votes, checked = sgot, svotes, len(have) + schecked
        if not got:
            log(f"  [{n}/{len(todo)}] {i}  produced nothing")
            failed += 1
            continue
        # where do the new terms stop matching the old?
        agree = 0
        for a, c in zip(have, got):
            if a != c: break
            agree += 1
        if agree < len(have):
            log(f"  [{n}/{len(todo)}] {i}  the rerun DIVERGES from the stored "
                f"terms at {agree} of {len(have)}, from {src}. Left alone.")
            b["extend_note"] = (f"rerun diverged at term {agree} of "
                                f"{len(have)}; terms not extended")
            drifted += 1
            continue
        if checked and checked < len(got):
            log(f"      trimming {len(got)} produced to the {checked} the runs "
                f"agreed on")
            got = got[:checked]
        # into terms_ext, never into terms. The annotator takes k from
        # len(terms) - n_M, so extending terms in place would change k and
        # with it h on every item that has already been measured.
        if len(got) > len(have):
            b["terms_ext"] = got
            b["terms_ext_to"] = len(got)
            grew += 1
        b["extend_from"] = src
        b["extend_checked_to"] = len(b["terms"])
        b["extend_build"] = BUILD
        log(f"  [{n}/{len(todo)}] {i}  {len(have)} -> "
            f"{len(b.get('terms_ext') or have)} terms"
            + (f", all agreed" if checked and checked >= len(b["terms"])
               else f", agreed to {checked}" if checked else "")
            + f"   ({src})")
        done += 1
        if n % 5 == 0:
            save_all()
            state_save(f"{BUILD}: {done} items extended")
    save_all()
    state_save(f"{BUILD}: extension finished, {done} items extended")
    import collections as _c
    for _k in (f"{RULE_FROM}_cold", f"{RULE_FROM}_shown"):
        _v = _c.Counter((b.get("repro", {}).get(_k) or {}).get("verdict")
                        for b in want if _k in b.get("repro", {}))
        if _v:
            log(f"\n  {_k}: " + ", ".join(f"{n} {k}" for k, n in _v.most_common()))
    _both = [b for b in want
             if (b.get("repro", {}).get(f"{RULE_FROM}_cold") or {}).get("verdict") != "reproduced"
             and (b.get("repro", {}).get(f"{RULE_FROM}_shown") or {}).get("verdict") == "reproduced"]
    if _both:
        log(f"  {len(_both)} items failed COLD and passed SHOWN: a loose rule "
            f"with a sound item, which is the case the shown trial exists for.")
        log(f"    " + ", ".join(b["id"] for b in _both[:10]))
    log(f"\n  {done} extended, {grew} of them grew, {drifted} left alone "
        f"because the rerun disagreed with what was stored, {failed} produced "
        f"nothing.")
    lens = [len(b.get("terms_ext") or b.get("terms") or []) for b in want]
    log(f"  terms per item now: median {sorted(lens)[len(lens)//2]}, "
        f"{min(lens)} to {max(lens)}")
    short = [b["id"] for b in want
             if len(b.get("terms_ext") or b.get("terms") or []) < EXTEND_TO]
    if short:
        log(f"  {len(short)} still short of {EXTEND_TO}: "
            + ", ".join(short[:8]) + (" ..." if len(short) > 8 else ""))
        log(f"  those need lambda(N_#) extrapolated from beta_20 rather than "
            f"priced, and the item records which.")

def next_id(used):
    n = 1
    while f"{ID_PREFIX}-{n:04d}" in used: n += 1
    return f"{ID_PREFIX}-{n:04d}"

def counts_characters(rule):
    """Does the rule take the LENGTH of something it had to recall, and then
    do arithmetic on it?

    A model sees tokens, not characters, so len("brisket") is not computed but
    recalled, and this is the same weakness behind failing to count letters in
    a word. Feed that count into a modulus and one character-level slip moves
    the index and every term after it. This is a property of how these models
    represent text, not a fact about this battery, which is why it is worth
    screening on where the other patterns we noticed are not.

    v17-0090 takes len(cut) of a retrieved primal cut, squares it, digit-sums
    and takes a modulus; v17-0091 does the same with the name of an element.
    Neither certified under Qwen3-14B, each after hours.

    Returns a reason, or None.
    """
    r = (rule or "").lower()
    if not re.search(r"\blen\s*\(|\blength of\b|\bnumber of (letters|characters)\b", r):
        return None
    if not re.search(r"%|\bmod\b|//|\bdigit[_ ]?sum\b|\bsum[_ ]?digits\b", r):
        return None
    return ("counts the characters of something recalled and does arithmetic "
            "on the count")

def clashes(terms, pool=None):
    """Does this open like an item already held? The paper records a pair that
    were rivals to each other and neither should have certified. A full check is
    O(n^2) over programs and belongs in pricing; this is the cheap prefix half.

    AGAINST THIS RUN'S KEEPERS AS WELL AS THE BATTERY. It looked only at ITEMS,
    which a candidate does not join until the run ends, so three slots came back
    with Twinkle Twinkle word by word and all three were kept. The rule-wording
    check does see them and could not help: "the words of twinkle twinkle little
    star", "the words of the nursery rhyme twinkle twinkle little star" and
    "first 28 words of Jane Taylor's 'The Star'" overlap at 0.40, 0.23 and 0.17,
    and the last names the same poem by its real title. The terms are identical
    and that is the thing to compare."""
    head = tuple(terms[:PREFIX_CLASH])
    for b in (pool if pool is not None else ITEMS):
        other = b.get("terms") or []
        if len(other) >= PREFIX_CLASH and tuple(other[:PREFIX_CLASH]) == head:
            return b.get("id") or "a candidate kept this run"
    return None

# Counters the summary reads, so they live beside the run rather than inside it.
RETRY_RESCUED, RETRY_STILL_NONE = [0], [0]
LOST_RETRIEVAL = []

LOST_RULE = []
REPAIR_LOG = []

def targets_for(spec, accepted_ids, kind):
    """Which items a later phase works on. See PROGRAMS_FOR in the config."""
    by_id = {b["id"]: b for b in ITEMS}
    if isinstance(spec, (list, tuple, set)):
        miss = [i for i in spec if i not in by_id]
        if miss: log(f"  not in the battery, ignored: {', '.join(map(str, miss))}")
        return [by_id[i] for i in spec if i in by_id]
    if spec == "new":
        return [by_id[i] for i in accepted_ids if i in by_id]
    have = PROG.get("items", {}) if kind == "programs" else None
    if spec == "missing":
        if kind == "programs":
            return [b for b in ITEMS if b["id"] not in have]
        return [b for b in ITEMS if b.get("gc_count") is None]
    if spec == "thin":
        if kind != "programs":
            return [b for b in ITEMS if b.get("gc_count") is None]
        out, exhausted = [], []
        for b in ITEMS:
            pr = have.get(b["id"])
            nv = sum(1 for e in (pr or {}).get("equivalents", [])
                     if e.get("verified"))
            nr = len((pr or {}).get("rivals", []))
            if pr is not None and nv >= THIN_EQUIV and nr >= THIN_RIVALS:
                continue                      # meets the bar
            if (pr or {}).get("attempts", 0) >= MAX_PROGRAM_ATTEMPTS:
                exhausted.append((b, nv, nr)); continue
            out.append(b)
        if exhausted:
            log(f"  {len(exhausted)} items have had {MAX_PROGRAM_ATTEMPTS} "
                f"attempts and are still short:")
            for b, nv, nr in exhausted[:10]:
                log(f"    {b['id']}  {nv} verified equivalents, {nr} rivals")
            if MARK_HOPELESS:
                n = sum(1 for b, nv, nr in exhausted
                        if mark(b, "flawed",
                                f"{MAX_PROGRAM_ATTEMPTS} programs passes left it "
                                f"with {nv} verified equivalents and {nr} rivals, "
                                f"below the bar of {THIN_EQUIV} and {THIN_RIVALS}"))
                log(f"  {n} marked flawed, so they stop being retried. An item with")
                log(f"  no program that reproduces its sequence has nothing to price.")
                save_all()
            else:
                log("  MARK_HOPELESS is off, so they are skipped this run but will "
                    "come back next time")
        # Most promising first: fewest attempts, then closest to the bar. A run
        # cut short then leaves behind the ones least likely to have worked.
        def promise(b):
            pr = have.get(b["id"]) or {}
            nv = sum(1 for e in pr.get("equivalents", []) if e.get("verified"))
            nr = len(pr.get("rivals", []))
            short = max(0, THIN_EQUIV - nv) + max(0, THIN_RIVALS - nr)
            return (pr.get("attempts", 0), short, b["id"])
        out.sort(key=promise)
        return out
    raise SystemExit(f"{spec!r} is not a valid target. Use 'new', 'missing', "
                     f"'thin', or a list of ids.")

def run():
    T0 = time.time()
    rejected = collections.Counter()
    accepted = []

    if THEME_TOPUP:
        stage(f"TOPPING UP THE THEME STORE BY {THEME_TOPUP}")
        log(f"  themes are otherwise fetched only when a run runs short, so "
            f"the store deepens as a side effect of generating items and never "
            f"faster.")
        _before = len(THM["themes"])
        theme_pool(sum(1 for t in THM["themes"] if not _uses(t)) + THEME_TOPUP)
        log(f"  store {_before} -> {len(THM['themes'])}")

    if DO_GENERATE and not SLOTS:
        stage("NOTHING TO GENERATE")
        log("  every cell in PLAN already holds N_PER_CELL usable items.")
        do_phase_5(ITEMS, accepted, T0, rejected)
        return
    if not DO_GENERATE:
        stage("GENERATION OFF")
        log("  DO_GENERATE is False, so phases 1 to 4 are skipped and this run")
        log(f"  only adds programs and components to items already in the battery.")
        do_phase_5(ITEMS, accepted, T0, rejected)
        return

    # ------------------------------------------------------------ phase 1 --
    stage("PHASE 1 - PLAN")
    pool = theme_pool(int(len(SLOTS) * THEMES_PER_SLOT))
    if not pool: raise SystemExit("no themes and none could be obtained")
    ti, plan, skipped, dry = 0, [], 0, False
    # WHAT TO ASK FOR, so that the cell wanted is the cell that arrives. The
    # composer delivers a little over half of any change asked for, so asking
    # for the target lands short of it at the top and past it at the bottom.
    # The fit is refitted here from every valid measured item.
    _cal = calibration(ITEMS)
    if _cal["live"]:
        log(f"  calibration from {_cal['n']} valid measured items:")
        log(f"    measured r = {_cal['r'][0]:+.2f} + {_cal['r'][1]:.2f} x asked")
        log(f"    measured t = {_cal['t'][0]:+.2f} + {_cal['t'][1]:.2f} x asked")
    else:
        log(f"  calibration off: {_cal['n']} valid measured items, "
            f"{CALIBRATE_FROM} needed. The ask is the target.")
    _asked = collections.Counter()
    # WITHOUT REPLACEMENT while there are orders left. Drawn independently,
    # eight slots over nine orders repeat one about nine times in ten, and a
    # repeated order is a repeated item: three slots came back with Twinkle
    # Twinkle word by word.
    _floors = random.sample(FLOORS, len(FLOORS))
    for slot, (_tr, _tt) in enumerate(SLOTS):
        # drawn per slot, not rounded: an ask of 1.6 is 1 two times in five, so
        # over a run the asks average where they were aimed
        r = draw(ask_for(_tr, _cal["r"], "r"))
        t = draw(ask_for(_tt, _cal["t"], "t"))
        _asked[((_tr, _tt), (r, t))] += 1
        # one order per slot at the floor, named in the prompt, so the ten in
        # FLOORS get used instead of the composer taking the first of a list
        _floor_for = None
        if t < 0:
            if not _floors:
                _floors = random.sample(FLOORS, len(FLOORS))
            _floor_for = _floors.pop()
        picked, tries = 0, 0
        while picked < OVERSAMPLE and tries < OVERSAMPLE * 5:
            tries += 1
            if ti >= len(pool):
                if dry: break
                held = {y["theme"] for y in pool}
                more = theme_pool(len(SLOTS), exclude=held)
                more = [x for x in more if x["theme"] not in held]
                if not more:
                    dry = True
                    log("  the theme store has nothing new to give")
                    break
                log(f"  {len(more)} more themes taken into the pool")
                pool += more
            th = pool[ti]; ti += 1
            if not theme_fits(th, r, t):
                skipped += 1; rejected["theme cannot carry the cell"] += 1; continue
            # r and t are what is ASKED, and what the item records, so the
            # next run's fit is measured against the ask that produced it.
            # target_r and target_t are what the slot is filling.
            plan.append(dict(slot=slot, r=r, t=t, theme=th, floor=_floor_for,
                             target_r=_tr, target_t=_tt))
            picked += 1
    if _cal["live"]:
        _moved = {k: v for k, v in _asked.items() if k[0] != k[1]}
        if _moved:
            log("  asked away from the target: " + ", ".join(
                f"r{a[0]}t{a[1]}->r{b[0]}t{b[1]} x{v}"
                for (a, b), v in sorted(_moved.items())[:8]))
    log(f"  {len(plan)} candidates across {len({p['slot'] for p in plan})} "
        f"of {len(SLOTS)} slots")
    if skipped:
        log(f"  {skipped} themes skipped as unsuitable, at no cost"
            + (": " + ", ".join(f"{v} {k}"
                                for k, v in THEME_SKIPS.most_common())
               if THEME_SKIPS else ""))
        if len(plan) < len(SLOTS):
            # How many themes a slot has to be offered before it finds one it
            # can use. The pool is sized on this, and it was estimated at one
            # when it is nearer seven, so a run planned 12 slots of 38.
            _rate = len(plan) / max(1, len(plan) + skipped)
            log(f"  {_rate:.0%} of the themes offered were used. "
                f"THEMES_PER_SLOT is {THEMES_PER_SLOT}; a slot needs about "
                f"{1/max(_rate, 0.01):.0f} offered to find one it can use.")

    # ------------------------------------------------------------ phase 2 --
    stage(f"PHASE 2 - GENERATE {len(plan)} CANDIDATES")
    def gen(job):
        th, t, r = job["theme"], job["t"], job["r"]
        txt, note = ask(GEN_PROMPT.format(
        # WHAT THE BATTERY ALREADY USES. Naming it is cheaper than screening
        # for it afterwards: the composer returns to the same few subjects
        # unless it is told which ones they are.
        used_line=_used_line(),
            # AT THE FLOOR THE ORDER IS THE THEME. A theme from the pool is
            # handed over before the floor text is read, and the composer
            # obliges both: "sandpaper grit size gradations" with terms 1921,
            # 1922, 1923. Naming the order as the theme leaves nothing to
            # dress up.
            theme=(job.get("floor") or th["theme"]) if job["t"] < 0
                  else th["theme"],
            retrieval_txt=retrieval_text(job["r"], job["t"]),
            transform_txt=transform_text(t, job.get("floor")), world_line=world_line(t, min_terms_for(t, r) - K_ADMIN),
            min_terms=min_terms_for(t, r), max_len=MAX_TERM_LEN,
            char_budget=chars_affordable(min_terms_for(t, r)),
            n_terms=min_terms_for(t, r),
            n_show=min_terms_for(t, r) - K_ADMIN,
            avoid_line=avoid_line((job["r"], job["t"])),
            shape_line=shape_line(t), shape_example=shape_example(t),
            low_t_example=low_t_example(r, t),
            terms_note=terms_note()), temperature=1.1,
            max_out=gen_max_out_for(t))
        out = dict(job)
        if txt is None: out["fail"] = f"no reply. {note}"; return out
        d, head = parse_json(txt)
        if d is None:
            out["fail"] = f"parse failed. {note + '. ' if note else ''}{head}"
            return out
        # "seed": what the composer first wrote. Older prompts said "rule".
        seed = str(d.get("seed") or d.get("rule") or "").strip()
        if not seed:
            out["fail"] = f"no seed in the reply. keys were {sorted(d.keys())}"
            return out
        terms, tnote = clean_terms(d.get("sequence", []))
        hint = d.get("answer_hint")
        def _lst(k):
            v = d.get(k)
            return [str(x).strip()[:40] for x in v if str(x).strip()][:12] \
                if isinstance(v, list) else None
        out.update(seed=seed, gloss=str(d.get("gloss", ""))[:300],
                   mechanism=str(d.get("mechanism", ""))[:80],
                   # what the composer SAYS it did. Free, arrives before
                   # anything is executed, and compared against the dials by
                   # the claim screen below. DECOMPOSE checks the same thing
                   # later against the bare rule and is the arbiter; this is
                   # only an early warning that costs nothing.
                   claimed_ops=_lst("operations"),
                   claimed_tables=_lst("tables"),
                   answer_hint=(int(hint) if isinstance(hint, (int, float)) else None),
                   terms=terms, term_note=tnote)
        return out
    cands = [c for c in parallel(gen, plan, label="generated") if c]

    # ------------------------------------------------------------ phase 3 --
    stage("PHASE 3 - SCREEN (no calls)")
    alive = collections.defaultdict(list)
    # Candidates kept so far, shaped like battery items so duplicate_of can read
    # them. Two slots drawing different themes can still land on one idea.
    kept_this_run = []
    for c in cands:
        tag = f"slot {c['slot']+1} r{c['r']}t{c['t']} {c['theme']['theme'][:26]}"
        if c.get("fail"):
            rejected["generation failed"] += 1
            log(f"  x {tag}\n      {c['fail'][:200]}"); continue
        if c.get("term_note"):
            log(f"  ! {tag}\n      terms cut short: {c['term_note'][:140]}")
        # With TERMS_FROM = "extend" the generation call only returns a sample;
        # the full sequence is produced in phase 4, so the floor applies there.
        # With "produce" the generation call returns no sequence, so there is
        # nothing to screen yet: the screens run in phase 4 on what the rule
        # actually produces.
        # The composer's own account of what it did, checked before a single
        # call is spent producing terms. It is not evidence about the rule, only
        # about the instruction being followed, so it is a screen and not the
        # arbiter: DECOMPOSE reads the bare rule later and that is what the
        # battery records. But a candidate that will not even CLAIM the right
        # number of operations is not going to measure them, and rejecting it
        # here costs nothing where rejecting it in phase 4 costs three produce
        # calls.
        _co, _ct = c.get("claimed_ops"), c.get("claimed_tables")
        _claim = []
        if _co is not None and abs(len(_co) - c["t"]) > DIAL_MARGIN:
            _claim.append(f"t asked {c['t']}, claims {len(_co)}: {_co}")
        if _ct is not None and abs(len(_ct) - c["r"]) > DIAL_MARGIN:
            _claim.append(f"r asked {c['r']}, claims {len(_ct)}: {_ct}")
        if _claim and CLAIM_SCREEN:
            rejected["claims the wrong number of operations or tables"] += 1
            log(f"  x {tag}\n      " + "; ".join(_claim)[:190])
            continue
        floor = 0 if TERMS_FROM == "produce" else accept_floor(c["t"], c.get("r", 0))
        if len(c["terms"]) < floor:
            rejected["too few terms"] += 1
            log(f"  x {tag}\n      only {len(c['terms'])} terms, floor is {floor} "
                f"at t={c['t']} (asked for {min_terms_for(c['t'], c.get('r', 0))})")
            continue
        # With "extend" this is only the sample, so the term screens cannot say
        # anything useful yet: they run in phase 4 on the full sequence.
        ok, reason, info = screen_terms(c["terms"], min_terms=floor,
                                        load=c["r"] + c["t"])
        if not ok:
            key = ("verbatim repeat" if "verbatim" in reason
                   else "continuable without the rule" if "continuable" in reason
                   else "too few terms")
            rejected[key] += 1
            if key == "verbatim repeat":
                # c["slot"], not `slot`. This loop is over CANDIDATES and the
                # bare name is left over from an earlier loop over slots, so
                # every repeat in phase 3 was filed under the same wrong slot.
                # The repair loop keeps one attempt per slot, so they all
                # collapsed into one and a run with seven lost rules repaired
                # a single item.
                LOST_RULE.append(dict(slot=c["slot"], r=c["r"], t=c["t"],
                                      floor=c.get("floor"),
                                      seed=c["seed"], kind="repeat",
                                      detail=reason))
            log(f"  x {tag}\n      {reason}"); continue
        if SCREEN_CHAR_COUNTS:
            _cc = counts_characters(c["seed"])
            if _cc:
                rejected["counts characters and does arithmetic on the count"] += 1
                log(f"  x {tag}\n      {_cc}"); continue
        clash = clashes(c["terms"], pool=ITEMS + kept_this_run)
        if clash:
            rejected["opens like an item already held"] += 1
            log(f"  x {tag}\n      first {PREFIX_CLASH} terms match {clash}"); continue
        twin, jr = duplicate_of(c["seed"], (c["r"], c["t"]),
                                pool=ITEMS + kept_this_run)
        if twin:
            rejected["measures the same as an item already held"] += 1
            log(f"  x {tag}\n      rule overlaps {twin['id']} at {jr:.2f} "
                f"({'same cell' if asked_cell(twin) == (c['r'], c['t']) else 'other cell'})")
            continue
        c["info"] = info
        bits = [f"{len(c['terms'])} terms", f"{info['n_distinct']} distinct"]
        if jr >= DEDUP_SAME_CELL * 0.7:
            bits.append(f"closest rule overlap {jr:.2f}, near the "
                        f"{DEDUP_SAME_CELL} line")
        if info.get("free_share", 0) >= FREE_SHARE_WARN:
            bits.append(f"{info['free_share']:.0%} scaffolding")
        if info.get("guess_best"): bits.append(f"guessable {info['guess_best']:.0%}")
        if info.get("period"):
            bits.append(f"period {info['period']} from {info['period_start']+1}")
        log(f"  ok {tag}\n      " + "; ".join(bits))
        alive[c["slot"]].append(c)
        kept_this_run.append(dict(id=f"(this run, slot {c['slot']+1})",
                                  seed=c["seed"], want_retrieval=c["r"],
                                  want_transformation=c["t"]))
    log(f"\n  {sum(len(v) for v in alive.values())} survived, "
        f"covering {len(alive)} of {len(SLOTS)} slots")
    tokens_report()

    # ------------------------------------------------------------ phase 4 --
    stage("PHASE 4 - DETERMINISM")
    def verify_slot(slot):
        tag = f"  slot {slot+1:>2}"
        for c in sorted(alive[slot], key=lambda x: -len(x["terms"])):
            # A rule runs to many lines now, so it is fenced. Without this the
            # theme, the rule and the verdict ran together in the log.
            rule_lines = "\n".join(f"{tag}| {ln}"
                                    for ln in c["seed"].split("\n")[:6])
            more = c["seed"].count("\n") - 5
            log(f"{tag}: THEME {c['theme']['theme'][:34]}  "
                f"({len(c['terms'])} terms, r{c['r']}t{c['t']})\n"
                f"{tag}: BEGIN SEED\n{rule_lines}"
                + (f"\n{tag}| ... {more} more lines" if more > 0 else "")
                + f"\n{tag}: END SEED")
            if TERMS_FROM == "produce":
                # Running the rule IS the determinism test. Nothing but the rule
                # is shown, so a rule that does not fix its own output fails
                # here, which is what should happen.
                full, votes, checked = produce_terms(
                    c["seed"], min_terms_for(c["t"], c.get("r", 0)), iid=f"slot {slot+1}")
                if full is None:
                    rejected["not deterministic"] += 1
                    # `checked` is how far the runs agreed before parting. It
                    # separates the two causes, which need opposite repairs:
                    # parting at once means a case was left open, parting late
                    # means the rule is clear but too heavy to carry.
                    # This is the branch that runs when TERMS_FROM is
                    # "produce", which it is. The appends were added to the
                    # other branch, so on one run 22 determinism failures and 4
                    # repeats went unrecorded and phase 4c repaired one rule out
                    # of 26 eligible.
                    LOST_RULE.append(dict(slot=slot, r=c["r"], t=c["t"],floor=c.get("floor"),
                                          seed=c["seed"], kind="determinism",
                                          agreed=checked,
                                          detail=(f"they agreed on the first "
                                                  f"{checked} term(s), then "
                                                  f"produced different ones")))
                    log(f"{tag}: rejected, the rule does not fix its own sequence")
                    continue
                floor = accept_floor(c["t"], c.get("r", 0))
                if len(full) < floor:
                    rejected["too few terms"] += 1
                    log(f"{tag}: rejected, the rule gave {len(full)} terms, "
                        f"floor is {floor}")
                    continue
                ok2, reason, info = screen_terms(full, load=c["r"] + c["t"])
                if not ok2:
                    key = ("verbatim repeat" if "verbatim" in reason
                           else "continuable without the rule"
                           if "continuable" in reason else "too few terms")
                    rejected[key] += 1
                    if key == "verbatim repeat":
                        LOST_RULE.append(dict(slot=slot, r=c["r"], t=c["t"],floor=c.get("floor"),
                                              seed=c["seed"], kind="repeat",
                                              detail=reason))
                    log(f"{tag}: rejected, {reason}")
                    continue
                if VALIDITY_SECOND_PASS:
                    # asks for VALIDITY_TO, not the item's own length. The
                    # terms beyond it are needed to price the listing at N_#,
                    # which reaches 82 terms on the v17 battery, and getting
                    # them here costs nothing extra: the call is being made
                    # either way.
                    again, v2, chk2 = produce_terms(
                        c["seed"], VALIDITY_TO,
                        iid=f"slot {slot+1} validity")
                    m = 0
                    for x, y in zip(full, again or []):
                        if x != y: break
                        m += 1
                    if again is None or m < len(full):
                        rejected["not valid, second pass"] += 1
                        LOST_RULE.append(dict(
                            slot=slot, r=c["r"], t=c["t"],
                            floor=c.get("floor"), seed=c["seed"],
                            kind="validity", agreed=m,
                            detail=("the second cold run did not agree with "
                                    "itself" if again is None else
                                    f"the second cold run matched the first "
                                    f"for {m} of {len(full)} terms")))
                        log(f"{tag}: rejected, NOT VALID: "
                            + ("the second run did not agree with itself"
                               if again is None else
                               f"the second run reproduced {m} of "
                               f"{len(full)} terms"))
                        continue
                    c["valid_votes"] = v2
                    c["terms_ext"] = again
                    c["terms_ext_to"] = len(again)
                    log(f"{tag}: valid, reproduced on a second cold run "
                        f"({len(again)} terms kept for pricing the listing)")
                # -----------------------------------------------------------
                c["terms"], c["info"] = full, info
                c["det_votes"], c["checked_to"] = votes, checked
                c["valid"] = bool(VALIDITY_SECOND_PASS)
                # The retrieval gate. One call, before any programs are bought.
                if DOMAIN_GATE:
                    knows, n_t, why = draws_on(c["seed"], iid=f"slot {slot+1}")
                    knows = knows or []
                    c["measured_r"], c["measured_t"] = len(knows), n_t
                    c["domain_seed"], c["domain_why"] = knows, why
                    # Recorded on every item either way, so the manipulation
                    # check is a table of asked against measured. Nothing else
                    # reads these.
                    off = []
                    if abs(len(knows) - c["r"]) > DIAL_MARGIN:
                        off.append(f"r asked {c['r']}, needs {len(knows)}")
                    if n_t >= 0 and abs(n_t - c["t"]) > DIAL_MARGIN:
                        off.append(f"t asked {c['t']}, does {n_t}")
                    if DOMAIN_GATE_NONE and c["r"] >= 1 and not knows:
                        off.append(f"r asked {c['r']} and it needs nothing")
                    if off:
                        rejected["off its cell by more than "
                                 f"{DIAL_MARGIN}"] += 1
                        LOST_RETRIEVAL.append(dict(slot=slot, r=c["r"], t=c["t"],
                                                   seed=c["seed"], why=why,
                                                   # without it the retry asks
                                                   # for the floor and records
                                                   # the pool's theme
                                                   floor=c.get("floor"),
                                                   got=len(knows), got_t=n_t))
                        log(f"{tag}: rejected, {'; '.join(off)}. {why}")
                        continue
                    log(f"{tag}: needs {len(knows)} table(s) {knows} and does "
                        f"{n_t} transformation(s), against r{c['r']}t{c['t']}. "
                        f"{why}")
                log(f"{tag}: accepted, {len(full)} terms, "
                    f"{info['n_distinct']} distinct, checked to term {checked}")
                return c
            ok, votes, agreed, why = deterministic(c["seed"], c["terms"],
                                                   indent=f"{tag}: ")
            if ok:
                c["det_votes"] = votes; return c
            # A rule whose executed output came back as state notation did not
            # disagree with its sequence: it could not be read. Counting the two
            # together overstates how often the composer is wrong.
            rejected["not deterministic" if why != "unreadable output"
                     else "output unreadable, rule not tested"] += 1
            if why != "unreadable output":
                LOST_RULE.append(dict(slot=slot, r=c["r"], t=c["t"],floor=c.get("floor"),
                                      seed=c["seed"], kind="determinism",
                                      detail=why, agreed=agreed))
            log(f"{tag}: rejected, the rule does not reproduce its own sequence")
        return None
    winners = [w for w in parallel(verify_slot, sorted(alive),
               workers=max(1, MAX_WORKERS // VERIFY_REPS)) if w]

    # ---------------------------------------------------- phase 4c ----------
    # Repair, rather than replace. A candidate lost to a repeat or to
    # determinism is handed back with the fault named and asked to fix THAT
    # world, up to REPAIR_ROUNDS times, each round told what the earlier ones
    # tried. The alternative, which is what happened before, is to write a
    # fresh rule from the same theme and lose the invention along with the
    # fault.
    if REPAIR_ROUNDS and LOST_RULE:
        filled = {w["slot"] for w in winners}
        todo, seen = [], set()
        for x in reversed(LOST_RULE):
            if x["slot"] in seen or x["slot"] in filled: continue
            seen.add(x["slot"]); todo.append(x)
        _orphans = len(todo)
        for x in reversed(LOST_RULE):
            if x["slot"] in seen: continue
            seen.add(x["slot"]); todo.append(x)
        todo = todo[:REPAIR_MAX]
        _det = sum(1 for x in LOST_RULE if x["kind"] == "determinism")
        _rep = sum(1 for x in LOST_RULE if x["kind"] == "repeat")
        log(f"  {len(LOST_RULE)} rule(s) lost and worth repairing: {_det} to "
            f"determinism, {_rep} to a repeat")
        log(f"  across {len(seen)} distinct slots, {_orphans} of them with "
            f"nothing else to show; repairing {len(todo)}")
        _lost_total = (rejected.get("not deterministic", 0)
                       + rejected.get("verbatim repeat", 0))
        if len(LOST_RULE) < _lost_total:
            log(f"  ({_lost_total} were rejected for those two reasons, so "
                f"{_lost_total - len(LOST_RULE)} were not recorded. A "
                f"rejection site is missing its LOST_RULE entry.)")
        if todo:
            stage(f"PHASE 4c - REPAIRING {len(todo)} RULE(S)")
            log(f"  a repeat or an unstated tie-break is a fault in the "
                f"writing, not in the idea, so the idea is kept and the fault "
                f"named.")
            log(f"  up to {REPAIR_ROUNDS} rounds each, every round told what "
                f"the last one tried.\n")

            def repair(x):
                slot, r, t = x["slot"], x["r"], x["t"]
                th = x.get("theme") or next(
                    (p["theme"] for p in plan if p["slot"] == slot), None)
                if th is None:
                    # silent before, which hid a slot recorded wrongly upstream
                    log(f"  slot {slot+1}: no theme in the plan for that slot, "
                        f"so it cannot be repaired. If this appears often, a "
                        f"phase is filing lost rules under the wrong slot.")
                    return None
                seed, kind, detail = x["seed"], x["kind"], x["detail"]
                tried = []
                n_terms = min_terms_for(t, r)
                for rd in range(1, REPAIR_ROUNDS + 1):
                    # parted inside OPEN_CASE_TERMS -> a case was left open;
                    # later -> the rule is clear and too heavy to carry
                    _ag = x.get("agreed")
                    if kind == "repeat":
                        _tpl = PROBLEM_REPEAT
                    elif _ag is not None and _ag > OPEN_CASE_TERMS:
                        _tpl = PROBLEM_HEAVY
                    else:
                        _tpl = PROBLEM_OPEN
                    problem = _tpl.format(detail=detail, n_terms=n_terms)
                    hist = ("" if not tried else REPAIR_HISTORY.format(
                        lines="".join(f"  {i}. {s[:150]}\n     failed: {w}\n"
                                      for i, (s, w) in enumerate(tried, 1))))
                    constraints = (
                        f"The cell asks for {r} memorised table(s) and "
                        f"{t} transformation(s). Give {n_terms} terms, each at "
                        f"most {MAX_TERM_LEN} characters, and keep the rule "
                        f"under about {chars_affordable(n_terms)} characters.")
                    txt, note = ask(REPAIR_PROMPT.format(
                        seed=seed[:400], problem=problem, history=hist,
                        world_txt=(REPAIR_WORLD + "\n" + world_line(t)
                                   if t >= WORLD_FROM else ""),
                        constraints=constraints),
                        temperature=0.7, max_out=gen_max_out_for(t))
                    d, head = parse_json(txt)
                    if not d or not d.get("seed"):
                        log(f"  slot {slot+1} round {rd}: no usable reply")
                        tried.append((seed, "the reply could not be read"))
                        continue
                    new = str(d["seed"]).strip()
                    fixed = str(d.get("fixed", ""))[:110]
                    log(f"  slot {slot+1} r{r}t{t} round {rd} "
                        f"[{'lighter' if _tpl is PROBLEM_HEAVY else 'clearer' if _tpl is PROBLEM_OPEN else 'longer period'}]: {fixed}")
                    log(f"      {new[:120]}")
                    # the same screens the candidate would have faced
                    full, votes, checked = produce_terms(
                        new, n_terms, iid=f"slot {slot+1} repair {rd}")
                    if full is None:
                        detail = "the runs still disagreed"
                        tried.append((new, "still not deterministic"))
                        REPAIR_LOG.append((slot, rd, kind, "determinism"))
                        seed = new; kind = "determinism"; continue
                    ok, reason, info = screen_terms(full, min_terms=n_terms
                                                    - TERMS_SLACK, load=r + t)
                    if not ok:
                        detail = reason
                        tried.append((new, reason[:60]))
                        REPAIR_LOG.append((slot, rd, kind,
                                           "repeat" if "verbatim" in reason
                                           else "terms"))
                        seed = new
                        kind = "repeat" if "verbatim" in reason else kind
                        continue
                    log(f"      repaired after {rd} round(s): {len(full)} "
                        f"terms, {info['n_distinct']} distinct")
                    REPAIR_LOG.append((slot, rd, x["kind"], "repaired"))
                    _cl = clashes(full, pool=ITEMS + winners)
                    if _cl:
                        log(f"      repaired but opens like {_cl}")
                        REPAIR_LOG.append((slot, rd, x["kind"], "duplicate"))
                        return None
                    _tw, _jr = duplicate_of(new, (r, t),
                                            pool=ITEMS + winners)
                    if _tw:
                        log(f"      repaired but its rule overlaps "
                            f"{_tw.get('id', 'a candidate')} at {_jr:.2f}")
                        REPAIR_LOG.append((slot, rd, x["kind"], "duplicate"))
                        return None
                    _v2 = _ext = None
                    if VALIDITY_SECOND_PASS:
                        _again, _v2, _ = produce_terms(
                            new, VALIDITY_TO,
                            iid=f"slot {slot+1} repair validity")
                        _m = 0
                        for _a, _b in zip(full, _again or []):
                            if _a != _b: break
                            _m += 1
                        if _again is None or _m < len(full):
                            log(f"      repaired but NOT VALID: "
                                + ("the second run produced nothing"
                                   if _again is None else
                                   f"it reproduced {_m} of {len(full)} terms"))
                            REPAIR_LOG.append((slot, rd, x["kind"],
                                               "repaired but not valid"))
                            return None
                        _ext = _again
                    return dict(slot=slot, r=r, t=t, theme=th, seed=new,
                                terms=full, det_votes=votes, info=info,
                                answer_hint=d.get("answer_hint"),
                                gloss=str(d.get("gloss", ""))[:300],
                                mechanism=str(d.get("mechanism", ""))[:80],
                                claimed_ops=d.get("operations"),
                                claimed_tables=d.get("tables"),
                                valid=bool(VALIDITY_SECOND_PASS),
                                valid_votes=_v2, terms_ext=_ext,
                                terms_ext_to=len(_ext) if _ext else None,
                                floor=x.get("floor"),
                                repaired_in=rd, repaired_from=x["seed"][:400])
                log(f"  slot {slot+1}: not repaired in {REPAIR_ROUNDS} rounds")
                REPAIR_LOG.append((slot, REPAIR_ROUNDS, x["kind"], "gave up"))
                return None

            fixed = [w for w in parallel(repair, todo,
                     workers=max(1, MAX_WORKERS // 2), label="repairing") if w]
            if fixed:
                log(f"\n  {len(fixed)} of {len(todo)} repaired")
                winners += fixed
            else:
                log(f"\n  none of the {len(todo)} could be repaired")

    # One more try for a slot that ran out of candidates having lost the
    # retrieval, told what went wrong. Same shape as the rival retry, and off
    # with DOMAIN_RETRY = False.
    if DOMAIN_GATE and DOMAIN_RETRY and LOST_RETRIEVAL:
        filled = {w["slot"] for w in winners}
        again = [x for x in LOST_RETRIEVAL if x["slot"] not in filled]
        seen_slots = set()
        again = [x for x in again
                 if not (x["slot"] in seen_slots or seen_slots.add(x["slot"]))]
        if again:
            stage(f"PHASE 4b - {len(again)} SLOTS LOST THE RETRIEVAL, ASKING AGAIN")
            for x in again:
                slot, r, t = x["slot"], x["r"], x["t"]
                th = x.get("theme") or next(
                    (p["theme"] for p in plan if p["slot"] == slot), None)
                if th is None: continue
                log(f"  slot {slot+1} r{r}t{t}: {th['theme'][:34]}")
                note = GEN_RETRY_NOTE.format(
                    seed=x["seed"][:400], why=x["why"], r=r,
                    got=(f"only {x['got']}" if x.get("got") else "none"))
                txt, _ = ask(GEN_PROMPT.format(
        used_line=_used_line(),
                    theme=(x.get("floor") or th["theme"]) if t < 0
                          else th["theme"],
                    retrieval_txt=retrieval_text(r, t),
                    # the repair keeps the candidate's own floor, as it keeps
                    # its own theme
                    transform_txt=transform_text(t, x.get("floor")), world_line=world_line(t, min_terms_for(t, r) - K_ADMIN),
                    min_terms=min_terms_for(t, r), max_len=MAX_TERM_LEN,
            char_budget=chars_affordable(min_terms_for(t, r)),
            n_terms=min_terms_for(t, r),
            n_show=min_terms_for(t, r) - K_ADMIN,
                    avoid_line=avoid_line((r, t)),
                    shape_line=shape_line(t), shape_example=shape_example(t),
                    low_t_example=low_t_example(r, t),
                    terms_note=terms_note() + note),
                    temperature=1.1, max_out=gen_max_out_for(t))
                d, head = parse_json(txt)
                if d is None:
                    log(f"    no usable reply"); continue
                seed = str(d.get("seed") or d.get("rule") or "").strip()
                if not seed: log(f"    no seed in the reply"); continue
                # x, not job: this is the retry loop and the candidate being
                # retried is what carries the floor
                cand = dict(slot=slot, r=r, t=t, theme=th,
                            floor=x.get("floor"), seed=seed,
                            gloss=str(d.get("gloss", ""))[:300],
                            mechanism=str(d.get("mechanism", ""))[:80],
                            answer_hint=None, terms=[], term_note="")
                full, votes, checked = produce_terms(
                    seed, min_terms_for(t, r), iid=f"slot {slot+1} again")
                if full is None:
                    log(f"    the new rule does not fix its own sequence"); continue
                ok2, reason, info = screen_terms(full, load=c["r"] + c["t"])
                if not ok2:
                    log(f"    {reason}"); continue
                knows, n_t, why = draws_on(seed, iid=f"slot {slot+1} again")
                knows = knows or []
                off = []
                if abs(len(knows) - r) > DIAL_MARGIN:
                    off.append(f"r asked {r}, needs {len(knows)}")
                if n_t >= 0 and abs(n_t - t) > DIAL_MARGIN:
                    off.append(f"t asked {t}, does {n_t}")
                if DOMAIN_GATE_NONE and r >= 1 and not knows:
                    off.append(f"r asked {r} and it needs nothing")
                if off:
                    log(f"    still off its cell: {'; '.join(off)}. {why}")
                    RETRY_STILL_NONE[0] += 1
                    continue
                # THE SAME DUPLICATE CHECKS AS THE SCREEN, which only phase 3
                # ran. A retried candidate was compared with nothing.
                _cl = clashes(full, pool=ITEMS + winners)
                if _cl:
                    log(f"    the new rule opens like {_cl}"); continue
                _tw, _jr = duplicate_of(seed, (r, t), pool=ITEMS + winners)
                if _tw:
                    log(f"    the new rule overlaps "
                        f"{_tw.get('id', 'a candidate')} at {_jr:.2f}")
                    continue
                cand.update(terms=full, info=info, det_votes=votes,
                            checked_to=checked, domain_seed=knows,
                            measured_r=len(knows), measured_t=n_t,
                            domain_why=why)
                if VALIDITY_SECOND_PASS:
                    _again, _v2, _ = produce_terms(
                        seed, VALIDITY_TO, iid=f"slot {slot+1} rescue validity")
                    _m = 0
                    for _x, _y in zip(full, _again or []):
                        if _x != _y: break
                        _m += 1
                    if _again is None or _m < len(full):
                        rejected["not valid, second pass"] += 1
                        log(f"    rescued but NOT VALID: "
                            + ("the second run produced nothing"
                               if _again is None else
                               f"it reproduced {_m} of {len(full)} terms"))
                        continue
                    cand["valid_votes"] = _v2
                    cand["terms_ext"] = _again
                    cand["terms_ext_to"] = len(_again)
                cand["valid"] = bool(VALIDITY_SECOND_PASS)
                winners.append(cand)
                RETRY_RESCUED[0] += 1
                log(f"    rescued: needs {len(knows)} table(s) and does {n_t} "
                    f"transformation(s), {len(full)} terms")

    _need = ("seed", "gloss", "mechanism", "terms", "theme", "r", "t", "info",
             "det_votes")
    _good = []
    for c in winners:
        _miss = [k for k in _need if k not in c]
        if _miss:
            log(f"  a candidate for slot {c.get('slot', '?')} is missing "
                f"{_miss} and cannot be saved. The others are kept.")
            log(f"     this is a fault in whichever phase produced it, not in "
                f"the item; the seed was: {str(c.get('seed'))[:90]}")
            rejected["incomplete candidate, not saved"] += 1
            continue
        _good.append(c)
    if len(_good) < len(winners):
        log(f"  {len(winners) - len(_good)} of {len(winners)} dropped as "
            f"incomplete, {len(_good)} saved")
    winners = _good

    used = {b["id"] for b in ITEMS}
    _made = []
    for c in winners:
        iid = next_id(used); used.add(iid)
        # a count, not a flag: a theme may carry up to THEME_REUSE items
        c["theme"]["uses"] = _uses(c["theme"]) + 1
        c["theme"]["used"] = True          # kept so older files still read
        item = dict(id=iid, seed=c["seed"], gloss=c["gloss"],
                    mechanism=c["mechanism"], terms=c["terms"],
                    theme=(c.get("floor") or c["theme"]["theme"])
                          if c["t"] < 0 else c["theme"]["theme"],
                    theme_domain=c["theme"].get("domain", ""),
                    want_retrieval=c["r"], want_transformation=c["t"],
                    load=c["r"] + c["t"],
                    mix=(round(c["r"]/(c["r"]+c["t"]), 2) if c["r"]+c["t"] else 0.0),
                    n_terms=len(c["terms"]), n_distinct=c["info"]["n_distinct"],
                    answer_hint=c.get("answer_hint"),
                    provenance=dict(build=BUILD, composer=COMPOSER,
                        gen_prompt=GEN_HASH,
                        run_date=datetime.datetime.now().isoformat(timespec="minutes"),
                        determinism=f"{c['det_votes']}/{VERIFY_REPS}"))
        # Carried straight through from the candidate. The measured dials are
        # recorded on every item and nothing else reads them: the manipulation
        # check is a table of asked against measured across the battery.
        for k in ("checked_to", "measured_r", "measured_t", "domain_seed",
                  "domain_why",
                  "valid", "valid_votes", "terms_ext", "terms_ext_to"):
            if c.get(k) is not None:
                item[k] = c[k]
        for k in ("period", "period_start", "period_confirmed_by", "free_share",
                  "guess_best", "guess_by", "collapse_at", "collapse_kind",
                  "guess_bar"):
            if c["info"].get(k) is not None: item[k] = c["info"][k]
        ITEMS.append(item); accepted.append(iid)
        _made.append((iid, c))
        log(f"  -> {iid}  r{c['r']}t{c['t']}  {c['theme']['theme'][:30]}  "
            f"{len(c['terms'])} terms")

    if _made:
        stage(f"WHAT THIS RUN MADE - {len(_made)} ITEMS")
        for iid, c in _made:
            t = c["terms"]
            log(f"\n{iid}  r{c['r']}t{c['t']}  [{c['theme']['theme'][:40]}]"
                + (f"  repaired in {c['repaired_in']} round(s)"
                   if c.get("repaired_in") else "")
                + f"\n  {len(t)} terms, {c['info'].get('n_distinct', '?')} "
                  f"distinct, verified to {c.get('checked_to', '?')}")
            log(f"  SEED  {str(c.get('seed', ''))[:400]}")
            if c.get("repaired_from"):
                log(f"  WAS   {str(c['repaired_from'])[:200]}")
            for k in range(0, min(SHOW_TERMS, len(t)), 10):
                log(f"   {k+1:>3d}: " + ", ".join(t[k:k+10]))
            if len(t) > SHOW_TERMS:
                log(f"   ... and {len(t)-SHOW_TERMS} more, ending "
                    f"{', '.join(t[-3:])}")

    save_all(f"{len(accepted)} items added")
    tokens_report()

    do_phase_5(ITEMS, accepted, T0, rejected,
               [(j["r"], j["t"]) for j in plan])

def do_phase_5(all_items, accepted, T0, rejected, planned_cells=()):
    new = [b for b in ITEMS if b["id"] in accepted]
    prog_targets = targets_for(PROGRAMS_FOR, accepted, "programs") if DO_PROGRAMS else []
    comp_targets = targets_for(COMPONENTS_FOR, accepted, "components") if DO_COMPONENTS else []
    if DO_PROGRAMS and PROGRAMS_MAX_ITEMS and len(prog_targets) > PROGRAMS_MAX_ITEMS:
        log(f"  budget of {PROGRAMS_MAX_ITEMS} items: taking the "
            f"{PROGRAMS_MAX_ITEMS} most promising of {len(prog_targets)}. The rest "
            f"keep their attempt count and come back next run.")
        prog_targets = prog_targets[:PROGRAMS_MAX_ITEMS]
    if DO_PROGRAMS and prog_targets:
        per_item = 1 + N_EQUIVALENTS*2 + len(RIVAL_PREFIXES)*(1+N_RIVALS)
        stage(f"PHASE 5 - PROGRAMS FOR {len(prog_targets)} ITEMS "
              f"(PROGRAMS_FOR = {PROGRAMS_FOR!r})")
        log(f"  about {per_item} calls per item, so roughly "
            f"{per_item*len(prog_targets)} calls in total")
        # Say what is actually short about each one. "Short" means the record
        # lacks what a pricing pass needs, not that the programs are brief.
        have = PROG.get("items", {})
        fresh = [b["id"] for b in prog_targets if b["id"] not in have]
        need_eq, need_rv = [], []
        for b in prog_targets:
            pr = have.get(b["id"])
            if pr is None: continue
            nv = sum(1 for e in pr.get("equivalents", []) if e.get("verified"))
            if nv < THIN_EQUIV: need_eq.append(f"{b['id']}({nv})")
            if len(pr.get("rivals", [])) < THIN_RIVALS: need_rv.append(b["id"])
        if fresh:
            log(f"  {len(fresh)} have no programs record at all")
        if need_eq:
            log(f"  {len(need_eq)} have fewer than {THIN_EQUIV} verified "
                f"equivalents: {', '.join(need_eq[:8])}"
                + (" ..." if len(need_eq) > 8 else ""))
        if need_rv:
            log(f"  {len(need_rv)} have no rival at any prefix: "
                f"{', '.join(need_rv[:8])}"
                + (" ..." if len(need_rv) > 8 else ""))
        log("  what already verified is kept. Only the missing part is asked for,")
        log("  so an item short of rivals does not pay for its equivalents again.")
        log(f"\n  THE QUEUE, in working order, most promising first:")
        log(f"    {'#':>3s}  {'item':10s} {'cell':7s} {'try':>3s}  what it needs")
        for _n, _b in enumerate(prog_targets, 1):
            _pr = have.get(_b["id"]) or {}
            _nv = sum(1 for e in _pr.get("equivalents", []) if e.get("verified"))
            _nr = len(_pr.get("rivals", []))
            if not _pr:
                _needs = "everything"
            else:
                _w = []
                if _nv < THIN_EQUIV: _w.append(f"{THIN_EQUIV-_nv} more equivalent(s)")
                if _nr < THIN_RIVALS: _w.append("a rival")
                _needs = ", ".join(_w) or "nothing"
            log(f"    {_n:3d}  {_b['id']:10s} "
                f"r{_b.get('want_retrieval')}t{_b.get('want_transformation'):<4d} "
                f"{_pr.get('attempts', 0):3d}  {_needs}")
            # The rule and the sequence, here too. A generation run prints them
            # in the listing before phase 5, but a thin pass has no generation
            # phase, so its queue was ids and cells only and there was no way
            # to see what was being worked on without opening the battery.
            _t = _b.get("terms") or []
            log(f"         [{str(_b.get('theme', ''))[:44]}]  {len(_t)} terms")
            log(f"         SEED  {str(_b.get('bare') or _b.get('seed') or '')[:200]}")
            for _k in range(0, min(SHOW_TERMS, len(_t)), 10):
                log("          " + ", ".join(_t[_k:_k+10])[:112])
            if len(_t) > SHOW_TERMS:
                log(f"          ... and {len(_t)-SHOW_TERMS} more")
        t5 = time.time()
        for n_done, b in enumerate(prog_targets):
            if TIME_BUDGET_MIN and (time.time() - T0)/60 > TIME_BUDGET_MIN:
                log(f"\n  time budget of {TIME_BUDGET_MIN} min reached after "
                    f"{n_done} of {len(prog_targets)} items. Stopping cleanly: "
                    f"everything already done is saved, and the items left keep "
                    f"their attempt count and come back next run.")
                break
            t = b["terms"]
            _att = (PROG.get("items", {}).get(b["id"]) or {}).get("attempts", 0) + 1
            head = [f"  {'-'*68}",
                    f"  [{n_done+1}/{len(prog_targets)}] {b['id']}  "
                    f"r{b['want_retrieval']}t{b['want_transformation']}"
                    f"  attempt {_att}  |  "
                    f"{len(prog_targets)-n_done-1} left after this one",
                    f"      THEME {b['theme']}  ({b.get('theme_domain','')})",
                    f"      GLOSS {b['gloss'][:200]}",
                    f"      BEGIN SEED"]
            head += [f"      | {ln}" for ln in b["seed"].split("\n")[:10]]
            if b["seed"].count("\n") > 9:
                head.append(f"      | ... {b['seed'].count(chr(10)) - 9} more lines")
            head += [f"      END SEED",
                     f"      TERMS {len(t)}, {len(set(t))} distinct:"]
            for i in range(0, min(len(t), 30), 10):
                head.append(f"      {i+1:4d}: " + ", ".join(t[i:i+10]))
            if len(t) > 30: head.append(f"      ... and {len(t)-30} more")
            log("\n".join(head))
            old = PROG.get("items", {}).get(b["id"])
            rec = programs_for(b, old)
            rec["attempts"] = (old or {}).get("attempts", 0) + 1
            rec["last_attempt"] = datetime.date.today().isoformat()
            PROG.setdefault("items", {})[b["id"]] = rec
            # mirror the proxy onto the item, so the Selector and a pricing pass
            # can read it without opening the programs file
            for k in ("shortest_equivalent_chars", "terms_needed_estimate",
                      "bare"):
                if rec.get(k) is not None:
                    b[k] = rec[k]
            if rec.get("terms_needed_estimate") and \
               rec["terms_needed_estimate"] > len(b["terms"]):
                b["terms_short_by"] = rec["terms_needed_estimate"] - len(b["terms"])
            save_all()
            # Phase 5 is the slow one, so say where it is and when it ends.
            elapsed = time.time() - t5
            per = elapsed / (n_done + 1)
            left = len(prog_targets) - (n_done + 1)
            eta = datetime.datetime.now() + datetime.timedelta(seconds=per*left)
            nxt = prog_targets[n_done+1]["id"] if left else None
            if STATE_PUSH_EVERY and (n_done + 1) % STATE_PUSH_EVERY == 0:
                state_save(f"{BUILD}: {n_done+1} items through phase 5")
            log(f"      [{n_done+1}/{len(prog_targets)}] {b['id']} done. "
                f"{elapsed/60:.0f} min so far, {per/60:.1f} min each. "
                + (f"{left} left, about {per*left/60:.0f} min (~{eta:%H:%M}), "
                   f"next is {nxt}" if left else "queue finished")
                + f". ${spend_usd():.2f} spent")
    elif DO_PROGRAMS:
        log(f"\n  nothing matched PROGRAMS_FOR = {PROGRAMS_FOR!r}")
    # After the programs pass, so `bare` exists and the domain is read from it
    # rather than from the theme's dressing.
    if DO_COMPONENTS and comp_targets:
        stage(f"PHASE 5b - COMPONENTS FOR {len(comp_targets)} ITEMS "
              f"(COMPONENTS_FOR = {COMPONENTS_FOR!r})")
        res = parallel(components_for, comp_targets, label="typed")
        for b, r in zip(comp_targets, res):
            if r: b.update(r)
        save_all()
    # The summary is a report on work already saved. A fault in it must not
    # look like a failed run.
    try:
        summarise(new, accepted, rejected, prog_targets, T0, planned_cells)
    except Exception as _e:
        log(f"\nthe summary failed: {type(_e).__name__}: {_e}")
        log("the items themselves were saved and pushed before this point.")

def summarise(new, accepted, rejected, prog_targets, T0, planned_cells=()):

    # --------------------------------------------------------------- done --
    stage("SUMMARY")
    if DO_GENERATE:
        log(f"accepted {len(accepted)} of {len(SLOTS)} slots")
    else:
        log(f"generation was off; worked on {len(prog_targets)} items")
    if rejected:
        log("\nwhere candidates died:")
        for k, v in rejected.most_common(): log(f"  {v:4d}  {k}")
        if rejected.get("output unreadable, rule not tested"):
            log("\n  'output unreadable' is not the composer being wrong about its")
            log("  own sequence. The rule was a function or generator and the")
            log("  verifier printed the state instead of the terms. If this count")
            log("  stays high, the instruction in RUN_PROMPT is not landing.")
        if REPAIR_LOG:
            _by = collections.Counter(x[3] for x in REPAIR_LOG)
            _ok = [x for x in REPAIR_LOG if x[3] == "repaired"]
            _slots = {x[0] for x in REPAIR_LOG}
            log(f"\n  repair: {len(_ok)} of {len(_slots)} rule(s) fixed rather "
                f"than replaced")
            log(f"  a candidate lost to a repeat or an unstated tie-break is "
                f"handed back with the fault named, up to {REPAIR_ROUNDS}")
            log(f"  times. Without it the idea goes with the fault: a whole "
                f"invented mechanism thrown away over a ring two cells")
            log(f"  too narrow.")
            _k = collections.Counter(x[2] for x in REPAIR_LOG
                                     if x[3] == "repaired")
            if _k:
                log(f"  what was fixed: "
                    + ", ".join(f"{v} {k}" for k, v in _k.items()))
            if _by.get("gave up"):
                log(f"  {_by['gave up']} could not be fixed in "
                    f"{REPAIR_ROUNDS} rounds")
        _by_id = {b["id"]: b for b in ITEMS}
        _fam = collections.Counter()
        for _w in accepted:
            w = _by_id.get(_w) if isinstance(_w, str) else _w
            if not isinstance(w, dict): continue
            _txt = str(w.get("seed", "")).lower()
            for _name, _pat in (
                    ("queue", r"\bqueue|dequeue|front of|first in\b"),
                    ("stack", r"\bstack|pile|topple\b"),
                    ("ring", r"\bring|wrap|lamps?\b"),
                    ("row or board", r"\brow|board|track|siding|cells?\b"),
                    ("string rewrite", r"\brewrit|substitut|replace each\b"),
                    ("counters", r"\bcounter|carry|odometer|running total\b"),
                    ("containers", r"\bvessel|jug|pour|boat|capacity\b"),
                    ("sorting", r"\bsort|tournament|compare|swap\b"),
                    ("grid", r"\bgrid\b")):
                if re.search(_pat, _txt): _fam[_name] += 1; break
            else:
                _fam["no world, a rule on the index"] += 1
        if _fam:
            log(f"\n  the shapes the accepted items took:")
            for _k, _v in _fam.most_common():
                log(f"    {_v:3d}  {_k}")
            _none = _fam.get("no world, a rule on the index", 0)
            _top = _fam.most_common(1)[0]
            if len(accepted) >= 6 and _none > len(accepted) * 0.4:
                log(f"  {_none/len(accepted):.0%} took no world at all. If "
                    f"WORLD_PUSH is on, that is the push not landing rather "
                    f"than the")
                log(f"  shapes being wrong: the composer is falling back on a "
                    f"rule over the index, which is what it does when a theme "
                    f"resists.")
            elif len(accepted) >= 6 and _top[1] > len(accepted) * 0.4:
                log(f"  {_top[0]} is {_top[1]/len(accepted):.0%} of them. WORLDS "
                    f"is a list of shapes picked at random, so a skew here "
                    f"usually")
                log(f"  means the list is skewed: it once held twelve entries "
                    f"of which five were queues.")
        if rejected.get("measures the same as an item already held"):
            log("\n  the duplicate check compares RULE wording, at 0.35 within a")
            log("  cell and 0.80 across. Thresholds measured on the v5 battery.")
            log("  It does not use `mechanism`, which scored higher on pairs that")
            log("  had to stay separate than on the real duplicates.")
        if rejected.get("theme cannot carry the cell"):
            log("\n  theme skips cost nothing, but they consume the pool, so a")
            log("  large count here means fewer slots got planned.")
    if not new and not prog_targets:
        log("\nnothing to report."); tokens_report(); return
    if not new:
        new = prog_targets       # a programs-only run reports on what it touched

    log(f"\nbattery now holds {len(ITEMS)} items")
    cells = collections.Counter((b["want_retrieval"], b["want_transformation"])
                                for b in new)
    log(f"  cells {'filled' if DO_GENERATE else 'covered'}: {dict(sorted(cells.items()))}")
    if DO_GENERATE or FILL:
        have, _ = battery_counts()
        short = [(c, have.get(c, 0)) for c in sorted(set(CELLS))
                 if have.get(c, 0) < N_PER_CELL]
        log(f"\n  grid against the target of {N_PER_CELL} per cell:")
        log(f"    {len(set(CELLS)) - len(short)} of {len(set(CELLS))} cells met")
        if short:
            log("    still short: " + ", ".join(f"r{c[0]}t{c[1]}({h}/{N_PER_CELL})"
                                                for c, h in short[:12])
                + (" ..." if len(short) > 12 else ""))
            log("    re-run with FILL = True to ask only for these")
    asked = sorted(set(planned_cells)) if DO_GENERATE else []
    missing = [c for c in asked if c not in cells]
    if missing:
        log(f"  of the {len(asked)} cell(s) this run planned, {len(missing)} got "
            f"nothing: {missing}")
        deep = [c for c in missing if c[1] >= 5]
        if deep:
            log(f"  {len(deep)} of them are t>=5, which is where the frontier")
            log(f"  fails. Re-run with PLAN = GRIDS['frontier'] to target them;")
            log(f"  the battery is appended to, so nothing is lost.")
    lens = [b.get("n_terms") or len(b["terms"]) for b in new]
    log(f"  terms: median {int(st.median(lens))}, {min(lens)} to {max(lens)}")

    scaff = [b for b in new if (b.get("free_share") or 0) >= FREE_SHARE_WARN]
    if scaff:
        log(f"  {len(scaff)} are mostly scaffolding: "
            + ", ".join(f"{b['id']} {b['free_share']:.0%}" for b in scaff))
    withp = [b for b in new if b.get("period")]
    for b in withp:
        log(f"  {b['id']} settles into period {b['period']} from term "
            f"{b['period_start']+1}, so administer below term "
            f"{b['period_start']+b['period']+1}")

    if DO_PROGRAMS:
        eq = [len(PROG["items"][b["id"]]["equivalents"]) for b in new
              if b["id"] in PROG["items"]]
        ev = [sum(1 for e in PROG["items"][b["id"]]["equivalents"] if e["verified"])
              for b in new if b["id"] in PROG["items"]]
        rv = [len(PROG["items"][b["id"]]["rivals"]) for b in new
              if b["id"] in PROG["items"]]
        log(f"\n  equivalents: {sum(eq)} proposed, {sum(ev)} reproduce the sequence")
        log(f"  rivals: {sum(rv)} confirmed across {RIVAL_PREFIXES}")
        norivals = [b["id"] for b in new
                    if not PROG.get("items", {}).get(b["id"], {}).get("rivals")]
        if norivals:
            log(f"  {len(norivals)} of {len(new)} ended with NO rival at any "
                f"prefix: {', '.join(norivals[:8])}"
                + (" ..." if len(norivals) > 8 else ""))
            log("     a pricing pass will certify these at L0 by default. Re-run")
            log("     with DO_GENERATE = False and PROGRAMS_FOR = 'thin' to retry.")
        if EQUIV_RESCUED[0]:
            log(f"\n  {EQUIV_RESCUED[0]} equivalents came only from the second, "
                f"informed round. Those items would otherwise have had fewer.")
        if RIVALS_SKIPPED[0]:
            log(f"  {RIVALS_SKIPPED[0]} items had their rivals skipped for "
                f"having no verified equivalent, saving about "
                f"{RIVALS_SKIPPED[0]*len(RIVAL_PREFIXES)*2.5:.0f} minutes of "
                f"work that could not have been priced against anything.")
        if RIVAL_RESCUED[0]:
            log(f"\n  {RIVAL_RESCUED[0]} rivals were found only by the second, "
                f"informed round. Those prefixes would otherwise have had none.")
        if RIVAL_FAILURES:
            tot = sum(RIVAL_FAILURES.values())
            log(f"\n  where rival candidates went, {tot} discarded in total:")
            for k, v in RIVAL_FAILURES.most_common():
                log(f"    {v:4d}  {k}")
            log("  what each means, and what it would take to fix:")
            log("    agreed on fewer than L  the executed output did not match the")
            log("      prefix. If the agreement is 0 it is usually the surface form:")
            log("      the rival is written with the terms in view and executed in a")
            log("      fresh call that sees only the rule, so a unit it never states")
            log("      is lost. If the agreement is close to L, the prefix is tightly")
            log("      pinned and cheap rivals may genuinely not exist.")
            log("    did not diverge        the composer produced an equivalent.")
            log("    none proposed          it looked and found nothing.")
            log("    call gave nothing      the reply was truncated or unparseable.")
            if RIVAL_NEAR:
                ag = [a for _, _, a in RIVAL_NEAR]
                zero = sum(1 for a in ag if a == 0)
                log(f"\n  of the {len(ag)} that missed the prefix, {zero} agreed on "
                    f"NOTHING and {len(ag)-zero} were near misses")
                log(f"    median agreement {int(st.median(ag))} terms")
                if zero > len(ag) * 0.6:
                    log("    mostly zero, so this is the surface form and the rival")
                    log("    prompt should require each rival to state its output format")
                elif ag and st.median(ag) >= 2:
                    log("    mostly near misses, so the prefixes are tightly pinned")
                    log("    and cheap rivals may not exist for these items")
        gaps = collections.Counter()
        for b in new:
            for L in PROG.get("items", {}).get(b["id"], {}).get("no_rival_at", []):
                gaps[L] += 1
        if gaps:
            log(f"  prefixes where no rival was found: "
                + ", ".join(f"L={L} on {n} items" for L, n in sorted(gaps.items())))
            log("     a pricing pass will certify there by default. That is a fact")
            log("     about the search, not about the item.")

    if (rejected.get(f"off its cell by more than {DIAL_MARGIN}")
            or RETRY_RESCUED[0]):
        log(f"\n  the retrieval gate:")
        log(f"    {rejected.get(f'off its cell by more than {DIAL_MARGIN}', 0)} "
            f"candidates discarded for being more than {DIAL_MARGIN} away from "
            f"their cell on r or t")
        if RETRY_RESCUED[0]:
            log(f"    {RETRY_RESCUED[0]} slots rescued by the second, informed "
                f"generation call")
        if RETRY_STILL_NONE[0]:
            log(f"    {RETRY_STILL_NONE[0]} still drew on nothing after being "
                f"told; those slots are empty")
        log(f"    each rejection cost 1 call rather than the ~"
            f"{1 + N_EQUIVALENTS*2 + len(RIVAL_PREFIXES)*(1+N_RIVALS)} its "
            f"programs would have")
    ds = [b for b in new if b.get("measured_r") is not None]
    if ds:
        log(f"\n  ASKED against MEASURED, the manipulation check. Nothing acts")
        log(f"  on these; they are recorded so the correspondence can be")
        log(f"  reported across the battery.")
        log(f"    {'item':10s} {'asked':>5s} {'measured':>8s}   what it needs")
        for b in sorted(ds, key=lambda x: (x.get("want_retrieval") or 0,
                                           x.get("want_transformation") or 0)):
            log(f"    {b['id']:10s} "
                f"r{b.get('want_retrieval')}t{b.get('want_transformation')}"
                f"    r{b['measured_r']}t{b['measured_t']}      "
                f"{', '.join(b.get('domain_seed') or ['nothing'])[:34]}")
        dr_ = [b["measured_r"] - (b.get("want_retrieval") or 0) for b in ds]
        dt_ = [b["measured_t"] - (b.get("want_transformation") or 0)
               for b in ds if b["measured_t"] >= 0]
        log(f"    r: measured minus asked, median {int(st.median(dr_)):+d}, "
            f"{sum(1 for x in dr_ if x == 0)} of {len(dr_)} exact")
        if dt_:
            log(f"    t: measured minus asked, median {int(st.median(dt_)):+d}, "
                f"{sum(1 for x in dt_ if x == 0)} of {len(dt_)} exact")
    dr = [b for b in new if b.get("domain_real")]
    if dr:
        real = collections.Counter(b["domain_real"] for b in dr)
        log(f"\n  the domain the RULE draws on, read from the bare rule:")
        for d, n in real.most_common():
            log(f"    {n:3d}  {d}")
        decor = [b for b in dr if b["domain_real"] == "none"
                 and b.get("theme_domain") not in ("", None)]
        if decor:
            log(f"  {len(decor)} of {len(dr)} have a theme domain but draw on "
                f"nothing, so the theme is decoration:")
            for b in decor[:8]:
                log(f"    {b['id']}  theme says {b.get('theme_domain')!r}, "
                    f"rule draws on nothing")
            log("  That is what r=0 asks for, and it means `theme_domain` is a")
            log("  label rather than a fact. Use `domain_real` for spread and")
            log("  for anything reported.")
    bares = [b for b in new if b.get("bare")]
    if bares:
        ln = [len(b["bare"]) for b in bares]
        au = [len(" ".join(b["seed"].split())) for b in bares]
        log(f"\n  bare rule: median {int(st.median(ln))} chars against "
            f"{int(st.median(au))} for the authored rule "
            f"({st.median(ln)/max(1,st.median(au)):.0%})")

    typed = [b for b in new if b.get("gc_count") is not None]
    if typed:
        log(f"\n  the gauge on the dials ({len(typed)} items typed):")
        rows = collections.defaultdict(list)
        for b in typed:
            rows[(b["want_retrieval"], b["want_transformation"])].append(b)
        for cell in sorted(rows):
            g = [b["gc_count"] for b in rows[cell]]
            k = st.mean(b["n_knowledge"] for b in rows[cell])
            o = st.mean(b["n_operation"] for b in rows[cell])
            want = cell[0] / (cell[0] + cell[1]) if sum(cell) else 0.0
            log(f"    r{cell[0]}t{cell[1]}  asked mix {want:.2f}  "
                f"got {st.mean(g):.2f}   knowledge {k:.1f}, operation {o:.1f}")
        log("    'asked' is r/(r+t); 'got' is knowledge spans over knowledge plus")
        log("    operation spans. They measure different things and will not match")
        log("    exactly. A cell where they disagree badly did not land.")
    failed = [b["id"] for b in new if b.get("components_source") == "failed"]
    if failed: log(f"  typing failed on {len(failed)}: {', '.join(failed[:6])}")

    have = PROG.get("items", {})
    ready = [b for b in ITEMS
             if sum(1 for e in have.get(b["id"], {}).get("equivalents", [])
                    if e.get("verified")) >= THIN_EQUIV
             and have.get(b["id"], {}).get("rivals")]
    log(f"\nbattery readiness: {len(ready)} of {len(ITEMS)} items have "
        f"{THIN_EQUIV}+ verified equivalents and at least one rival")
    log(f"  those are the ones a pricing pass can say something about")

    log(f"\ntotal elapsed: {(time.time()-T0)/60:.1f} min")
    tokens_report()
    # The confirmation. With AFC disabled the SDK should stop warning, and with
    # no tools configured nothing else should change.
    log(f"  automatic function calling: "
        + (f"disabled, and the SDK warned {_AFC.n} times"
           if DISABLE_AFC else f"left at its default, {_AFC.n} warnings")
        + (". No tool tokens were billed on any call, so no interpreter ran."
           if not TOKENS["tool_tokens"] else
           f". {TOKENS['tool_tokens']:,} TOOL TOKENS were billed."))
    if accepted:
        log(f"  per accepted item: {(time.time()-T0)/60/len(accepted):.1f} min, "
            f"${spend_usd()/len(accepted):.2f}")
    log(f"\nwrote {BATTERY}, {PROGRAMS}, {THEMES}, {PROMPTS}"
        + (f", copied to {OUTDIR}" if OUTDIR != "." else ""))
    account(units=len(accepted) or len(prog_targets),
            note=f"{PLAN_NAME}, {len(accepted)} accepted")
    state_save(f"{BUILD}: {len(accepted)} items added, battery now {len(ITEMS)}")
    log("\nNext: a pricing pass adds n_M and h under a reference machine. Nothing "
        "here decided whether an item is unquestionable.")

# Each of these does one thing and nothing else, so a battery can be
# lengthened, or judged by a second executor, without touching generation.
if SECOND_ONLY:
    second_opinion()
elif EXTEND_ONLY:
    extend_terms()
else:
    run()
